# UNIVERSAL_LOG v7.4.0 — Monolithe Jupyter

Notebook auto-suffisant généré depuis le package modulaire `universal_log/`.
Les 13 modules sont fusionnés dans un namespace unique (une cellule = une section du pipeline).
Exécution de haut en bas : le code du package, puis une démo `analyze()` et le self-test.

**SHA-256 des modules sources :**
- `config.py` — `d27ea0a644fe17eed82379bc4e5975089f58fd20a0c8823a109aaf222be606b1`
- `data.py` — `37160b873becd385c53e20d9e9a3df86be5005359bd652021a88d390d101e8a9`
- `stats.py` — `985189924659c7b231d1bef967583974f0d718b6c28f22ca138a55d9174377e2`
- `detection.py` — `f8eb0d34a0b4b5a8fb44492beffcd69600414c872a5d9b94fa8a281e2d4ecbf3`
- `estimators.py` — `8beda00a17db9f2062a80d1316160e1c4066c0958e7cc7918d0c30ecd770cd7a`
- `diagnostics.py` — `147b7dc2bb340319f3c414e11fb2c98d417808973b352923dbae8629b6575430`
- `falsification.py` — `46aabf2ffafed6be3308680db692f6974c75a2de80e5700d28a60eec4787c2d8`
- `levels.py` — `bcc34214ac94e8208ce69f0ae17bb4026c357c40ccafb2de2d062a2aa7a011ee`
- `report_html.py` — `2087df322d5b54a40a96ee82ed1d5a33b8611f20fa259373de66c92e2b2cc2f0`
- `prediction.py` — `11d78de12fcb655698b8ecd5b70329e4bf785ff9e4772787e837374d181923f4`
- `pipeline.py` — `aa8fe909373cdc24e60017b6a7b192b67a2249f5af16f0b372e3c804e585b640`
- `regression.py` — `69367cf23b78433745275b1a3bfac43e125f40c78290d027b7be87f1fd617b2a`
- `cli.py` — `2526a4b5018a2fa1e375bbbecb9a5b50392c155e90e4c5ad03f37b66e6c86ec3`

> T_log = diagnostic calibré, jamais une preuve (P8). Voir `theory/calibrated/TLOG_calibrated_v7.4.0.md` et `LIVRE_BLANC.md`.

In [ ]:
from __future__ import annotations

## Module `config.py`

In [ ]:
"""Configuration and constants for UNIVERSAL_LOG.

v7 Design Principles (unchanged):
- P1: No domain names or domain-specific heuristics in code.
- P2: No hand-tuned weights (inverse-variance stability weighting, or equal weights).
- P3: No semantic direction from column names (direction = empirical Cohen's d sign).
- P4: No harmonic prior (uniform MC + permutation; harmonic ablation is comparative only).
- P5: Falsification first (permutation, white noise, surrogate, split-half, walk-forward).
- P6: Effect size + 95% CI + p-value, never a bare score alone.
- P7: Full reproducibility (seed, manifest, input/code SHA-256 hashes).
- P8: T_log = permutation-calibrated complexity diagnostic, never a proof.
"""


import math
from dataclasses import asdict, dataclass, field
from typing import Any, Dict, List

VERSION = "v7.4.0-universal"
VERSION_MASTER = "v7.4.0-universal-modular-calibrated"
RANDOM_SEED = 42

# Mathematical constants retained solely for optional harmonic ablation tests (P4).
PHI = (1.0 + 5.0 ** 0.5) / 2.0
PI = math.pi
E = math.e
SQRT2 = math.sqrt(2.0)
SQRT3 = math.sqrt(3.0)

MODE_PRESETS: Dict[str, Dict[str, Any]] = {
    "fast": {
        "n_bootstrap": 100,
        "n_permutations": 200,
        "n_mc_paths": 100,
        "window_sizes": [50, 80],
        "n_scan_permutations": 200,
    },
    "robust": {
        "n_bootstrap": 300,
        "n_permutations": 500,
        "n_mc_paths": 200,
        "window_sizes": [50, 80, 120],
        "n_scan_permutations": 500,
    },
    "full": {
        "n_bootstrap": 1000,
        "n_permutations": 1000,
        "n_mc_paths": 400,
        "window_sizes": [50, 80, 120, 180, 220],
        "n_scan_permutations": 1000,
    },
}


@dataclass
class UniversalConfig:
    """Centralized configuration for UNIVERSAL_LOG analyses."""

    seed: int = 42
    mode: str = "fast"
    horizons: List[int] = field(default_factory=lambda: list(range(1, 13)))
    n_bootstrap: int = 200
    n_permutations: int = 500
    n_mc_paths: int = 200
    window_sizes: List[int] = field(default_factory=lambda: [50, 80, 120])
    min_group: int = 4
    min_length: int = 50
    harmonic_ablation: bool = False
    harmonic_prob: float = 0.45
    use_relative_windows: bool = True
    n_scan_permutations: int = 200
    n_scan_refined_permutations: int = 2000
    min_analyses_for_prediction: int = 5
    run_complexity_diagnostics: bool = False  # Exploratory 29 estimators + L5/L7 sliding interactions
    run_estimator_validation: bool = False
    detection_mode: str = "auto"  # "auto" (mean / detrended_jump / slope / vol) or "mean"
    grade_rules: Dict[str, Any] = field(
        default_factory=lambda: {
            "min_cohen_d": 0.5,
            "max_p_perm": 0.05,
            "min_stability": 0.60,
            "min_walkforward_f1": 0.50,
        }
    )
    output_dir: str = "./results_v7"
    verbose: bool = True

    def to_dict(self) -> Dict[str, Any]:
        """Serialize configuration to a plain dictionary."""
        return asdict(self)

    @classmethod
    def from_mode(cls, mode: str = "fast", **overrides: Any) -> "UniversalConfig":
        """Create a UniversalConfig pre-populated with MODE_PRESETS[mode] plus overrides."""
        params: Dict[str, Any] = dict(MODE_PRESETS.get(mode, {}))
        params.update(overrides)
        params["mode"] = mode
        return cls(**params)


def log_message(cfg: UniversalConfig, msg: str) -> None:
    """Print log line when verbose mode is enabled."""
    if cfg.verbose:
        print(msg, flush=True)

## Module `data.py`

In [ ]:
"""Data loading, validation, numeric guards, and synthetic demo generators."""


import math
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence

import numpy as np
import pandas as pd


def _safe_float(x: Any, default: float = float("nan")) -> float:
    """Convert a scalar to float, returning default if non-finite or unconvertible."""
    try:
        v = float(x)
        return v if math.isfinite(v) else default
    except Exception:
        return default


def _finite(arr: Any) -> np.ndarray:
    """Return a 1D float64 NumPy array containing only finite elements."""
    a = np.asarray(arr, dtype=float).ravel()
    return a[np.isfinite(a)]


@dataclass
class UniversalDataset:
    """Domain-agnostic container for ordered control, target, and auxiliary fluxes."""

    control: np.ndarray
    control_name: str
    target: np.ndarray
    target_name: str
    fluxes: Dict[str, np.ndarray]
    flux_names: List[str]
    n: int
    control_monotone: bool
    source_hash: str = ""
    source_name: str = ""
    dropped_rows: int = 0
    metadata: Dict[str, Any] = field(default_factory=dict)


def load_universal_dataset(
    df: pd.DataFrame,
    control: Optional[Any] = None,
    target: Optional[Any] = None,
    fluxes: Optional[Sequence[Any]] = None,
    source_name: str = "dataframe",
    source_hash: str = "",
) -> UniversalDataset:
    """Infer control, target, and auxiliary fluxes with zero domain priors.

    - Control: explicitly provided, else first strictly monotonic numeric column,
      else synthetic row index ``__index__`` (if >= 2 numeric columns exist or
      if target is explicitly provided with >= 1 numeric column).
    - Target: explicitly provided, else first numeric column != control.
    - Fluxes: all numeric columns != control (normalized-name match excluded).
    - Rows with NaN/Inf in control or target are dropped explicitly and logged,
      never silently imputed.
    """
    if df is None or len(df) == 0:
        raise ValueError("Input DataFrame is empty.")

    numeric = df.select_dtypes(include=[np.number]).columns.tolist()
    if control is None and target is None and len(numeric) < 2:
        raise ValueError(f"At least 2 numeric columns are required, found {numeric}.")
    if len(numeric) < 1:
        raise ValueError("At least 1 numeric column is required.")

    if control is not None and control not in df.columns:
        raise ValueError(f"Control column '{control}' not found. Available: {list(df.columns)}")

    control_name = control
    if control_name is None:
        mono = [
            c
            for c in numeric
            if len(_finite(df[c].values)) >= 2
            and bool(np.all(np.diff(_finite(df[c].values)) > 0))
        ]
        control_name = mono[0] if mono else None

    if control_name is None:
        control_name = "__index__"
        control_values = np.arange(len(df), dtype=float)
    else:
        control_values = pd.to_numeric(df[control_name], errors="coerce").to_numpy(dtype=float)

    if target is not None and target not in df.columns:
        raise ValueError(f"Target column '{target}' not found. Available: {list(df.columns)}")

    if target is not None:
        target_name = target
    else:
        target_name = next((c for c in numeric if c != control_name), numeric[0])

    if target_name in df.columns:
        target_values = pd.to_numeric(df[target_name], errors="coerce").to_numpy(dtype=float)
    else:
        target_values = control_values.copy()

    def _norm(s: Any) -> str:
        return "".join(ch for ch in str(s).lower() if ch.isalnum())

    ctrl_norm = _norm(control_name)
    if fluxes is None:
        flux_names = [c for c in numeric if _norm(c) != ctrl_norm]
    else:
        flux_names = [c for c in fluxes if c in df.columns and _norm(c) != ctrl_norm]

    order = np.argsort(control_values, kind="stable")
    c_sorted = control_values[order]
    y_sorted = target_values[order]
    mask = np.isfinite(c_sorted) & np.isfinite(y_sorted)
    n_drop = int((~mask).sum())
    if n_drop:
        print(
            f"[DATA] Dropped {n_drop}/{len(df)} rows with non-finite control/target. "
            "No silent imputation performed.",
            flush=True,
        )

    c_sorted = c_sorted[mask]
    y_sorted = y_sorted[mask]
    control_monotone = bool(len(c_sorted) >= 2 and np.all(np.diff(c_sorted) > 0))

    fluxes_dict: Dict[str, np.ndarray] = {}
    for name in flux_names:
        arr = pd.to_numeric(df[name], errors="coerce").to_numpy(dtype=float)
        fluxes_dict[str(name)] = arr[order][mask]

    if str(target_name) not in fluxes_dict and target_name in df.columns:
        fluxes_dict[str(target_name)] = y_sorted

    return UniversalDataset(
        control=c_sorted,
        control_name=str(control_name),
        target=y_sorted,
        target_name=str(target_name),
        fluxes=fluxes_dict,
        flux_names=list(fluxes_dict.keys()),
        n=int(len(c_sorted)),
        control_monotone=control_monotone,
        source_hash=source_hash,
        source_name=source_name,
        dropped_rows=n_drop,
    )


def demo_dataset(kind: str = "logistic", n: int = 300, seed: int = 42) -> pd.DataFrame:
    """Generate a domain-agnostic synthetic dataset ('step', 'logistic', or 'null')."""
    rng = np.random.default_rng(seed)
    x = np.linspace(0.0, 100.0, n)
    if kind == "step":
        y = np.where(x < 50.0, 0.0, 1.0) + rng.normal(0.0, 0.15, n)
    elif kind == "null":
        y = rng.normal(0.0, 1.0, n)
    else:  # logistic
        y = 1.0 / (1.0 + np.exp(-(x - 50.0) / 5.0)) + rng.normal(0.0, 0.08, n)

    return pd.DataFrame(
        {
            "axis": x,
            "observable": y,
            "observable_noisy": y + rng.normal(0.0, 0.1, n),
            "pure_noise": rng.normal(0.0, 1.0, n),
        }
    )


def generate_example_csv(kind: str, path: str | Path, n: int = 300, seed: int = 42) -> Path:
    """Write a synthetic demo CSV to disk and return its Path."""
    df = demo_dataset(kind=kind, n=n, seed=seed)
    out_path = Path(path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)
    return out_path

## Module `stats.py`

In [ ]:
"""Core statistical utilities: Cohen's d, bootstrap CI, permutation tests, and stability weights."""


import math
from typing import Any, Tuple

import numpy as np



def cohen_d(a: Any, b: Any) -> float:
    """Compute standardized effect size (mean(b) - mean(a)) / pooled_std.

    Positive values indicate group ``b`` has a higher mean than group ``a``.
    Returns NaN if either group has fewer than 2 finite observations or if pooled
    standard deviation is non-finite. Returns 0.0 when both groups are constant
    with identical means.
    """
    a_fin = _finite(a)
    b_fin = _finite(b)
    if len(a_fin) < 2 or len(b_fin) < 2:
        return float("nan")

    ma = float(np.mean(a_fin))
    mb = float(np.mean(b_fin))
    va = float(np.var(a_fin, ddof=1))
    vb = float(np.var(b_fin, ddof=1))
    denom = max(len(a_fin) + len(b_fin) - 2, 1)
    pooled = math.sqrt(((len(a_fin) - 1) * va + (len(b_fin) - 1) * vb) / denom)
    if pooled == 0.0 or not math.isfinite(pooled):
        return 0.0 if ma == mb else float("nan")
    return (mb - ma) / pooled


def bootstrap_ci_cohen_d(
    a: Any,
    b: Any,
    n_boot: int = 200,
    seed: int = 42,
    alpha: float = 0.05,
) -> Tuple[float, float, float]:
    """Compute Cohen's d and its percentile bootstrap confidence interval."""
    d = cohen_d(a, b)
    if not math.isfinite(d):
        return d, float("nan"), float("nan")

    a_fin = _finite(a)
    b_fin = _finite(b)
    rng = np.random.default_rng(seed)
    na, nb = len(a_fin), len(b_fin)
    n_b = int(n_boot)

    # Vectorized bootstrap resampling
    idx_a = rng.integers(0, na, size=(n_b, na))
    idx_b = rng.integers(0, nb, size=(n_b, nb))
    samp_a = a_fin[idx_a]
    samp_b = b_fin[idx_b]

    ma = np.mean(samp_a, axis=1)
    mb = np.mean(samp_b, axis=1)
    va = np.var(samp_a, axis=1, ddof=1)
    vb = np.var(samp_b, axis=1, ddof=1)
    pooled = np.sqrt(((na - 1) * va + (nb - 1) * vb) / max(na + nb - 2, 1))

    with np.errstate(divide="ignore", invalid="ignore"):
        boots = np.where(pooled > 0.0, (mb - ma) / pooled, np.where(ma == mb, 0.0, np.nan))

    boots = boots[np.isfinite(boots)]
    if len(boots) < 10:
        return float(d), float("nan"), float("nan")

    lo, hi = np.percentile(boots, [100.0 * alpha / 2.0, 100.0 * (1.0 - alpha / 2.0)])
    return float(d), float(lo), float(hi)


def permutation_p_value(a: Any, b: Any, n_perm: int = 500, seed: int = 42) -> float:
    """Compute exact two-sided Monte Carlo permutation p-value for Cohen's d."""
    d_obs = cohen_d(a, b)
    if not math.isfinite(d_obs):
        return float("nan")

    a_fin = _finite(a)
    b_fin = _finite(b)
    pooled = np.concatenate([a_fin, b_fin])
    na = len(a_fin)
    nb = len(b_fin)
    n_tot = na + nb
    rng = np.random.default_rng(seed)
    n_p = int(n_perm)
    denom_df = max(n_tot - 2, 1)

    # Vectorized permutations via random keys + argpartition/argsort
    # Using exact pooled statistics invariant to permutation to compute Cohen's d fast:
    # Since sum_a + sum_b = S and sum2_a + sum2_b = S2, we only need sample_a!
    total_sum = float(np.sum(pooled))
    total_sum2 = float(np.sum(pooled * pooled))
    abs_obs = abs(d_obs)

    count = 0
    for _ in range(n_p):
        perm = rng.permutation(pooled)
        pa = perm[:na]
        sa = float(np.sum(pa))
        s2a = float(np.sum(pa * pa))
        sb = total_sum - sa
        s2b = total_sum2 - s2a
        ma = sa / na
        mb = sb / nb
        va = max(0.0, (s2a - sa * ma) / max(na - 1, 1))
        vb = max(0.0, (s2b - sb * mb) / max(nb - 1, 1))
        sp = math.sqrt(((na - 1) * va + (nb - 1) * vb) / denom_df)
        if sp > 0.0 and math.isfinite(sp):
            dp = (mb - ma) / sp
            if abs(dp) >= abs_obs:
                count += 1
        elif ma == mb and 0.0 >= abs_obs:
            count += 1

    return float((count + 1) / (n_p + 1))


def stability_weight(values: Any) -> float:
    """Inverse-CV stability weight: 1 / (1 + CV).

    Returns 0.0 for empty inputs and 1.0 when fewer than 2 finite values exist
    (callers needing genuine multi-point stability should check sample count).
    """
    v = _finite(values)
    if len(v) == 0:
        return 0.0
    if len(v) < 2:
        return 1.0
    sd = float(np.std(v))
    m = abs(float(np.mean(v)))
    if m == 0.0:
        return 0.5
    return float(1.0 / (1.0 + sd / m))

## Module `detection.py`

In [ ]:
"""Threshold scanning, multi-channel structural break detection, and F1/F2 cascade."""


import math
from typing import Any, Callable, Dict, List, Optional, Tuple

import numpy as np



def _gaussian_smooth_finite(y: Any, sigma: float = 2.0, truncate: float = 4.0) -> np.ndarray:
    """Apply 1D Gaussian smoothing independently across contiguous finite blocks."""
    a = np.asarray(y, dtype=float)
    out = np.full(a.shape, np.nan, dtype=float)
    good = np.isfinite(a)
    if int(np.sum(good)) == 0:
        return out

    ids = np.flatnonzero(good)
    cuts = np.where(np.diff(ids) > 1)[0] + 1
    for block in np.split(ids, cuts):
        vals = a[block]
        if len(vals) == 1:
            out[block] = vals
            continue
        radius = max(1, int(truncate * float(sigma)))
        grid = np.arange(-radius, radius + 1, dtype=float)
        kernel = np.exp(-0.5 * (grid / float(sigma)) ** 2)
        kernel /= np.sum(kernel)
        pad = np.pad(vals, (radius, radius), mode="reflect")
        out[block] = np.convolve(pad, kernel, mode="valid")
    return out


def _f1_f2(y: Any, p_coord: Any = None, gaussian_sigma: float = 2.0) -> Dict[str, Any]:
    """Compute first and second derivatives (F1, F2) of smoothed target wrt control."""
    a = np.asarray(y, dtype=float)
    x = np.arange(len(a), dtype=float) if p_coord is None else np.asarray(p_coord, dtype=float)
    if len(x) != len(a):
        raise ValueError("p_coord and y must have the same length.")

    smooth = _gaussian_smooth_finite(a, sigma=gaussian_sigma)
    f1 = np.full(a.shape, np.nan, dtype=float)
    f2 = np.full(a.shape, np.nan, dtype=float)
    good = np.isfinite(smooth) & np.isfinite(x)
    ids = np.flatnonzero(good)
    cuts = np.where(np.diff(ids) > 1)[0] + 1 if len(ids) else []
    blocks = np.split(ids, cuts) if len(ids) else []

    for block in blocks:
        if len(block) < 2:
            continue
        xb = x[block]
        if np.any(np.diff(xb) <= 0.0):
            continue
        f1b = np.gradient(smooth[block], xb, edge_order=1)
        f1[block] = f1b
        if len(block) >= 3:
            f2[block] = np.gradient(f1b, xb, edge_order=1)

    fin1 = f1[np.isfinite(f1)]
    fin2 = f2[np.isfinite(f2)]
    return {
        "status": "ok" if len(fin1) >= 2 else "insufficient_data",
        "gaussian_sigma": float(gaussian_sigma),
        "smoothing": "Gaussian, finite contiguous blocks only",
        "f1_definition": "gradient(y_smooth, p_coord)",
        "f2_definition": "gradient(F1, p_coord)",
        "f1": f1.tolist(),
        "f2": f2.tolist(),
        "f1_mean": float(np.mean(fin1)) if len(fin1) else float("nan"),
        "f1_std": float(np.std(fin1)) if len(fin1) else float("nan"),
        "f2_mean": float(np.mean(fin2)) if len(fin2) else float("nan"),
        "f2_std": float(np.std(fin2)) if len(fin2) else float("nan"),
        "n_finite_f1": int(len(fin1)),
        "n_finite_f2": int(len(fin2)),
    }


def _sliding_cohen_d_vec(target: np.ndarray, w: int) -> Tuple[np.ndarray, np.ndarray]:
    """O(n) prefix-sum vectorized sliding-window Cohen's d for centered finite array."""
    n = len(target)
    if n < 2 * w + 1 or w < 2:
        return np.array([], dtype=int), np.array([], dtype=float)

    xc = target - float(np.mean(target))
    cs1 = np.concatenate(([0.0], np.cumsum(xc, dtype=float)))
    cs2 = np.concatenate(([0.0], np.cumsum(xc * xc, dtype=float)))

    idx = np.arange(w, n - w, dtype=int)
    s_pre = cs1[idx] - cs1[idx - w]
    s_post = cs1[idx + w] - cs1[idx]
    s2_pre = cs2[idx] - cs2[idx - w]
    s2_post = cs2[idx + w] - cs2[idx]

    m_pre = s_pre / w
    m_post = s_post / w
    v_pre = np.maximum(0.0, (s2_pre - s_pre * m_pre) / (w - 1))
    v_post = np.maximum(0.0, (s2_post - s_post * m_post) / (w - 1))
    pooled = np.sqrt(0.5 * (v_pre + v_post))

    with np.errstate(divide="ignore", invalid="ignore"):
        d = np.where(
            (pooled > 0.0) & np.isfinite(pooled),
            (m_post - m_pre) / pooled,
            np.where(m_pre == m_post, 0.0, np.nan),
        )
    return idx, d


def _local_linear_stats_vec(
    target: np.ndarray,
    w: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Vectorized sliding local-linear discontinuity (d_jump) and slope change (t_slope)."""
    n = len(target)
    if n < 2 * w + 1 or w < 4:
        empty_i = np.array([], dtype=int)
        empty_f = np.array([], dtype=float)
        return empty_i, empty_f, empty_f

    win = np.lib.stride_tricks.sliding_window_view(target, w)
    t = np.arange(w, dtype=float)
    tc = t - float(np.mean(t))
    denom = float(np.sum(tc * tc))
    ym = np.mean(win, axis=1)
    b = np.sum((win - ym[:, None]) * tc[None, :], axis=1) / denom
    a = ym - b * float(np.mean(t))
    resid = win - (a[:, None] + b[:, None] * t[None, :])
    sse = np.sum(resid * resid, axis=1)

    idx = np.arange(w, n - w, dtype=int)
    il = idx - w
    ir = idx
    s2 = (sse[il] + sse[ir]) / max(2 * w - 4, 1)
    sd = np.sqrt(np.maximum(s2, 1e-18))

    jump = a[ir] - (a[il] + b[il] * w)
    d_jump = jump / sd
    se_slope = sd * math.sqrt(2.0 / denom)
    t_slope = (b[ir] - b[il]) / np.maximum(se_slope, 1e-18)
    return idx, d_jump, t_slope


def _scan_max_d(
    control: Any,
    target: Any,
    window: int = 80,
    min_group: int = 4,
) -> Tuple[float, Optional[int]]:
    """Fast scan returning signed Cohen's d of maximum |d| and its index."""
    y = np.asarray(target, dtype=float)
    n = len(y)
    if n < 2 * min_group + 4:
        return float("nan"), None

    w = int(max(min_group, min(window, max(min_group, n // 3))))
    if np.all(np.isfinite(y)):
        idx, d_arr = _sliding_cohen_d_vec(y, w)
        finite_mask = np.isfinite(d_arr)
        if not np.any(finite_mask):
            return float("nan"), None
        valid_idx = idx[finite_mask]
        valid_d = d_arr[finite_mask]
        pos = int(np.argmax(np.abs(valid_d)))
        return float(valid_d[pos]), int(valid_idx[pos])

    # Fallback when NaNs remain in target
    cands = list(range(w, n - w))
    if len(cands) > 400:
        cands = sorted(
            {
                int(x)
                for x in np.linspace(w, n - w - 1, 400).astype(int)
                if w <= int(x) < n - w
            }
        )
    best_d, best_idx = float("nan"), None
    for i in cands:
        pre, post = y[max(0, i - w) : i], y[i : i + w]
        d = cohen_d(pre, post)
        if math.isfinite(d) and (best_idx is None or abs(d) > abs(best_d)):
            best_d, best_idx = float(d), int(i)
    return best_d, best_idx


def _block_permute(
    arr: np.ndarray,
    block_len: int,
    rng: np.random.Generator,
) -> np.ndarray:
    """Contiguous block permutation preserving local autocorrelation up to block_len."""
    n = len(arr)
    b = max(1, int(block_len))
    if b <= 1 or b >= n:
        return rng.permutation(arr)
    blocks = [arr[i : i + b] for i in range(0, n, b)]
    order = rng.permutation(len(blocks))
    return np.concatenate([blocks[i] for i in order])[:n]


def _refine_scan_p(
    screen_fn: Callable[[int, np.random.Generator], float],
    n_screen: int,
    n_refined: int,
    seed: int,
    channel_salt: int,
) -> Tuple[float, Optional[float], float, int]:
    """Screen with n_screen permutations; if p_screen <= 0.10, run an independent n_refined test."""
    rng1 = np.random.default_rng(np.random.SeedSequence([int(seed), int(channel_salt), 1]))
    p_screen = float(screen_fn(int(n_screen), rng1))
    if math.isfinite(p_screen) and p_screen <= 0.10 and int(n_refined) > int(n_screen):
        rng2 = np.random.default_rng(np.random.SeedSequence([int(seed), int(channel_salt), 2000]))
        p_refined = float(screen_fn(int(n_refined), rng2))
        return p_screen, p_refined, p_refined, int(n_refined)
    return p_screen, None, p_screen, int(n_screen)


def scan_corrected_p(
    control: Any,
    target: Any,
    window: int = 80,
    min_group: int = 4,
    n_perm: int = 200,
    seed: int = 42,
    rng: Optional[np.random.Generator] = None,
) -> Tuple[float, float, float]:
    """Full-scan permutation-corrected p-value eliminating search-max selection bias."""
    y = np.asarray(target, dtype=float)
    d_obs, _ = _scan_max_d(control, y, window=window, min_group=min_group)
    if not math.isfinite(d_obs):
        return float("nan"), float("nan"), float("nan")

    r_gen = rng if rng is not None else np.random.default_rng(seed)
    null_max: List[float] = []
    for _ in range(int(n_perm)):
        perm = r_gen.permutation(y)
        d_perm, _ = _scan_max_d(control, perm, window=window, min_group=min_group)
        if math.isfinite(d_perm):
            null_max.append(abs(d_perm))

    if not null_max:
        return float("nan"), float("nan"), float("nan")

    null_arr = np.asarray(null_max, dtype=float)
    p_corr = (float(np.sum(null_arr >= abs(d_obs))) + 1.0) / (len(null_arr) + 1.0)
    return float(p_corr), float(np.median(null_arr)), float(np.percentile(null_arr, 95))


def scan_threshold(
    control: Any,
    target: Any,
    window: int = 80,
    min_group: int = 4,
    seed: int = 42,
    n_boot: int = 100,
    n_perm: int = 200,
    n_scan_perm: int = 200,
    compute_scan_p: bool = True,
    n_refined_perm: int = 2000,
) -> Dict[str, Any]:
    """Scan every valid interior split using a symmetric local window [idx-w, idx) vs [idx, idx+w)."""
    y = np.asarray(target, dtype=float)
    x = np.asarray(control, dtype=float)
    n = len(y)
    if n < 2 * min_group + 4:
        return {"status": "insufficient_data", "reason": f"n={n} is too small"}

    w = int(max(min_group, min(window, max(min_group, n // 3))))
    d_best, best_idx = _scan_max_d(x, y, window=w, min_group=min_group)
    if best_idx is None or not math.isfinite(d_best):
        return {"status": "no_finite_threshold"}

    pre_win = y[max(0, best_idx - w) : best_idx]
    post_win = y[best_idx : best_idx + w]
    d, lo, hi = bootstrap_ci_cohen_d(pre_win, post_win, n_boot=n_boot, seed=seed)
    p_local = permutation_p_value(pre_win, post_win, n_perm=n_perm, seed=seed)
    d_global = cohen_d(y[:best_idx], y[best_idx:])
    p_global = permutation_p_value(y[:best_idx], y[best_idx:], n_perm=n_perm, seed=seed + 7)

    p_screen: Optional[float] = None
    p_refined: Optional[float] = None
    n_perm_used = 0
    if compute_scan_p:
        stage_stats: Dict[int, Tuple[float, float]] = {}

        def _lvl_scan(m_perm: int, r_gen: np.random.Generator) -> float:
            pc, med, p95 = scan_corrected_p(
                x, y, window=w, min_group=min_group, n_perm=m_perm, seed=seed + 11, rng=r_gen
            )
            stage_stats[m_perm] = (med, p95)
            return pc

        p_screen, p_refined, p_scan_corr, n_perm_used = _refine_scan_p(
            _lvl_scan, int(n_scan_perm), int(n_refined_perm), seed, channel_salt=11
        )
        p_scan_null_med, p_scan_null_p95 = stage_stats.get(n_perm_used, (float("nan"), float("nan")))
    else:
        p_scan_corr, p_scan_null_med, p_scan_null_p95 = float("nan"), float("nan"), float("nan")

    p_decide = (
        p_scan_corr
        if math.isfinite(p_scan_corr)
        else (max(p_local, p_global) if math.isfinite(p_global) else p_local)
    )

    return {
        "status": "ok",
        "detector": "level_shift",
        "idx": int(best_idx),
        "control_value": float(x[best_idx]),
        "cohen_d": float(d),
        "n_pre": int(len(pre_win)),
        "n_post": int(len(post_win)),
        "mean_pre": float(np.mean(pre_win)),
        "mean_post": float(np.mean(post_win)),
        "window_used": int(w),
        "window": int(window),
        "cohen_d_ci_low": float(lo),
        "cohen_d_ci_high": float(hi),
        "p_permutation": float(p_decide),
        "p_permutation_local": float(p_local),
        "p_permutation_global": float(p_global),
        "p_permutation_scan_corrected": float(p_scan_corr),
        "p_screen": p_screen,
        "p_refined": p_refined,
        "n_scan_perm_used": int(n_perm_used),
        "p_scan_null": float(p_decide),
        "p_channel_raw": float(p_decide),
        "null_model": "target_permutation_scan",
        "p_scan_null_median_abs_d": float(p_scan_null_med),
        "p_scan_null_p95_abs_d": float(p_scan_null_p95),
        "cohen_d_global": float(d_global),
    }


def _ar1_lag1(seg: np.ndarray, detrend: bool = True) -> float:
    """Estimate lag-1 autocorrelation inside a segment (optionally linearly detrended)."""
    fin = seg[np.isfinite(seg)]
    if len(fin) < 12 or np.std(fin) == 0.0:
        return 0.0
    if detrend:
        t = np.arange(len(fin), dtype=float)
        slope, intercept = np.polyfit(t, fin, 1)
        r = fin - (intercept + slope * t)
    else:
        r = fin
    if np.std(r) == 0.0:
        return 0.0
    val = float(np.corrcoef(r[:-1], r[1:])[0, 1])
    return val if math.isfinite(val) else 0.0


def within_regime_ar1(y: np.ndarray, idx: int) -> float:
    """Maximum-clamped median lag-1 residual autocorrelation inside pre and post regimes."""
    phis = []
    for seg in (y[:idx], y[idx:]):
        if len(seg) >= 12:
            phis.append(_ar1_lag1(seg, detrend=True))
    if not phis:
        return 0.0
    return float(max(0.0, min(0.98, float(np.median(phis)))))


def _volatility_deviation_series(y: np.ndarray) -> np.ndarray:
    """Scale-normalized absolute first-difference deviations (domain-free, invariant to level scaling)."""
    dy = np.diff(y)
    sd_y = float(np.std(y)) if float(np.std(y)) > 0.0 else 1.0
    denom = np.maximum(0.5 * (np.abs(y[:-1]) + np.abs(y[1:])), sd_y)
    rel = dy / denom
    return np.abs(rel - float(np.median(rel)))


def _deseasonalize_fft_dominant(
    y: np.ndarray,
    min_dom_frac: float = 0.30,
) -> Optional[Tuple[np.ndarray, float, int, List[int]]]:
    """Domain-free spectral deseasonalization removing dominant narrowband cycle (>= 3 cycles).

    Returns ``(y_deseasonalized, dominant_power_fraction, fundamental_bin, removed_bins)``
    when a dominant periodic component accounts for >= ``min_dom_frac`` of detrended AC power,
    else ``None``.
    """
    n = len(y)
    if n < 24 or np.std(y) == 0.0 or not np.all(np.isfinite(y)):
        return None
    t = np.arange(n, dtype=float)
    slope, intercept = np.polyfit(t, y, 1)
    res = y - (intercept + slope * t)
    spec = np.fft.rfft(res)
    power = np.abs(spec) ** 2
    if len(power) <= 5:
        return None
    total_ac = float(np.sum(power[1:]))
    if total_ac <= 0.0:
        return None

    k_dom = 3 + int(np.argmax(power[3:]))
    med_p = float(np.median(power[3:]))
    cluster_bins = [
        k
        for k in (k_dom - 1, k_dom, k_dom + 1)
        if 3 <= k < len(spec) and (k == k_dom or power[k] > 10.0 * med_p)
    ]
    dom_frac = float(sum(power[k] for k in cluster_bins) / total_ac)
    if dom_frac < min_dom_frac:
        return None

    spec_seas = np.zeros_like(spec)
    removed: List[int] = []
    for m in (1, 2, 3, 4):
        center = m * k_dom
        for kb in (center - 1, center, center + 1):
            if 3 <= kb < len(spec) and (kb in cluster_bins or (m > 1 and power[kb] > 10.0 * med_p)):
                spec_seas[kb] = spec[kb]
                removed.append(kb)

    seasonal = np.fft.irfft(spec_seas, n=n)
    return y - seasonal, dom_frac, k_dom, sorted(set(removed))


def scan_structural_channels(
    control: Any,
    target: Any,
    w_small: int,
    w_large: int,
    min_group: int = 4,
    n_perm: int = 200,
    seed: int = 42,
    n_refined_perm: int = 2000,
) -> Dict[str, Any]:
    """Evaluate complementary domain-free break channels: jump, slope, volatility, and FFT deseasonalization."""
    y = np.asarray(target, dtype=float)
    x = np.asarray(control, dtype=float)
    n = len(y)
    out: Dict[str, Any] = {}

    # 1. Detrended jump (local linear discontinuity at boundary)
    t_idx = np.arange(n, dtype=float)
    if n >= 2 * max(4, w_small) + 2 and np.all(np.isfinite(y)):
        w_j = int(max(6, min(w_small, n // 3)))
        idx_j, dj, _ = _local_linear_stats_vec(y, w_j)
        if len(dj):
            pos_j = int(np.argmax(np.abs(dj)))
            best_i = int(idx_j[pos_j])
            best_dj = float(dj[pos_j])
            t_jump = float(abs(best_dj) / math.sqrt(8.0 / w_j))
            phi_j = within_regime_ar1(y, best_i)
            # Scan-corrected residual block-permutation null reconstructing base_j + permuted residual blocks
            deg_j = min(3, max(1, n // 30))
            base_j = np.polyval(np.polyfit(t_idx, y, deg_j), t_idx)
            res_j = y - base_j
            bl_j = int(max(2, min(8, w_j // 4)))

            def _dj_scan_p(m_perm: int, r_gen: np.random.Generator) -> float:
                null_j: List[float] = []
                for _ in range(int(m_perm)):
                    rb = _block_permute(res_j, bl_j, r_gen)
                    _, sim_dj, _ = _local_linear_stats_vec(base_j + rb, w_j)
                    if len(sim_dj):
                        null_j.append(float(np.max(np.abs(sim_dj))))
                return (
                    (1.0 + float(np.sum(np.asarray(null_j) >= abs(best_dj)))) / (len(null_j) + 1.0)
                    if null_j
                    else float("nan")
                )

            p_scr_j, p_ref_j, p_j, n_used_j = _refine_scan_p(
                _dj_scan_p, int(n_perm), int(n_refined_perm), seed, channel_salt=301
            )
            out["detrended_jump"] = {
                "status": "ok",
                "idx": best_i,
                "control_value": float(x[best_i]),
                "effect_d": best_dj,
                "t_stat": t_jump,
                "window_used": w_j,
                "phi_within": phi_j,
                "p_residual_block_perm_scan": float(p_j),
                "p_screen": p_scr_j,
                "p_refined": p_ref_j,
                "n_scan_perm_used": int(n_used_j),
                "p_scan_null": float(p_j),
                "p_channel_raw": float(p_j),
                "null_model": "residual_block_permutation_scan",
                "null_block_len": bl_j,
            }
        else:
            out["detrended_jump"] = {"status": "insufficient_data"}
    else:
        out["detrended_jump"] = {"status": "insufficient_data"}

    # 2. Slope break (local linear slope difference on wider window)
    if n >= 2 * max(8, w_large) + 2 and np.all(np.isfinite(y)):
        w_s = int(max(8, min(w_large, n // 3)))
        idx_s, _, ts = _local_linear_stats_vec(y, w_s)
        if len(ts):
            pos_s = int(np.argmax(np.abs(ts)))
            best_is = int(idx_s[pos_s])
            best_ts = float(ts[pos_s])
            phi_s = within_regime_ar1(y, best_is)
            d_slope_equiv = float(best_ts / math.sqrt(w_s))
            # Scan-corrected residual block-permutation null under single global-line H0
            res_s = y - np.polyval(np.polyfit(t_idx, y, 1), t_idx)
            bl_s = int(max(2, min(8, w_s // 4)))

            def _sb_scan_p(m_perm: int, r_gen: np.random.Generator) -> float:
                null_s: List[float] = []
                for _ in range(int(m_perm)):
                    rb = _block_permute(res_s, bl_s, r_gen)
                    _, _, sim_ts = _local_linear_stats_vec(rb, w_s)
                    if len(sim_ts):
                        null_s.append(float(np.max(np.abs(sim_ts))))
                return (
                    (1.0 + float(np.sum(np.asarray(null_s) >= abs(best_ts)))) / (len(null_s) + 1.0)
                    if null_s
                    else float("nan")
                )

            p_scr_s, p_ref_s, p_s, n_used_s = _refine_scan_p(
                _sb_scan_p, int(n_perm), int(n_refined_perm), seed, channel_salt=302
            )
            out["slope_break"] = {
                "status": "ok",
                "idx": best_is,
                "control_value": float(x[best_is]),
                "t_stat": best_ts,
                "cohen_d_equiv": d_slope_equiv,
                "window_used": w_s,
                "phi_within": phi_s,
                "p_residual_block_perm_scan": float(p_s),
                "p_screen": p_scr_s,
                "p_refined": p_ref_s,
                "n_scan_perm_used": int(n_used_s),
                "p_scan_null": float(p_s),
                "p_channel_raw": float(p_s),
                "null_model": "residual_block_permutation_scan",
                "null_block_len": bl_s,
            }
        else:
            out["slope_break"] = {"status": "insufficient_data"}
    else:
        out["slope_break"] = {"status": "insufficient_data"}

    # 3. Volatility / dispersion shift on scale-normalized first differences
    if n >= 2 * max(8, w_large) + 4 and np.all(np.isfinite(y)):
        vy = _volatility_deviation_series(y)
        w_v = int(max(min_group, min(w_large, len(vy) // 3)))
        d_vol, idx_v = _scan_max_d(x[1:], vy, window=w_v, min_group=min_group)
        if idx_v is not None and math.isfinite(d_vol):
            real_idx = int(min(n - 1, idx_v + 1))

            def _vol_scan_p(m_perm: int, r_gen: np.random.Generator) -> float:
                pv, _, _ = scan_corrected_p(
                    x[1:], vy, window=w_v, min_group=min_group, n_perm=m_perm, seed=seed + 307, rng=r_gen
                )
                return pv

            p_scr_v, p_ref_v, p_vol, n_used_v = _refine_scan_p(
                _vol_scan_p, int(n_perm), int(n_refined_perm), seed, channel_salt=307
            )
            out["volatility_shift"] = {
                "status": "ok",
                "idx": real_idx,
                "idx_vy": int(idx_v),
                "control_value": float(x[real_idx]),
                "cohen_d": float(d_vol),
                "window_used": w_v,
                "p_permutation_scan_corrected": float(p_vol),
                "p_screen": p_scr_v,
                "p_refined": p_ref_v,
                "n_scan_perm_used": int(n_used_v),
                "p_scan_null": float(p_vol),
                "p_channel_raw": float(p_vol),
                "null_model": "target_permutation_scan",
            }
        else:
            out["volatility_shift"] = {"status": "no_finite_threshold"}
    else:
        out["volatility_shift"] = {"status": "insufficient_data"}

    # 4. Spectral deseasonalized level shift (when strong periodic oscillation masks shifts)
    deseas = _deseasonalize_fft_dominant(y) if (n >= 24 and np.all(np.isfinite(y))) else None
    if deseas is not None:
        y_des, dom_frac, k_dom, removed_bins = deseas
        w_d = int(max(min_group, min(w_small, n // 3)))
        d_des, idx_des = _scan_max_d(x, y_des, window=w_d, min_group=min_group)
        if idx_des is not None and math.isfinite(d_des):
            def _des_scan_p(m_perm: int, r_gen: np.random.Generator) -> float:
                pd_val, _, _ = scan_corrected_p(
                    x, y_des, window=w_d, min_group=min_group, n_perm=m_perm, seed=seed + 311, rng=r_gen
                )
                return pd_val

            p_scr_d, p_ref_d, p_des, n_used_d = _refine_scan_p(
                _des_scan_p, int(n_perm), int(n_refined_perm), seed, channel_salt=311
            )
            out["deseasonalized_shift"] = {
                "status": "ok",
                "idx": int(idx_des),
                "control_value": float(x[int(idx_des)]),
                "cohen_d": float(d_des),
                "window_used": w_d,
                "p_permutation_scan_corrected": float(p_des),
                "p_screen": p_scr_d,
                "p_refined": p_ref_d,
                "n_scan_perm_used": int(n_used_d),
                "p_scan_null": float(p_des),
                "p_channel_raw": float(p_des),
                "null_model": "target_permutation_scan",
                "dominant_power_fraction": float(dom_frac),
                "fundamental_fft_bin": int(k_dom),
                "removed_fft_bins": removed_bins,
            }
        else:
            out["deseasonalized_shift"] = {"status": "no_finite_threshold"}
    else:
        out["deseasonalized_shift"] = {"status": "not_strongly_periodic"}

    return out


def cascade_5_thresholds(
    control: Any,
    target: Any,
    central_idx: Optional[int],
    seed: int = 42,
    gaussian_sigma: float = 2.0,
) -> Dict[str, Any]:
    """Derive 5-point F1/F2 cascade (T-2, T-1, T0, T+1, T+2) around central threshold T0."""
    y = np.asarray(target, dtype=float)
    x = np.asarray(control, dtype=float)
    n = len(y)
    if n < 20 or len(x) != n or central_idx is None or not (0 <= int(central_idx) < n):
        return {"status": "insufficient_data"}

    derivs = _f1_f2(y, x, gaussian_sigma=gaussian_sigma)
    f1 = np.asarray(derivs["f1"], dtype=float)
    f2 = np.asarray(derivs["f2"], dtype=float)
    ci = int(central_idx)

    def _peak(values: np.ndarray, lo: int, hi: int, order_name: str) -> Dict[str, Any]:
        lo_c = max(0, int(lo))
        hi_c = min(n, int(hi))
        if hi_c - lo_c < 3:
            return {
                "idx": None,
                "control_value": float("nan"),
                "metric_value": float("nan"),
                "derivative_order": order_name,
                "note": "insufficient segment",
            }
        ix = np.arange(lo_c, hi_c)
        ix = ix[np.isfinite(values[ix])]
        if len(ix) == 0:
            return {
                "idx": None,
                "control_value": float("nan"),
                "metric_value": float("nan"),
                "derivative_order": order_name,
                "note": "derivative unavailable",
            }
        j = int(ix[np.argmax(np.abs(values[ix]))])
        return {
            "idx": j,
            "control_value": float(x[j]),
            "metric_value": float(values[j]),
            "absolute_metric": float(abs(values[j])),
            "derivative_order": order_name,
        }

    t0 = {
        "idx": ci,
        "control_value": float(x[ci]),
        "f1_velocity": float(f1[ci]) if math.isfinite(f1[ci]) else float("nan"),
        "f2_acceleration": float(f2[ci]) if math.isfinite(f2[ci]) else float("nan"),
        "derivative_order": "F1+F2",
    }
    return {
        "status": "ok",
        "T_minus2": _peak(f2, 3, ci, "F2"),
        "T_minus1": _peak(f1, 3, ci, "F1"),
        "T0": t0,
        "T_plus1": _peak(f1, ci + 1, n - 3, "F1"),
        "T_plus2": _peak(f2, ci + 1, n - 3, "F2"),
        "derivative_specification": f"{derivs['f1_definition']}; {derivs['f2_definition']}",
        "smoothing": "Gaussian",
        "gaussian_sigma": float(gaussian_sigma),
    }


def scan_multivariate_channels(
    control: Any,
    fluxes: Dict[str, np.ndarray],
    window: int = 40,
    min_group: int = 4,
    n_perm: int = 200,
    seed: int = 42,
) -> Dict[str, Any]:
    """Composite multivariate break scan across all fluxes via P2 stability weighting.

    Preserves instantaneous cross-flux correlations by permuting the time index
    jointly across all fluxes when computing the scan-corrected p-value.
    """
    x = np.asarray(control, dtype=float)
    n = len(x)
    usable: List[Tuple[str, np.ndarray]] = []
    for name, arr in fluxes.items():
        a = np.asarray(arr, dtype=float)
        if len(a) == n and np.all(np.isfinite(a)) and np.std(a) > 0.0:
            usable.append((name, a))

    if len(usable) < 2:
        return {
            "status": "univariate_only",
            "n_fluxes_usable": len(usable),
            "note": "at least 2 finite non-constant fluxes required for multivariate composite scan",
        }

    if n < 2 * min_group + 4:
        return {"status": "insufficient_data", "n_fluxes_usable": len(usable)}

    w = int(max(min_group, min(window, max(min_group, n // 3))))
    d_curves: List[np.ndarray] = []
    weights: List[float] = []
    idx_ref: Optional[np.ndarray] = None

    for _, arr in usable:
        idx_c, d_c = _sliding_cohen_d_vec(arr, w)
        if len(d_c) == 0:
            continue
        idx_ref = idx_c
        d_clean = np.where(np.isfinite(d_c), d_c, 0.0)
        d_curves.append(d_clean)
        weights.append(stability_weight(np.abs(d_clean)))

    if idx_ref is None or len(d_curves) < 2:
        return {"status": "insufficient_data", "n_fluxes_usable": len(usable)}

    w_arr = np.asarray(weights, dtype=float)
    w_sum = float(np.sum(w_arr))
    w_norm = w_arr / w_sum if w_sum > 0.0 else np.full(len(w_arr), 1.0 / len(w_arr))

    d_mat = np.vstack(d_curves)  # (K, n_cands)
    comp = np.sqrt(np.sum((w_norm[:, None]) * (d_mat ** 2), axis=0))
    pos = int(np.argmax(comp))
    best_idx = int(idx_ref[pos])
    obs_comp = float(comp[pos])

    per_flux_at_t0 = {
        name: float(d_mat[k, pos]) for k, (name, _) in enumerate(usable)
    }

    rng = np.random.default_rng(seed + 404)
    null_max: List[float] = []
    for _ in range(int(n_perm)):
        perm = rng.permutation(n)
        d_perm_list = []
        for _, arr in usable:
            _, dp = _sliding_cohen_d_vec(arr[perm], w)
            d_perm_list.append(np.where(np.isfinite(dp), dp, 0.0))
        d_pmat = np.vstack(d_perm_list)
        comp_p = np.sqrt(np.sum((w_norm[:, None]) * (d_pmat ** 2), axis=0))
        null_max.append(float(np.max(comp_p)))

    null_arr = np.asarray(null_max, dtype=float)
    p_corr = (
        (1.0 + float(np.sum(null_arr >= obs_comp))) / (len(null_arr) + 1.0)
        if len(null_arr)
        else float("nan")
    )

    return {
        "status": "ok",
        "idx": best_idx,
        "control_value": float(x[best_idx]),
        "composite_effect": obs_comp,
        "p_permutation_scan_corrected": float(p_corr),
        "window_used": w,
        "n_fluxes": len(usable),
        "flux_weights": {usable[k][0]: float(w_norm[k]) for k in range(len(usable))},
        "per_flux_cohen_d_at_threshold": per_flux_at_t0,
    }


def detect_multiple_thresholds(
    control: Any,
    target: Any,
    min_group: int = 4,
    min_seg_len: int = 40,
    min_cohen_d: float = 0.5,
    max_p_perm: float = 0.05,
    max_thresholds: int = 5,
    n_perm: int = 100,
    seed: int = 42,
) -> Dict[str, Any]:
    """Iterative binary segmentation finding multiple significant mean-shift thresholds."""
    x = np.asarray(control, dtype=float)
    y = np.asarray(target, dtype=float)
    n = len(y)
    if n < min_seg_len:
        return {"status": "insufficient_data", "n_thresholds": 0, "thresholds": []}

    found: List[Dict[str, Any]] = []
    queue: List[Tuple[int, int]] = [(0, n)]
    step_seed = seed + 606

    while queue and len(found) < int(max_thresholds):
        # Process longest candidate segment first
        queue.sort(key=lambda seg: seg[1] - seg[0], reverse=True)
        lo, hi = queue.pop(0)
        seg_len = hi - lo
        if seg_len < min_seg_len:
            continue
        w = int(max(min_group, min(seg_len // 5, seg_len // 3)))
        if seg_len < 2 * w + 2:
            continue

        x_seg, y_seg = x[lo:hi], y[lo:hi]
        d_seg, local_idx = _scan_max_d(x_seg, y_seg, window=w, min_group=min_group)
        if local_idx is None or not math.isfinite(d_seg) or abs(d_seg) < min_cohen_d:
            continue

        p_seg, _, _ = scan_corrected_p(
            x_seg,
            y_seg,
            window=w,
            min_group=min_group,
            n_perm=n_perm,
            seed=step_seed,
        )
        step_seed += 1
        if not math.isfinite(p_seg) or p_seg > max_p_perm:
            continue

        global_idx = int(lo + local_idx)
        found.append(
            {
                "idx": global_idx,
                "control_value": float(x[global_idx]),
                "cohen_d": float(d_seg),
                "p_permutation_scan_corrected": float(p_seg),
                "window_used": w,
                "segment_start_idx": int(lo),
                "segment_end_idx": int(hi),
            }
        )
        if global_idx - lo >= min_seg_len:
            queue.append((lo, global_idx))
        if hi - global_idx >= min_seg_len:
            queue.append((global_idx, hi))

    found.sort(key=lambda item: item["control_value"])
    return {
        "status": "ok",
        "n_thresholds": len(found),
        "thresholds": found,
    }

## Module `estimators.py`

In [ ]:
"""29 complexity, entropy, fractal, correlation, and topological estimators.

Includes bug fixes and vectorization over v7.1.5:
- Fixed ``est_lz_complexity`` (Kaspar-Schuster LZ76 algorithm; previous ``i > 0`` bug
  caused constant output ``log2(n)/n`` for every signal).
- Fixed ``est_boxcount`` (true 2D Minkowski-Bouligand polyline box counting in [1, 2]).
- Fixed ``est_higuchi`` (standard ``N_m = len(idx) - 1`` normalization and constant guard).
- Fixed ``est_hurst_rs`` (averaged R/S across all non-overlapping sub-blocks per scale).
- Fixed ``est_lyapunov_rosenstein`` (added Theiler temporal exclusion window).
- Removed dead duplicate ``est_danco`` definition; kept explicit NaN placeholder.
- Vectorized ``_mfdfa_q_spectrum``, ``est_dfa``, ``est_local_pca``, ``est_perm_entropy``,
  ``est_sample_entropy`` (via cKDTree), and shared visibility-graph adjacency in
  ``apply_estimators()``.
- Implemented true permutation-null calibration for ``compute_tlog()`` (P8).
"""


import math
from typing import Any, Callable, Dict, List, Optional, Sequence, Set, Tuple

import numpy as np
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist, pdist



def _embed(x: Any, dim: int = 2, tau: int = 1) -> Optional[np.ndarray]:
    """Delay-embed a 1D series into shape (L, dim)."""
    xf = _finite(x)
    if len(xf) < (dim - 1) * tau + 2:
        return None
    n = len(xf)
    length = n - (dim - 1) * tau
    return np.column_stack([xf[i * tau : i * tau + length] for i in range(dim)])


def est_higuchi(x: Any, kmax: int = 10) -> float:
    """Higuchi fractal dimension estimator."""
    xf = _finite(x)
    n = len(xf)
    if n < 20 or np.ptp(xf) == 0.0:
        return float("nan")
    lk: List[float] = []
    ks_used: List[int] = []
    for k in range(1, min(kmax, n // 4) + 1):
        lm: List[float] = []
        for m in range(k):
            idx = np.arange(m, n, k)
            n_intervals = len(idx) - 1
            if n_intervals < 1:
                continue
            curve_len = float(np.sum(np.abs(np.diff(xf[idx]))))
            lm.append(curve_len * (n - 1) / (n_intervals * k * k))
        if lm:
            mean_lk = float(np.mean(lm))
            if mean_lk > 0.0:
                lk.append(mean_lk)
                ks_used.append(k)
    if len(lk) < 3:
        return float("nan")
    ks_arr = np.asarray(ks_used, dtype=float)
    return float(np.polyfit(np.log(1.0 / ks_arr), np.log(np.asarray(lk)), 1)[0])


def est_katz(x: Any) -> float:
    """Katz fractal dimension estimator."""
    xf = _finite(x)
    n = len(xf)
    if n < 10:
        return float("nan")
    length = float(np.sum(np.abs(np.diff(xf))))
    diam = float(np.max(np.abs(xf - xf[0])))
    if length <= 0.0 or diam <= 0.0:
        return float("nan")
    denom = math.log10(n) + math.log10(diam / length)
    if denom == 0.0 or not math.isfinite(denom):
        return float("nan")
    return float(math.log10(n) / denom)


def est_petrosian(x: Any) -> float:
    """Petrosian fractal dimension estimator."""
    xf = _finite(x)
    n = len(xf)
    if n < 10 or np.ptp(xf) == 0.0:
        return float("nan")
    d = np.diff(np.sign(np.diff(xf)))
    n_delta = int(np.sum(d != 0))
    if n_delta == 0:
        return float("nan")
    return float(math.log10(n) / (math.log10(n) + math.log10(n / (n + 0.4 * n_delta))))


def est_sevcik(x: Any) -> float:
    """Sevcik fractal dimension estimator."""
    xf = _finite(x)
    n = len(xf)
    if n < 10 or np.ptp(xf) == 0.0:
        return float("nan")
    xn = (xf - np.min(xf)) / np.ptp(xf)
    yn = np.linspace(0.0, 1.0, n)
    length = float(np.sum(np.sqrt(np.diff(xn) ** 2 + np.diff(yn) ** 2)))
    if length <= 0.0:
        return float("nan")
    return float(1.0 + math.log(length) / math.log(2.0 * (n - 1)))


def est_boxcount(x: Any, scales: Sequence[int] = (2, 4, 8, 16)) -> float:
    """2D Minkowski-Bouligand box-counting dimension on normalized (t, x) curve."""
    xf = _finite(x)
    n = len(xf)
    ptp = float(np.ptp(xf))
    if n < 32 or ptp == 0.0:
        return float("nan")

    xn = (xf - np.min(xf)) / ptp
    counts: List[int] = []
    valid_scales: List[int] = []

    for s in scales:
        if s >= n // 2:
            continue
        bx = np.minimum(s - 1, (np.arange(n, dtype=float) / n * s).astype(int))
        by = np.minimum(s - 1, np.floor(xn * s).astype(int))
        # For each x-column, count all y-bins spanned including vertical continuity
        # between consecutive sample points so steep jumps cover intermediate boxes.
        n_boxes = 0
        for col in range(s):
            mask = bx == col
            if not np.any(mask):
                continue
            ys = by[mask]
            lo = int(np.min(ys))
            hi = int(np.max(ys))
            idx_first = int(np.argmax(mask))
            if idx_first > 0:
                prev_y = int(by[idx_first - 1])
                lo = min(lo, prev_y)
                hi = max(hi, prev_y)
            n_boxes += hi - lo + 1
        if n_boxes > 0:
            counts.append(n_boxes)
            valid_scales.append(s)

    if len(counts) < 2:
        return float("nan")
    return float(np.polyfit(np.log(np.asarray(valid_scales, dtype=float)), np.log(np.asarray(counts, dtype=float)), 1)[0])


def est_dfa(x: Any, scales: Sequence[int] = (4, 8, 16, 32)) -> float:
    """Detrended Fluctuation Analysis (order-1, vectorized)."""
    xf = _finite(x)
    n = len(xf)
    if n < 40 or np.ptp(xf) == 0.0:
        return float("nan")
    y = np.cumsum(xf - np.mean(xf))
    fs: List[float] = []
    sc: List[int] = []
    for s in scales:
        if s >= n // 2:
            continue
        m = n // s
        if m < 2:
            continue
        segs = y[: m * s].reshape(m, s)
        t = np.arange(s, dtype=float)
        tc = t - np.mean(t)
        denom = float(np.sum(tc * tc))
        ym = np.mean(segs, axis=1, keepdims=True)
        b = np.sum((segs - ym) * tc[None, :], axis=1, keepdims=True) / denom
        a = ym - b * np.mean(t)
        resid = segs - (a + b * t[None, :])
        rms = float(np.sqrt(np.mean(resid * resid)))
        if rms > 0.0:
            fs.append(rms)
            sc.append(s)
    if len(fs) < 2:
        return float("nan")
    return float(np.polyfit(np.log(np.asarray(sc, dtype=float)), np.log(np.asarray(fs, dtype=float)), 1)[0])


def est_hurst_rs(x: Any) -> float:
    """Rescaled range (R/S) Hurst exponent averaged across non-overlapping blocks."""
    xf = _finite(x)
    n = len(xf)
    if n < 20 or np.ptp(xf) == 0.0:
        return float("nan")
    vals: List[Tuple[int, float]] = []
    for size in sorted({max(8, n // 8), max(8, n // 4), max(8, n // 2), n}):
        if size < 8 or size > n:
            continue
        n_blocks = n // size
        rs_blocks: List[float] = []
        for b in range(n_blocks):
            seg = xf[b * size : (b + 1) * size]
            s = float(np.std(seg, ddof=1))
            if s <= 0.0:
                continue
            dev = np.cumsum(seg - np.mean(seg))
            r = float(np.ptp(dev))
            if r > 0.0:
                rs_blocks.append(r / s)
        if rs_blocks:
            vals.append((size, float(np.mean(rs_blocks))))
    if len(vals) < 2:
        return float("nan")
    return float(np.polyfit(np.log([v[0] for v in vals]), np.log([v[1] for v in vals]), 1)[0])


def est_sample_entropy(x: Any, m: int = 2, r_frac: float = 0.2) -> float:
    """Sample entropy computed via cKDTree (Chebyshev metric) in C."""
    xf = _finite(x)
    n = len(xf)
    if n < 30:
        return float("nan")
    sd = float(np.std(xf, ddof=1))
    r = r_frac * sd
    if r <= 0.0 or not math.isfinite(r):
        return float("nan")

    def _phi(mm: int) -> float:
        emb = np.lib.stride_tricks.sliding_window_view(xf, mm)
        k_len = len(emb)
        if k_len < 2:
            return 0.0
        tree = cKDTree(emb)
        n_pairs = len(tree.query_pairs(r, p=np.inf))
        return (2.0 * n_pairs) / (k_len * (k_len - 1))

    a, b = _phi(m), _phi(m + 1)
    if a <= 0.0 or b <= 0.0:
        return float("nan")
    return float(-math.log(b / a))


def est_perm_entropy(x: Any, order: int = 3, tau: int = 1) -> float:
    """Normalized permutation entropy (vectorized ordinal encoding)."""
    xf = _finite(x)
    n = len(xf)
    span = (order - 1) * tau + 1
    if n < 30 or n < span:
        return float("nan")

    win = np.lib.stride_tricks.sliding_window_view(xf, span)[:, ::tau]
    ranks = np.argsort(np.argsort(win, axis=1, kind="stable"), axis=1)
    powers = (order ** np.arange(order - 1, -1, -1, dtype=np.int64))
    codes = ranks @ powers
    _, counts = np.unique(codes, return_counts=True)
    probs = counts / float(np.sum(counts))
    probs = probs[probs > 0.0]
    if len(probs) == 0:
        return float("nan")
    return float(-np.sum(probs * np.log(probs)) / math.log(math.factorial(order)))


def est_spectral_beta(x: Any) -> float:
    """Spectral exponent beta from log-log power spectrum slope."""
    xf = _finite(x)
    n = len(xf)
    if n < 32 or np.ptp(xf) == 0.0:
        return float("nan")
    spec = np.abs(np.fft.rfft(xf - np.mean(xf))) ** 2
    freq = np.fft.rfftfreq(n)[1:]
    spec = spec[1:]
    mask = (freq > 0.0) & (spec > 0.0)
    if int(np.sum(mask)) < 5:
        return float("nan")
    return float(-np.polyfit(np.log(freq[mask]), np.log(spec[mask]), 1)[0])


def est_lz_complexity(x: Any) -> float:
    """Normalized Lempel-Ziv (LZ76, Kaspar & Schuster 1987) complexity on median-binarized series."""
    xf = _finite(x)
    n = len(xf)
    if n < 20 or np.ptp(xf) == 0.0:
        return float("nan")
    med = float(np.median(xf))
    s = "".join("1" if v > med else "0" for v in xf)

    i, k, l = 0, 1, 1
    c = 1
    k_max = 1
    while True:
        if s[i + k - 1] == s[l + k - 1]:
            k += 1
            if l + k > n:
                c += 1
                break
        else:
            if k > k_max:
                k_max = k
            i += 1
            if i == l:
                c += 1
                l += k_max
                if l + 1 > n:
                    break
                i = 0
                k = 1
                k_max = 1
            else:
                k = 1
    return float(c / (n / max(math.log2(n), 1e-9)))


def est_autocorr1(x: Any) -> float:
    """Lag-1 Pearson autocorrelation."""
    xf = _finite(x)
    if len(xf) < 10 or np.std(xf) == 0.0:
        return float("nan")
    return float(np.corrcoef(xf[:-1], xf[1:])[0, 1])


def est_skew(x: Any) -> float:
    """Sample skewness."""
    xf = _finite(x)
    if len(xf) < 5 or np.std(xf) == 0.0:
        return float("nan")
    m, s = float(np.mean(xf)), float(np.std(xf))
    return float(np.mean(((xf - m) / s) ** 3))


def est_kurtosis(x: Any) -> float:
    """Excess kurtosis."""
    xf = _finite(x)
    if len(xf) < 5 or np.std(xf) == 0.0:
        return float("nan")
    m, s = float(np.mean(xf)), float(np.std(xf))
    return float(np.mean(((xf - m) / s) ** 4) - 3.0)


def est_corr_dim_d2(x: Any, dim: int = 2) -> float:
    """Correlation dimension D2 estimator."""
    xf = _finite(x)
    if len(xf) < 50:
        return float("nan")
    emb = _embed(xf, dim=dim)
    if emb is None:
        return float("nan")
    if len(emb) > 300:
        emb = emb[np.linspace(0, len(emb) - 1, 300).astype(int)]
    d = pdist(emb)
    d = d[d > 0.0]
    if len(d) < 10:
        return float("nan")
    rs = np.percentile(d, [10, 25, 50, 75])
    if np.any(rs <= 0.0) or len(np.unique(rs)) < 2:
        return float("nan")
    cs = [float(np.mean(d < r)) for r in rs]
    if any(c <= 0.0 for c in cs):
        return float("nan")
    return float(np.polyfit(np.log(rs), np.log(cs), 1)[0])


def est_levina_bickel(x: Any, k: int = 10) -> float:
    """Levina-Bickel MLE intrinsic dimension estimator."""
    xf = _finite(x)
    if len(xf) < 30:
        return float("nan")
    emb = _embed(xf, dim=2)
    if emb is None or len(emb) <= k + 1:
        return float("nan")
    if len(emb) > 300:
        emb = emb[np.linspace(0, len(emb) - 1, 300).astype(int)]
    dist_mat = np.sort(cdist(emb, emb), axis=1)[:, 1 : k + 1]
    dist_mat = np.maximum(dist_mat, 1e-12)
    vals = np.mean(np.log(dist_mat[:, -1:] / dist_mat[:, :-1]), axis=1)
    vals = vals[vals > 0.0]
    if len(vals) == 0:
        return float("nan")
    return float(1.0 / np.mean(vals))


def est_pca_participation(x: Any, dim: int = 3) -> float:
    """Global PCA participation ratio."""
    xf = _finite(x)
    if len(xf) < 30:
        return float("nan")
    emb = _embed(xf, dim=dim)
    if emb is None:
        return float("nan")
    w = np.linalg.eigvalsh(np.cov(emb.T))
    w = w[w > 0.0]
    if len(w) == 0:
        return float("nan")
    p = w / np.sum(w)
    return float(1.0 / np.sum(p ** 2))


def est_information_d1(x: Any, dim: int = 2, bins: int = 8) -> float:
    """Normalized 2D phase-space Shannon entropy proxy (D1-style)."""
    xf = _finite(x)
    if len(xf) < 50 or np.ptp(xf) == 0.0:
        return float("nan")
    emb = _embed(xf, dim=dim)
    if emb is None:
        return float("nan")
    hist, _ = np.histogramdd(emb, bins=bins)
    tot = float(np.sum(hist))
    if tot <= 0.0:
        return float("nan")
    p = hist.ravel() / tot
    p = p[p > 0.0]
    return float(-np.sum(p * np.log(p)) / math.log(bins))


def est_two_nn(x: Any) -> float:
    """Two-NN intrinsic dimension estimator."""
    xf = _finite(x)
    if len(xf) < 30:
        return float("nan")
    emb = _embed(xf, dim=2)
    if emb is None:
        return float("nan")
    if len(emb) > 300:
        emb = emb[np.linspace(0, len(emb) - 1, 300).astype(int)]
    d = np.sort(cdist(emb, emb), axis=1)[:, 1:3]
    ratio = d[:, 1] / np.maximum(d[:, 0], 1e-12)
    ratio = ratio[np.isfinite(ratio) & (ratio > 1.0)]
    if len(ratio) < 10:
        return float("nan")
    return float(1.0 / np.mean(np.log(ratio)))


def est_lyapunov_rosenstein(x: Any, dim: int = 2, tau: int = 1, max_t: int = 10) -> float:
    """Rosenstein largest Lyapunov exponent estimator with Theiler window."""
    xf = _finite(x)
    if len(xf) < 60 or np.ptp(xf) == 0.0:
        return float("nan")
    emb = _embed(xf, dim=dim, tau=tau)
    if emb is None or len(emb) < max_t + 8:
        return float("nan")
    if len(emb) > 300:
        emb = emb[np.linspace(0, len(emb) - 1, 300).astype(int)]

    m = len(emb)
    dist_mat = cdist(emb, emb)
    theiler = max(2, min(10, m // 15))
    idx_grid = np.arange(m)
    mask_theiler = np.abs(idx_grid[:, None] - idx_grid[None, :]) <= theiler
    dist_mat[mask_theiler] = np.inf
    nn = np.argmin(dist_mat, axis=1)

    divs: List[float] = []
    for t in range(1, max_t + 1):
        valid = np.arange(m - t)
        keep = nn[valid] + t < m
        if not np.any(keep):
            continue
        i_ok = valid[keep]
        j_ok = nn[i_ok]
        diffs = emb[i_ok + t] - emb[j_ok + t]
        norms = np.linalg.norm(diffs, axis=1)
        divs.append(float(np.mean(np.log(np.maximum(norms, 1e-12)))))
    if len(divs) < 3:
        return float("nan")
    return float(np.polyfit(np.arange(1, len(divs) + 1, dtype=float), np.asarray(divs), 1)[0])


def est_grassberger_procaccia(x: Any, dims: Tuple[int, ...] = (2, 3, 4), tau: int = 1) -> float:
    """Grassberger-Procaccia multi-dimension correlation exponent."""
    xf = _finite(x)
    if len(xf) < 60:
        return float("nan")
    cs: List[float] = []
    for dim in dims:
        emb = _embed(xf, dim=dim, tau=tau)
        if emb is None or len(emb) < 30:
            continue
        if len(emb) > 250:
            emb = emb[np.linspace(0, len(emb) - 1, 250).astype(int)]
        d = pdist(emb)
        d = d[d > 0.0]
        if len(d) < 100:
            continue
        radii = np.quantile(d, [0.05, 0.08, 0.12, 0.18, 0.25])
        corr = np.array([np.mean(d < r) for r in radii])
        good = (corr > 0.0) & np.isfinite(corr) & (radii > 0.0)
        if int(np.sum(good)) >= 3 and len(np.unique(radii[good])) >= 2:
            slope = float(np.polyfit(np.log(radii[good]), np.log(corr[good]), 1)[0])
            cs.append(slope)
    return float(np.median(cs)) if cs else float("nan")


def _natural_visibility_adjacency(x: Any, max_n: int = 240) -> Optional[List[Set[int]]]:
    """Construct natural visibility graph adjacency list."""
    xf = _finite(x)
    if len(xf) < 8:
        return None
    if len(xf) > max_n:
        xf = xf[np.linspace(0, len(xf) - 1, max_n).astype(int)]
    n = len(xf)
    adj: List[Set[int]] = [set() for _ in range(n)]
    for i in range(n - 1):
        adj[i].add(i + 1)
        adj[i + 1].add(i)
    for i in range(n - 2):
        slope = -np.inf
        xi = xf[i]
        for j in range(i + 2, n):
            cur = (xf[j] - xi) / (j - i)
            if cur > slope:
                adj[i].add(j)
                adj[j].add(i)
                slope = cur
    return adj


def _visibility_tail_from_adj(adj: Optional[List[Set[int]]]) -> float:
    if adj is None:
        return float("nan")
    deg = np.asarray([len(a) for a in adj], dtype=float)
    deg = deg[deg > 0.0]
    if len(deg) < 8:
        return float("nan")
    vals = np.sort(deg)
    tail = vals[max(0, int(0.6 * len(vals))) :]
    if len(tail) < 4 or np.all(tail == tail[0]) or tail.min() <= 0.0:
        return float("nan")
    denom = float(np.sum(np.log(tail / tail.min())))
    if denom <= 0.0:
        return float("nan")
    return float(1.0 + len(tail) / denom)


def _graph_cluster_from_adj(adj: Optional[List[Set[int]]]) -> float:
    if adj is None:
        return float("nan")
    vals: List[float] = []
    for nei in adj:
        k = len(nei)
        if k < 2:
            continue
        links = sum(len(nei & adj[a]) for a in nei) // 2
        vals.append(2.0 * links / (k * (k - 1)))
    return float(np.mean(vals)) if vals else float("nan")


def est_visibility_graph(x: Any) -> float:
    """Degree-tail power-law exponent of the natural visibility graph."""
    return _visibility_tail_from_adj(_natural_visibility_adjacency(x))


def est_graph_cluster(x: Any) -> float:
    """Mean local clustering coefficient of the natural visibility graph."""
    return _graph_cluster_from_adj(_natural_visibility_adjacency(x))


def est_fisher_dimension(x: Any, bins: int = 16) -> float:
    """Effective informational perplexity exp(H) of the empirical distribution."""
    xf = _finite(x)
    if len(xf) < 20 or np.std(xf) == 0.0:
        return float("nan")
    hist, _ = np.histogram(xf, bins=bins, density=True)
    tot = float(np.sum(hist))
    if tot <= 0.0:
        return float("nan")
    p = hist / tot
    p = p[p > 0.0]
    return float(np.exp(-np.sum(p * np.log(p)))) if len(p) else float("nan")


def est_local_pca(x: Any, dim: int = 5, k: int = 20) -> float:
    """Local k-NN PCA participation ratio (batched NumPy eigvalsh)."""
    xf = _finite(x)
    emb = _embed(xf, dim=dim)
    if emb is None or len(emb) < k + 2:
        return float("nan")
    if len(emb) > 300:
        emb = emb[np.linspace(0, len(emb) - 1, 300).astype(int)]

    n_pts = len(emb)
    k_eff = min(k, n_pts - 1)
    dist_mat = cdist(emb, emb)
    np.fill_diagonal(dist_mat, np.inf)
    nn_idx = np.argpartition(dist_mat, k_eff, axis=1)[:, :k_eff]  # (n_pts, k_eff)
    neighbors = emb[nn_idx]  # (n_pts, k_eff, dim)
    centered = neighbors - np.mean(neighbors, axis=1, keepdims=True)
    covs = np.einsum("nki,nkj->nij", centered, centered) / max(k_eff - 1, 1)
    eigvals = np.maximum(np.linalg.eigvalsh(covs), 0.0)
    sums = np.sum(eigvals, axis=1)
    valid = sums > 0.0
    if not np.any(valid):
        return float("nan")
    q = eigvals[valid] / sums[valid, None]
    pr = 1.0 / np.sum(q * q, axis=1)
    return float(np.median(pr))


def est_persistent_homology(x: Any) -> float:
    """Persistent entropy of 0D/1D Vietoris-Rips persistence diagrams (requires ripser)."""
    xf = _finite(x)
    if len(xf) < 20:
        return float("nan")
    try:
        from ripser import ripser  # type: ignore
    except ImportError:
        return float("nan")

    emb = _embed(xf, dim=2)
    if emb is None:
        return float("nan")
    if len(emb) > 250:
        emb = emb[np.linspace(0, len(emb) - 1, 250).astype(int)]
    dgms = ripser(emb, maxdim=1)["dgms"]
    lifetimes: List[float] = []
    for dgm in dgms:
        if len(dgm):
            life = dgm[:, 1] - dgm[:, 0]
            life = life[np.isfinite(life) & (life > 0.0)]
            lifetimes.extend(life.tolist())
    if len(lifetimes) < 2:
        return float("nan")
    q = np.asarray(lifetimes, dtype=float)
    q = q / np.sum(q)
    return float(-np.sum(q * np.log(q)))


def _mfdfa_q_spectrum(x: Any, q_values: Sequence[float]) -> Optional[np.ndarray]:
    """Vectorized multifractal DFA generalized Hurst spectrum h(q)."""
    xf = _finite(x)
    n = len(xf)
    if n < 80 or np.std(xf) == 0.0:
        return None
    y = np.cumsum(xf - np.mean(xf))
    scales = sorted(
        {
            max(8, n // 32),
            max(10, n // 16),
            max(12, n // 8),
            max(16, n // 4),
        }
    )
    scales = [v for v in scales if v < n // 2]
    if len(scales) < 3:
        return None

    # Compute local segment variances ONCE per scale, then evaluate all q values
    vars_by_scale: List[Tuple[int, np.ndarray]] = []
    for scale in scales:
        m = n // scale
        if m < 2:
            continue
        t = np.arange(scale, dtype=float)
        tc = t - np.mean(t)
        denom = float(np.sum(tc * tc))
        segs_f = y[: m * scale].reshape(m, scale)
        segs_b = y[n - m * scale :].reshape(m, scale)
        segs = np.vstack([segs_f, segs_b])
        ym = np.mean(segs, axis=1, keepdims=True)
        b = np.sum((segs - ym) * tc[None, :], axis=1, keepdims=True) / denom
        a = ym - b * np.mean(t)
        resid = segs - (a + b * t[None, :])
        local_var = np.mean(resid * resid, axis=1)
        local_var = local_var[local_var > 0.0]
        if len(local_var) >= 4:
            vars_by_scale.append((scale, local_var))

    if len(vars_by_scale) < 3:
        return None

    hq: List[float] = []
    for q in q_values:
        fs: List[float] = []
        ss: List[int] = []
        for scale, a_var in vars_by_scale:
            if q != 0:
                f_val = float((np.mean(a_var ** (q / 2.0))) ** (1.0 / q))
            else:
                f_val = float(np.exp(0.5 * np.mean(np.log(a_var))))
            if f_val > 0.0 and math.isfinite(f_val):
                fs.append(f_val)
                ss.append(scale)
        if len(fs) >= 3:
            hq.append(float(np.polyfit(np.log(np.asarray(ss, dtype=float)), np.log(np.asarray(fs)), 1)[0]))
        else:
            hq.append(float("nan"))
    return np.asarray(hq, dtype=float)


def est_mfa(x: Any) -> float:
    """Multifractal Hurst-exponent width h(q_max) - h(q_min)."""
    h = _mfdfa_q_spectrum(x, [-4.0, -2.0, 0.0, 2.0, 4.0])
    if h is None or int(np.sum(np.isfinite(h))) < 3:
        return float("nan")
    return float(np.nanmax(h) - np.nanmin(h))


def est_mfdfa_spectrum(x: Any) -> float:
    """Width of multifractal scaling exponent tau(q) = q*h(q) - 1."""
    q = np.asarray([-5.0, -3.0, -1.0, 1.0, 3.0, 5.0], dtype=float)
    h = _mfdfa_q_spectrum(x, q)
    if h is None or int(np.sum(np.isfinite(h))) < 4:
        return float("nan")
    tau = q * h - 1.0
    good = np.isfinite(tau)
    return float(np.nanmax(tau[good]) - np.nanmin(tau[good]))


def est_danco(x: Any) -> float:
    """DANCo intrinsic dimension (explicitly not implemented; returns NaN)."""
    return float("nan")


ESTIMATORS: List[Tuple[str, Callable[[Any], float], bool]] = [
    ("Higuchi", est_higuchi, True),
    ("DFA", est_dfa, True),
    ("Katz", est_katz, True),
    ("Hurst", est_hurst_rs, False),
    ("SampleEn", est_sample_entropy, False),
    ("Petrosian", est_petrosian, True),
    ("Sevcik", est_sevcik, True),
    ("BoxCounting", est_boxcount, True),
    ("PermutationEntropy", est_perm_entropy, False),
    ("CorrelationD2", est_corr_dim_d2, True),
    ("GrassbergerProcaccia", est_grassberger_procaccia, True),
    ("InformationD1", est_information_d1, True),
    ("LevinaBickel", est_levina_bickel, True),
    ("TwoNN", est_two_nn, True),
    ("PCA_Participation", est_pca_participation, True),
    ("LyapunovRosenstein", est_lyapunov_rosenstein, False),
    ("SpectralBeta", est_spectral_beta, False),
    ("KolmogorovLZ", est_lz_complexity, False),
    ("VisibilityGraph", est_visibility_graph, False),
    ("Danco", est_danco, False),
    ("FisherDimension", est_fisher_dimension, False),
    ("LocalPCA", est_local_pca, True),
    ("GraphCluster", est_graph_cluster, False),
    ("PersistentHomology", est_persistent_homology, False),
    ("MFA", est_mfa, False),
    ("MFDFASpectrum", est_mfdfa_spectrum, False),
    ("Autocorr1", est_autocorr1, False),
    ("Skewness", est_skew, False),
    ("Kurtosis", est_kurtosis, False),
]
assert len(ESTIMATORS) == 29

DIMENSION_NAMES: Set[str] = {n for n, _, d in ESTIMATORS if d}


def _unique_estimators() -> List[Tuple[str, Callable[[Any], float], bool]]:
    """Return deduplicated estimators by underlying callable name."""
    seen: Set[str] = set()
    out: List[Tuple[str, Callable[[Any], float], bool]] = []
    for name, fn, is_dim in ESTIMATORS:
        key = getattr(fn, "__name__", str(fn))
        if key in seen:
            continue
        seen.add(key)
        out.append((name, fn, is_dim))
    return out


UNIQUE_ESTIMATORS = _unique_estimators()
UNIQUE_DIMENSION_NAMES: Set[str] = {n for n, _, d in UNIQUE_ESTIMATORS if d}


def apply_estimators(series: Any) -> Dict[str, float]:
    """Evaluate all 29 estimators on a series, sharing expensive graph intermediates."""
    xf = _finite(series)
    adj_cache: Optional[List[Set[int]]] = None
    adj_computed = False
    out: Dict[str, float] = {}

    for name, fn, _ in ESTIMATORS:
        try:
            if name in ("VisibilityGraph", "GraphCluster"):
                if not adj_computed:
                    adj_cache = _natural_visibility_adjacency(xf)
                    adj_computed = True
                if name == "VisibilityGraph":
                    out[name] = _safe_float(_visibility_tail_from_adj(adj_cache))
                else:
                    out[name] = _safe_float(_graph_cluster_from_adj(adj_cache))
            else:
                out[name] = _safe_float(fn(xf))
        except Exception:
            out[name] = float("nan")
    return out


def apply_dimension_estimators(series: Any) -> Dict[str, float]:
    """Evaluate only unique dimension estimators (fast helper for T_log permutation null)."""
    xf = _finite(series)
    out: Dict[str, float] = {}
    for name, fn, is_dim in UNIQUE_ESTIMATORS:
        if not is_dim:
            continue
        try:
            out[name] = _safe_float(fn(xf))
        except Exception:
            out[name] = float("nan")
    return out


def compute_tlog(
    est: Dict[str, float],
    n: int,
    series: Any = None,
    n_perm: int = 25,
    seed: int = 42,
) -> Dict[str, Any]:
    """Compute T_log complexity diagnostic calibrated against its permutation null (P8).

    When ``series`` is provided, ``d_median`` is calibrated against ``n_perm`` temporal
    permutations of ``series`` (preserving marginal distribution while breaking order):
        t_log = (d_median - d_null_median) * log(n)
        z_score = (d_median - d_null_median) / max(d_null_std, 1e-6)
    If ``series`` is omitted, falls back to uncalibrated ``d_median`` summary with
    ``t_log = NaN`` (no magic domain reference constant).
    """
    seen_fn: Set[str] = set()
    dims: List[float] = []
    for name, fn, is_dim in ESTIMATORS:
        if not is_dim:
            continue
        key = getattr(fn, "__name__", str(fn))
        if key in seen_fn:
            continue
        seen_fn.add(key)
        v = est.get(name, float("nan"))
        if math.isfinite(v):
            dims.append(float(v))

    if not dims or n <= 1:
        return {
            "t_log": float("nan"),
            "d_median": float("nan"),
            "d_null_median": float("nan"),
            "d_null_std": float("nan"),
            "z_score": float("nan"),
            "p_permutation": float("nan"),
            "n_dimension_estimators": 0,
            "calibrated": False,
            "note": "not computable — diagnostic only",
        }

    d_med = float(np.median(dims))
    if series is None or n_perm < 1:
        return {
            "t_log": float("nan"),
            "d_median": d_med,
            "d_null_median": float("nan"),
            "d_null_std": float("nan"),
            "z_score": float("nan"),
            "p_permutation": float("nan"),
            "n_dimension_estimators": len(dims),
            "calibrated": False,
            "note": "uncalibrated (pass series for permutation-null calibration) — diagnostic only",
        }

    xf = _finite(series)
    rng = np.random.default_rng(seed)
    null_medians: List[float] = []
    for _ in range(int(n_perm)):
        perm = rng.permutation(xf)
        perm_dims = [v for v in apply_dimension_estimators(perm).values() if math.isfinite(v)]
        if perm_dims:
            null_medians.append(float(np.median(perm_dims)))

    if len(null_medians) < 3:
        return {
            "t_log": float("nan"),
            "d_median": d_med,
            "d_null_median": float("nan"),
            "d_null_std": float("nan"),
            "z_score": float("nan"),
            "p_permutation": float("nan"),
            "n_dimension_estimators": len(dims),
            "calibrated": False,
            "note": "permutation null insufficient — diagnostic only",
        }

    null_arr = np.asarray(null_medians, dtype=float)
    d_null_med = float(np.median(null_arr))
    d_null_std = float(np.std(null_arr, ddof=1))
    delta = d_med - d_null_med
    t_log_cal = float(delta * math.log(n))
    z_score = float(delta / max(d_null_std, 1e-6))
    p_perm = float((1 + np.sum(np.abs(null_arr - d_null_med) >= abs(delta))) / (len(null_arr) + 1))

    return {
        "t_log": t_log_cal,
        "d_median": d_med,
        "d_null_median": d_null_med,
        "d_null_std": d_null_std,
        "z_score": z_score,
        "p_permutation": p_perm,
        "n_dimension_estimators": len(dims),
        "calibrated": True,
        "note": "permutation-calibrated diagnostic only, never proof",
    }

## Module `diagnostics.py`

In [ ]:
"""Optional segmented estimator validation suite (rolling trajectories, Step 1-4 tests, MC)."""


import math
from typing import Any, Dict, List, Optional, Tuple

import numpy as np



def _rolling_estimator_trajectories(
    series: Any,
    window: int,
    step: int = 5,
    coordinate: Any = None,
    return_coordinates: bool = False,
) -> Any:
    """Compute sliding-window estimator trajectories without NaN imputation."""
    x = np.asarray(series, dtype=float)
    c = np.arange(len(x), dtype=float) if coordinate is None else np.asarray(coordinate, dtype=float)
    if len(c) != len(x):
        raise ValueError("coordinate must have the same length as series")
    if window < 8 or len(x) < window:
        empty = {name: np.array([], dtype=float) for name, _, _ in ESTIMATORS}
        return (empty, np.array([], dtype=float)) if return_coordinates else empty

    rows: List[Dict[str, float]] = []
    centers: List[float] = []
    for i in range(0, len(x) - window + 1, max(1, step)):
        rows.append(apply_estimators(x[i : i + window]))
        cc = c[i : i + window]
        centers.append(float(np.nanmean(cc)) if np.isfinite(cc).any() else float("nan"))

    trajectories = {
        name: np.asarray([r.get(name, np.nan) for r in rows], dtype=float)
        for name, _, _ in ESTIMATORS
    }
    centers_arr = np.asarray(centers, dtype=float)
    return (trajectories, centers_arr) if return_coordinates else trajectories


def _step1_white_noise_q5(y: Any, seed: int = 42, n_perm: int = 199) -> Dict[str, Any]:
    yf = _finite(y)
    if len(yf) < 8 or np.std(yf) == 0.0:
        return {"status": "insufficient_or_constant", "statistic": float("nan"), "p_value": float("nan")}
    obs = float(np.corrcoef(yf[:-1], yf[1:])[0, 1])
    rng = np.random.default_rng(seed)
    null = []
    for _ in range(n_perm):
        z = rng.permutation(yf)
        null.append(float(np.corrcoef(z[:-1], z[1:])[0, 1]))
    lo, hi = np.quantile(null, [0.025, 0.975])
    return {
        "status": "ok",
        "statistic": obs,
        "null_q025": float(lo),
        "null_q975": float(hi),
        "p_value": float((1 + sum(abs(v) >= abs(obs) for v in null)) / (n_perm + 1)),
        "flag": bool(obs < lo or obs > hi),
        "definition": "provisional Q5: lag-1 autocorrelation against permutation null",
    }


def _step2_local_volatility(y: Any) -> Dict[str, Any]:
    yf = _finite(y)
    if len(yf) < 8 or np.std(yf, ddof=1) == 0.0:
        return {"status": "insufficient_or_constant", "ratio": float("nan"), "flag": None}
    w = max(3, len(yf) // 4)
    local = [np.std(yf[i : i + w], ddof=1) for i in range(0, len(yf) - w + 1, max(1, w // 2))]
    ratio = float(np.max(local) / np.std(yf, ddof=1)) if local else float("nan")
    return {
        "status": "ok",
        "ratio": ratio,
        "flag": bool(ratio > 1.5),
        "definition": "max local std / global std; provisional threshold 1.5",
    }


def _step3_hurst_drift(y: Any) -> Dict[str, Any]:
    yf = _finite(y)
    if len(yf) < 20 or np.std(yf) == 0.0:
        return {"status": "insufficient_or_constant", "hurst": float("nan"), "drift": float("nan")}
    h = est_hurst_rs(yf)
    t = np.arange(len(yf), dtype=float)
    drift = float(np.polyfit(t, yf, 1)[0])
    return {
        "status": "ok" if math.isfinite(h) else "insufficient",
        "hurst_rs": h,
        "linear_drift": drift,
        "definition": "Hurst R/S + linear slope; provisional joint diagnostic",
    }


def _step4_fft_spectral(y: Any) -> Dict[str, Any]:
    yf = _finite(y)
    if len(yf) < 16 or np.std(yf) == 0.0:
        return {
            "status": "insufficient_or_constant",
            "spectral_entropy": float("nan"),
            "dominant_fraction": float("nan"),
        }
    p = np.abs(np.fft.rfft(yf - np.mean(yf))) ** 2
    if len(p) > 0:
        p[0] = 0.0
    p = p[p > 0.0]
    if len(p) < 2:
        return {
            "status": "insufficient",
            "spectral_entropy": float("nan"),
            "dominant_fraction": float("nan"),
        }
    q = p / np.sum(p)
    ent = float(-np.sum(q * np.log(q)) / math.log(len(q)))
    dom = float(np.max(q))
    return {
        "status": "ok",
        "spectral_entropy": ent,
        "dominant_fraction": dom,
        "definition": "FFT spectral entropy + dominant power fraction",
    }


def _four_noise_tests(y: Any, seed: int = 42) -> Dict[str, Any]:
    """Run Steps 1-4 noise diagnostics on a single estimator trajectory."""
    return {
        "step1_white_noise_q5": _step1_white_noise_q5(y, seed),
        "step2_local_volatility": _step2_local_volatility(y),
        "step3_hurst_drift": _step3_hurst_drift(y),
        "step4_fft_spectral": _step4_fft_spectral(y),
    }


def monte_carlo_estimators(
    series: Any,
    n_paths: int = 200,
    seed: int = 42,
    window: int = 80,
) -> Dict[str, Any]:
    """Uniform input-perturbation Monte Carlo across all 29 estimators."""
    x = np.asarray(series, dtype=float)
    fin = _finite(x)
    if len(fin) < max(20, window // 2):
        return {"status": "insufficient_data", "estimators": {}, "n_paths": 0, "seed": int(seed)}

    dif = np.diff(fin)
    scale = (
        float(1.4826 * np.median(np.abs(dif - np.median(dif))) / math.sqrt(2.0))
        if len(dif)
        else 0.0
    )
    if not math.isfinite(scale):
        scale = 0.0

    results: Dict[str, List[float]] = {name: [] for name, _, _ in ESTIMATORS}
    seeds: List[int] = []
    n_total_paths = max(50, int(n_paths))
    mask = np.isfinite(x)
    n_mask = int(np.sum(mask))

    for k in range(n_total_paths):
        path_seed = int(np.random.SeedSequence([int(seed), k]).generate_state(1)[0])
        seeds.append(path_seed)
        rng = np.random.default_rng(path_seed)
        perturbed = x.copy()
        if scale > 0.0:
            perturbed[mask] = perturbed[mask] + rng.normal(0.0, scale, size=n_mask)
        est = apply_estimators(perturbed)
        for name, _, _ in ESTIMATORS:
            results[name].append(est.get(name, np.nan))

    out: Dict[str, Any] = {}
    for name, vals in results.items():
        a = np.asarray(vals, dtype=float)
        f = a[np.isfinite(a)]
        out[name] = {
            "mean": float(np.mean(f)) if len(f) else float("nan"),
            "std": float(np.std(f, ddof=1)) if len(f) > 1 else float("nan"),
            "median": float(np.median(f)) if len(f) else float("nan"),
            "q05": float(np.quantile(f, 0.05)) if len(f) else float("nan"),
            "q95": float(np.quantile(f, 0.95)) if len(f) else float("nan"),
            "valid_fraction": float(len(f) / len(a)),
            "n_valid": int(len(f)),
            "n_total": int(len(a)),
        }

    return {
        "status": "ok",
        "method": "uniform input-perturbation MC; segmented cohorts",
        "n_paths": len(seeds),
        "seed": int(seed),
        "noise_scale_mad_diff": scale,
        "weighting": "equal weight per Monte Carlo path",
        "estimators": out,
        "draw_order_is_not_time": True,
    }


def _cohort_validation(
    series: Any,
    seed: int,
    n_mc_paths: int,
    window: int,
    coordinate: Any = None,
    gaussian_sigma: float = 2.0,
) -> Dict[str, Any]:
    x = np.asarray(series, dtype=float)
    n = len(x)
    coord = np.arange(n, dtype=float) if coordinate is None else np.asarray(coordinate, dtype=float)
    win = max(20, min(int(window), max(20, n // 3)))
    step = max(1, win // 6)

    mc = monte_carlo_estimators(x, n_paths=n_mc_paths, seed=seed, window=win)
    trajs, centers = _rolling_estimator_trajectories(
        x, win, step=step, coordinate=coord, return_coordinates=True
    )
    by_est: Dict[str, Any] = {}
    for i, (name, _, _) in enumerate(ESTIMATORS):
        tr = trajs[name]
        tests = _four_noise_tests(tr, seed + 1009 * i)
        deriv = _f1_f2(tr, centers, gaussian_sigma=gaussian_sigma)
        by_est[name] = {
            "n_trajectory_points": int(len(tr)),
            "tests": tests,
            "f1_f2_summary": {
                "status": deriv.get("status"),
                "f1_mean": deriv.get("f1_mean"),
                "f2_mean": deriv.get("f2_mean"),
            },
        }
    return {
        "status": "ok" if n >= win else "insufficient_data",
        "n": int(n),
        "window": int(win),
        "step": int(step),
        "per_estimator": by_est,
        "monte_carlo": mc,
    }


def estimator_validation_suite(
    series: Any,
    seed: int = 42,
    n_mc_paths: int = 200,
    control: Any = None,
    threshold_idx: Optional[int] = None,
    gaussian_sigma: float = 2.0,
) -> Dict[str, Any]:
    """Three-cohort segmented validation (initial, pre-threshold, post-threshold)."""
    x = np.asarray(series, dtype=float)
    n = len(x)
    c = np.arange(n, dtype=float) if control is None else np.asarray(control, dtype=float)
    if len(c) != n:
        raise ValueError("control must have the same length as series")

    cut = int(threshold_idx) if (threshold_idx is not None and 0 < int(threshold_idx) < n) else None
    if cut is None:
        cohorts: Dict[str, Tuple[np.ndarray, np.ndarray]] = {"initial": (x, c)}
    else:
        cohorts = {
            "initial": (x, c),
            "pre_threshold": (x[:cut], c[:cut]),
            "post_threshold": (x[cut:], c[cut:]),
        }

    win = max(20, min(128, n // 5))
    stages: Dict[str, Any] = {}
    for stage, (sx, sc) in cohorts.items():
        stages[stage] = _cohort_validation(
            sx, seed, n_mc_paths, win, coordinate=sc, gaussian_sigma=gaussian_sigma
        )
        stages[stage]["raw_target_f1_f2"] = _f1_f2(sx, sc, gaussian_sigma=gaussian_sigma)

    comparison: Dict[str, Any] = {"status": "not_applicable"}
    predictive_mc: Dict[str, Any] = {
        "status": "not_applicable",
        "interpretation": "no valid threshold or insufficient pre/post cohorts",
    }

    if cut is not None and "pre_threshold" in stages and "post_threshold" in stages:
        comparison = {"status": "ok", "per_estimator": {}}
        for name, _, _ in ESTIMATORS:
            a = stages["pre_threshold"]["monte_carlo"].get("estimators", {}).get(name, {})
            b = stages["post_threshold"]["monte_carlo"].get("estimators", {}).get(name, {})
            ma, mb = a.get("median", float("nan")), b.get("median", float("nan"))
            comparison["per_estimator"][name] = {
                "pre_mc_median": ma,
                "post_mc_median": mb,
                "delta_post_minus_pre": float(mb - ma)
                if math.isfinite(ma) and math.isfinite(mb)
                else float("nan"),
            }

        pre = stages["pre_threshold"]["monte_carlo"].get("estimators", {})
        post_obs = apply_estimators(x[cut:])
        checks: Dict[str, Any] = {}
        n_tested, n_inside = 0, 0
        for name, _, _ in ESTIMATORS:
            p = pre.get(name, {})
            obs = post_obs.get(name, float("nan"))
            lo, hi = p.get("q05", float("nan")), p.get("q95", float("nan"))
            valid = math.isfinite(obs) and math.isfinite(lo) and math.isfinite(hi)
            inside = bool(valid and lo <= obs <= hi)
            checks[name] = {
                "post_observed": obs,
                "pre_mc_q05": lo,
                "pre_mc_q95": hi,
                "inside_pre_mc_interval": inside,
                "status": "ok" if valid else "insufficient",
            }
            if valid:
                n_tested += 1
                n_inside += int(inside)
        predictive_mc = {
            "status": "ok" if n_tested else "insufficient_data",
            "method": "pre-threshold MC interval vs post-threshold observed estimate",
            "n_estimators_tested": n_tested,
            "n_inside_90pct_interval": n_inside,
            "empirical_coverage": float(n_inside / n_tested) if n_tested else float("nan"),
            "per_estimator": checks,
            "warning": "retrospective pre/post check, not an independent prospective forecast",
        }

    return {
        "status": "segmented_validation_protocol",
        "n_estimators_named": len(ESTIMATORS),
        "n_unique_callables": len(UNIQUE_ESTIMATORS),
        "threshold_idx": cut,
        "stages": stages,
        "pre_post_comparison": comparison,
        "predictive_mc_check": predictive_mc,
        "f1_f2_specification": {
            "F1": "gradient(y_smooth, p_coord)",
            "F2": "gradient(F1, p_coord)",
            "smoothing": "Gaussian",
            "gaussian_sigma": float(gaussian_sigma),
        },
    }

## Module `falsification.py`

In [ ]:
"""Falsification suite, walk-forward validation, and conservative evidence grading."""


import math
from typing import Any, Dict, List

import numpy as np
import pandas as pd



def _simulate_ar1(n: int, phi: float, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Simulate a stationary Gaussian AR(1) series with matched lag-1 autocorrelation."""
    x = np.empty(n, dtype=float)
    denom = math.sqrt(max(1.0 - phi * phi, 0.05))
    x[0] = rng.normal(0.0, sigma / denom)
    eps = rng.normal(0.0, sigma, size=n)
    for i in range(1, n):
        x[i] = phi * x[i - 1] + eps[i]
    return x


def _simulate_arima110(n: int, phi_dy: float, rng: np.random.Generator) -> np.ndarray:
    """Simulate an integrated ARIMA(1,1,0) smooth momentum series matching delta autocorrelation."""
    burn = 50
    dy = np.zeros(n + burn, dtype=float)
    eps = rng.normal(0.0, 1.0, size=n + burn)
    p = max(-0.90, min(0.90, float(phi_dy)))
    for i in range(1, n + burn):
        dy[i] = p * dy[i - 1] + eps[i]
    return np.cumsum(dy[burn:])


def _local_linear_at_cut(
    y: np.ndarray,
    idx: int,
    w: int,
    detector: str,
) -> float:
    """Evaluate exact matched slope_break or detrended_jump effect metric at a fixed cut."""
    if idx < w or idx + w > len(y) or w < 4:
        return float("nan")
    t_arr = np.arange(w, dtype=float)
    tc = t_arr - float(np.mean(t_arr))
    denom = float(np.sum(tc * tc))
    yl = y[idx - w : idx]
    yr = y[idx : idx + w]
    if denom <= 0.0:
        return float("nan")
    bl = float(np.sum(tc * (yl - np.mean(yl))) / denom)
    br = float(np.sum(tc * (yr - np.mean(yr))) / denom)
    al = float(np.mean(yl)) - bl * float(np.mean(t_arr))
    ar = float(np.mean(yr)) - br * float(np.mean(t_arr))
    sse = float(np.sum((yl - (al + bl * t_arr)) ** 2) + np.sum((yr - (ar + br * t_arr)) ** 2))
    sd = math.sqrt(max(sse / max(2 * w - 4, 1), 1e-18))
    if detector == "slope_break":
        se = sd * math.sqrt(2.0 / denom)
        return float(((br - bl) / max(se, 1e-18)) / math.sqrt(w))
    return float((ar - (al + bl * w)) / sd)


def _scan_active_channel(
    ctrl: np.ndarray,
    y: np.ndarray,
    detector: str,
    window: int,
    min_group: int,
    seed: int,
    n_boot: int = 20,
    n_perm: int = 50,
    compute_scan_p: bool = True,
    n_scan_perm: int = 200,
) -> Dict[str, Any]:
    """Run threshold scan matched to the active detector family (symmetrized falsification null)."""
    y_arr = np.asarray(y, dtype=float)
    x_arr = np.asarray(ctrl, dtype=float)
    n = len(y_arr)
    if detector == "slope_break":
        w_s = int(max(8, min(window, n // 3)))
        if n < 2 * w_s + 2 or not np.all(np.isfinite(y_arr)):
            return {"status": "insufficient_data", "cohen_d": float("nan"), "p_permutation_scan_corrected": float("nan")}
        idx_s, _, ts = _local_linear_stats_vec(y_arr, w_s)
        if len(ts) == 0:
            return {"status": "no_finite_threshold", "cohen_d": float("nan"), "p_permutation_scan_corrected": float("nan")}
        pos = int(np.argmax(np.abs(ts)))
        best_i = int(idx_s[pos])
        best_d = float(ts[pos] / math.sqrt(w_s))
        p_corr = float("nan")
        if compute_scan_p:
            t_idx = np.arange(n, dtype=float)
            res_s = y_arr - np.polyval(np.polyfit(t_idx, y_arr, 1), t_idx)
            bl_s = int(max(2, min(8, w_s // 4)))
            rng = np.random.default_rng(seed)
            null_max: List[float] = []
            for _ in range(max(10, int(n_scan_perm))):
                rb = _block_permute(res_s, bl_s, rng)
                _, _, sim_ts = _local_linear_stats_vec(rb, w_s)
                if len(sim_ts):
                    null_max.append(float(np.max(np.abs(sim_ts))) / math.sqrt(w_s))
            if null_max:
                p_corr = (1.0 + float(np.sum(np.asarray(null_max) >= abs(best_d)))) / (len(null_max) + 1.0)
        return {
            "status": "ok",
            "idx": best_i,
            "control_value": float(x_arr[best_i]),
            "cohen_d": best_d,
            "p_permutation_scan_corrected": float(p_corr),
            "p_permutation_local": float(p_corr),
        }

    if detector == "detrended_jump":
        w_j = int(max(6, min(window, n // 3)))
        if n < 2 * w_j + 2 or not np.all(np.isfinite(y_arr)):
            return {"status": "insufficient_data", "cohen_d": float("nan"), "p_permutation_scan_corrected": float("nan")}
        idx_j, dj, _ = _local_linear_stats_vec(y_arr, w_j)
        if len(dj) == 0:
            return {"status": "no_finite_threshold", "cohen_d": float("nan"), "p_permutation_scan_corrected": float("nan")}
        pos = int(np.argmax(np.abs(dj)))
        best_i = int(idx_j[pos])
        best_d = float(dj[pos])
        p_corr = float("nan")
        if compute_scan_p:
            t_idx = np.arange(n, dtype=float)
            deg_j = min(3, max(1, n // 30))
            base_j = np.polyval(np.polyfit(t_idx, y_arr, deg_j), t_idx)
            res_j = y_arr - base_j
            bl_j = int(max(2, min(8, w_j // 4)))
            rng = np.random.default_rng(seed)
            null_max = []
            for _ in range(max(10, int(n_scan_perm))):
                rb = _block_permute(res_j, bl_j, rng)
                _, sim_dj, _ = _local_linear_stats_vec(base_j + rb, w_j)
                if len(sim_dj):
                    null_max.append(float(np.max(np.abs(sim_dj))))
            if null_max:
                p_corr = (1.0 + float(np.sum(np.asarray(null_max) >= abs(best_d)))) / (len(null_max) + 1.0)
        return {
            "status": "ok",
            "idx": best_i,
            "control_value": float(x_arr[best_i]),
            "cohen_d": best_d,
            "p_permutation_scan_corrected": float(p_corr),
            "p_permutation_local": float(p_corr),
        }

    return scan_threshold(
        x_arr,
        y_arr,
        window=window,
        min_group=min_group,
        n_boot=n_boot,
        n_perm=n_perm,
        seed=seed,
        compute_scan_p=compute_scan_p,
        n_scan_perm=n_scan_perm,
    )


def falsification_suite(
    ds: UniversalDataset,
    best_scan: Dict[str, Any],
    cfg: UniversalConfig,
) -> Dict[str, Any]:
    """Run rigorous falsification checks against multiple null hypotheses (P5).

    Fixes over v7.1.5:
    - All null scans use the EXACT same primary window ``w_primary`` as ``best_scan``.
    - Independent RNG streams per test (no cross-dataset identical white-noise artifacts).
    - Adds ``ar1_surrogate_null`` so pure random walks / smooth seasonal waves cannot
      masquerade as sharp regime breaks against an i.i.d. permutation null.
    - Tightens ``multi_window`` stability to penalize drift relative to ``0.20 * ptp(control)``.
    """
    target, control = ds.target, ds.control
    idx = best_scan.get("idx") if best_scan.get("status") == "ok" else None
    w_primary = int(
        best_scan.get("window_used")
        or (cfg.window_sizes[0] if cfg.window_sizes else 80)
    )
    n_scan_perm = int(getattr(cfg, "n_scan_permutations", 200))
    rng = np.random.default_rng(cfg.seed)
    out: Dict[str, Any] = {}

    detector = best_scan.get("detector", "level_shift")
    if detector == "volatility_shift" and len(target) >= 4:
        active_target = _volatility_deviation_series(np.asarray(target, dtype=float))
        active_control = np.asarray(control, dtype=float)[1:]
        active_idx = (
            int(max(0, min(len(active_target) - 1, int(idx) - 1)))
            if idx is not None
            else None
        )
    elif detector == "deseasonalized_shift" and len(target) >= 24:
        des = _deseasonalize_fft_dominant(np.asarray(target, dtype=float))
        active_target = des[0] if des is not None else np.asarray(target, dtype=float)
        active_control = np.asarray(control, dtype=float)
        active_idx = idx
    else:
        active_target = np.asarray(target, dtype=float)
        active_control = np.asarray(control, dtype=float)
        active_idx = idx

    # 1. Permuted control / target at fixed split (matched to active detector statistic)
    if active_idx is not None:
        w_eff_cut = int(max(8 if detector == "slope_break" else 6, min(w_primary, len(active_target) // 3)))
        perm_controls: List[float] = []
        for _ in range(30):
            perm_idx = rng.permutation(len(active_target))
            target_perm = active_target[perm_idx]
            if detector in ("slope_break", "detrended_jump"):
                d_p = abs(_local_linear_at_cut(target_perm, int(active_idx), w_eff_cut, detector))
            else:
                d_p = abs(cohen_d(target_perm[:active_idx], target_perm[active_idx:]))
            if math.isfinite(d_p):
                perm_controls.append(d_p)
        out["permuted_control_same_cut"] = {
            "note": "target permuted at fixed cut: |d| must collapse if split reflects real ordering",
            "d_observed": float(best_scan["cohen_d"]),
            "d_permuted_median": float(np.median(perm_controls)) if perm_controls else float("nan"),
        }

    # 2. Monotonic trend diagnostic (Spearman) on the ACTIVE detector series (P0.2 fix)
    #    The relevant series depends on the detector: deseasonalized target for
    #    deseasonalized_shift, volatility-deviation series for volatility_shift, raw otherwise.
    #    Using the raw target here missed strong monotone trends hidden by a seasonal cycle
    #    (e.g. monthly sea ice: raw rho ~ -0.22 but deseasonalized rho ~ -0.89).
    try:
        rho_active = float(
            pd.Series(active_control).corr(pd.Series(active_target), method="spearman")
        )
    except Exception:
        rho_active = float("nan")
    try:
        rho_raw = float(pd.Series(control).corr(pd.Series(target), method="spearman"))
    except Exception:
        rho_raw = float("nan")
    if best_scan.get("status") == "ok":
        best_scan["monotonicity_rho_active"] = rho_active
        best_scan["monotonicity_rho_raw"] = rho_raw
    out["trend_spearman"] = {
        "rho_control_target": rho_active,
        "rho_raw_target": rho_raw,
        "active_series": detector,
        "note": (
            "rho computed on the detector's active target (deseasonalized/volatility when "
            "applicable); |rho| >= 0.85 caps level_shift at C and shape detectors at B"
        ),
    }

    # 3. White-noise null matched to active detector statistic & window
    hash_salt = int(ds.source_hash[:8], 16) if ds.source_hash else 0
    rng_wn = np.random.default_rng(np.random.SeedSequence([int(cfg.seed), 101, hash_salt]))
    std_t = float(np.std(active_target)) if float(np.std(active_target)) > 0.0 else 1.0
    white = rng_wn.normal(float(np.mean(active_target)), std_t, size=len(active_target))
    w_scan = _scan_active_channel(
        active_control,
        white,
        detector=detector,
        window=w_primary,
        min_group=cfg.min_group,
        seed=cfg.seed + 1,
        n_boot=20,
        n_perm=50,
        compute_scan_p=True,
        n_scan_perm=n_scan_perm,
    )
    out["white_noise_null"] = {
        "cohen_d": _safe_float(w_scan.get("cohen_d")),
        "p_permutation": _safe_float(w_scan.get("p_permutation_scan_corrected")),
        "p_permutation_local": _safe_float(w_scan.get("p_permutation_local")),
        "detector_matched": detector,
        "window_used": w_primary,
        "expected": "high corrected p and low |d|; otherwise detector invents spurious breaks",
    }

    # 4. Shuffled-target null matched to active detector statistic & window
    rng_sh = np.random.default_rng(np.random.SeedSequence([int(cfg.seed), 202, hash_salt]))
    shuffled = rng_sh.permutation(active_target)
    s_scan = _scan_active_channel(
        active_control,
        shuffled,
        detector=detector,
        window=w_primary,
        min_group=cfg.min_group,
        seed=cfg.seed + 2,
        n_boot=20,
        n_perm=50,
        compute_scan_p=True,
        n_scan_perm=n_scan_perm,
    )
    out["shuffled_target_null"] = {
        "cohen_d": _safe_float(s_scan.get("cohen_d")),
        "p_permutation": _safe_float(s_scan.get("p_permutation_scan_corrected")),
        "p_permutation_local": _safe_float(s_scan.get("p_permutation_local")),
        "detector_matched": detector,
        "window_used": w_primary,
    }

    # 5. Autocorrelated surrogate null (AR(1), or integrated ARIMA(1,1,0) for smooth momentum)
    if active_idx is not None:
        phi_within = within_regime_ar1(active_target, int(active_idx))
        dy = np.diff(active_target)
        phi_dy = 0.0
        if len(dy) >= 10 and np.std(dy) > 0.0:
            phi_dy = float(np.corrcoef(dy[:-1], dy[1:])[0, 1])
            if not math.isfinite(phi_dy):
                phi_dy = 0.0
        use_arima = (phi_within >= 0.75) and (phi_dy >= 0.25)
        surrogate_model = "arima_1_1_0" if use_arima else "ar1"
        phi_sim = (
            float(phi_dy)
            if use_arima
            else (max(phi_within, 0.98) if (phi_within >= 0.80 and abs(phi_dy) < 0.25) else phi_within)
        )
        stat_name = "abs_t_stat" if detector == "slope_break" else ("abs_effect_d" if detector == "detrended_jump" else "abs_cohen_d")
        obs_stat = (
            abs(float(best_scan.get("t_stat", 0.0)))
            if detector == "slope_break"
            else abs(float(best_scan.get("cohen_d", 0.0)))
        )
        n_ar_perm = min(200, max(50, n_scan_perm))
        rng_ar = np.random.default_rng(np.random.SeedSequence([int(cfg.seed), 303, hash_salt]))
        ar_max: List[float] = []
        w_eff = int(max(8 if detector == "slope_break" else 6, min(w_primary, len(active_target) // 3)))
        for _ in range(n_ar_perm):
            sim = (
                _simulate_arima110(len(active_target), phi_dy, rng_ar)
                if use_arima
                else _simulate_ar1(len(active_target), phi_sim, 1.0, rng_ar)
            )
            if detector == "slope_break":
                _, _, ts_sim = _local_linear_stats_vec(sim, w_eff)
                v_sim = float(np.max(np.abs(ts_sim))) if len(ts_sim) else float("nan")
            elif detector == "detrended_jump":
                _, dj_sim, _ = _local_linear_stats_vec(sim, w_eff)
                v_sim = float(np.max(np.abs(dj_sim))) if len(dj_sim) else float("nan")
            else:
                d_sim, _ = _scan_max_d(active_control, sim, window=w_primary, min_group=cfg.min_group)
                v_sim = abs(d_sim) if math.isfinite(d_sim) else float("nan")
            if math.isfinite(v_sim):
                ar_max.append(v_sim)
        p_ar1 = (
            (1.0 + float(np.sum(np.asarray(ar_max) >= obs_stat))) / (len(ar_max) + 1.0)
            if ar_max
            else float("nan")
        )
        out["ar1_surrogate_null"] = {
            "phi_within": float(phi_within),
            "phi_diff1": float(phi_dy),
            "phi_simulated": float(phi_sim),
            "surrogate_model": surrogate_model,
            "statistic_tested": stat_name,
            "observed_stat": float(obs_stat),
            "p_ar1_scan_corrected": float(p_ar1),
            "ar1_null_p95": float(np.percentile(ar_max, 95)) if ar_max else float("nan"),
            "ar1_null_p95_abs_d": float(np.percentile(ar_max, 95)) if ar_max else float("nan"),
            "note": f"tests whether observed scan statistic exceeds a matched {surrogate_model.upper()} surrogate without break",
        }

    # 6. Split-half diagnostic matched to active detector (same relative window proportion)
    half = len(active_target) // 2
    w_half = max(cfg.min_group, min(w_primary, max(cfg.min_group, half // 3)))
    h1 = _scan_active_channel(
        active_control[:half],
        active_target[:half],
        detector=detector,
        window=w_half,
        min_group=cfg.min_group,
        seed=cfg.seed + 3,
        n_boot=20,
        n_perm=50,
        compute_scan_p=True,
        n_scan_perm=min(100, n_scan_perm),
    )
    h2 = _scan_active_channel(
        active_control[half:],
        active_target[half:],
        detector=detector,
        window=w_half,
        min_group=cfg.min_group,
        seed=cfg.seed + 4,
        n_boot=20,
        n_perm=50,
        compute_scan_p=True,
        n_scan_perm=min(100, n_scan_perm),
    )
    d1 = _safe_float(h1.get("cohen_d"))
    d2 = _safe_float(h2.get("cohen_d"))
    p1_h = _safe_float(h1.get("p_permutation_scan_corrected"))
    p2_h = _safe_float(h2.get("p_permutation_scan_corrected"))
    opposite_sig = (
        math.isfinite(d1)
        and math.isfinite(d2)
        and (d1 * d2 < 0.0)
        and math.isfinite(p1_h)
        and math.isfinite(p2_h)
        and p1_h <= 0.05
        and p2_h <= 0.05
    )
    out["split_half"] = {
        "half1_idx": h1.get("idx"),
        "half1_d": d1,
        "half1_p_scan_corrected": p1_h,
        "half2_idx": h2.get("idx"),
        "half2_d": d2,
        "half2_p_scan_corrected": p2_h,
        "opposite_significant_halves": bool(opposite_sig),
        "agreement": "opposite strong splits in both halves suggest wandering/non-stationary drift rather than one stable threshold",
    }

    # 7. Multi-window stability (penalized wrt 20% of control span so random picks don't score 0.9+)
    scans: List[float] = []
    for w in cfg.window_sizes:
        if detector == "detrended_jump" and np.all(np.isfinite(target)) and len(target) >= 2 * max(6, w) + 2:
            w_eff = int(max(6, min(w, len(target) // 3)))
            idx_j, dj, _ = _local_linear_stats_vec(np.asarray(target, dtype=float), w_eff)
            if len(dj):
                best_i = int(idx_j[int(np.argmax(np.abs(dj)))])
                scans.append(float(control[best_i]))
        elif detector == "slope_break" and np.all(np.isfinite(target)) and len(target) >= 2 * max(8, w) + 2:
            w_eff = int(max(8, min(w, len(target) // 3)))
            idx_s, _, ts = _local_linear_stats_vec(np.asarray(target, dtype=float), w_eff)
            if len(ts):
                best_is = int(idx_s[int(np.argmax(np.abs(ts)))])
                scans.append(float(control[best_is]))
        else:
            s = scan_threshold(
                active_control,
                active_target,
                window=w,
                min_group=cfg.min_group,
                n_boot=20,
                n_perm=50,
                seed=cfg.seed,
                compute_scan_p=False,
            )
            if s.get("status") == "ok":
                scans.append(float(s["control_value"]))
    ctrl_span = float(np.ptp(control)) if float(np.ptp(control)) > 0.0 else 1.0
    if len(scans) >= 2:
        stab = max(0.0, 1.0 - float(np.std(scans)) / (0.20 * ctrl_span))
    else:
        stab = 0.0
    out["multi_window"] = {
        "threshold_values": scans,
        "std_thresholds": float(np.std(scans)) if len(scans) >= 2 else float("nan"),
        "stability": float(stab),
        "n_windows": len(scans),
        "note": (
            "single valid window = stability NOT proven (0.0)"
            if len(scans) < 2
            else "stability measured against 20% of control span"
        ),
    }

    # 8. Multi-seed permutation stability
    ps = [
        permutation_p_value(active_target[:active_idx], active_target[active_idx:], n_perm=100, seed=cfg.seed + i)
        if active_idx is not None
        else float("nan")
        for i in range(5)
    ]
    out["multi_seed_p"] = {
        "p_values": ps,
        "p_max": float(np.nanmax(ps)) if len(ps) else float("nan"),
    }

    # 9. Optional harmonic ablation test (P4)
    if cfg.harmonic_ablation and idx is not None:
        harm_ds: List[float] = []
        for trial in range(10):
            r = np.random.default_rng(cfg.seed + 100 + trial)
            mask = r.random(len(target)) < cfg.harmonic_prob
            jitter = np.zeros_like(target)
            scale_j = 0.05 * std_t * (PHI if trial % 2 == 0 else PI)
            jitter[mask] = r.normal(0.0, scale_j, size=int(np.sum(mask)))
            hs = scan_threshold(
                control,
                target + jitter,
                window=w_primary,
                min_group=cfg.min_group,
                n_boot=20,
                n_perm=50,
                seed=cfg.seed + 100 + trial,
                compute_scan_p=False,
            )
            harm_ds.append(_safe_float(hs.get("cohen_d")))
        out["harmonic_ablation"] = {
            "cohen_d_with_harmonic_jitter_median": float(np.nanmedian(harm_ds))
            if harm_ds
            else float("nan"),
            "cohen_d_observed": float(best_scan["cohen_d"]),
            "interpretation": "if median collapses under mild perturbation, reject threshold",
        }

    return out


def walk_forward(
    ds: UniversalDataset,
    best_scan: Dict[str, Any],
    cfg: UniversalConfig,
    n_splits: int = 5,
) -> Dict[str, Any]:
    """Honest walk-forward validation counting abstentions explicitly.

    Trains threshold on past prefix ``[:train_end]``, evaluates on future block
    ``[edges[s+1]:edges[s+2]]`` using consistent effect scales matching the active detector.
    """
    if best_scan.get("status") != "ok":
        return {"status": "no_threshold"}

    detector = best_scan.get("detector", "level_shift")
    n = ds.n
    edges = np.linspace(0, n, n_splits + 1).astype(int)
    successes, trials, abstentions = 0, 0, 0
    details: List[Dict[str, Any]] = []
    w_primary = int(
        best_scan.get("window_used")
        or (cfg.window_sizes[0] if cfg.window_sizes else 80)
    )

    if detector == "slope_break":
        for s in range(n_splits - 1):
            train_end = int(edges[s + 1])
            test_a, test_b = int(edges[s + 1]), int(edges[s + 2])
            if test_b - test_a < 8 or train_end < 2 * max(8, cfg.min_group) + 4:
                continue
            w_tr = max(8, min(w_primary, train_end // 3))
            idx_s, _, ts = _local_linear_stats_vec(ds.target[:train_end], w_tr)
            if len(ts) == 0:
                continue
            pos = int(np.argmax(np.abs(ts)))
            t_max = float(ts[pos])
            if abs(t_max) < 3.0:
                abstentions += 1
                details.append({"split": s, "abstained": True, "t_train": t_max})
                continue
            cut_tr = int(idx_s[pos])
            t_pre = np.arange(cut_tr, dtype=float)
            t_post = np.arange(train_end - cut_tr, dtype=float)
            if len(t_pre) < 4 or len(t_post) < 4:
                continue
            b_pre = float(np.polyfit(t_pre, ds.target[:cut_tr], 1)[0])
            b_post = float(np.polyfit(t_post, ds.target[cut_tr:train_end], 1)[0])
            b_test = float(np.polyfit(np.arange(test_b - test_a, dtype=float), ds.target[test_a:test_b], 1)[0])
            ok = abs(b_test - b_post) < abs(b_test - b_pre)
            successes += int(ok)
            trials += 1
            details.append(
                {
                    "split": s,
                    "b_pre": b_pre,
                    "b_post": b_post,
                    "b_test": b_test,
                    "success": bool(ok),
                    "case": "slope_continuation",
                }
            )
    else:
        if detector == "volatility_shift" and len(ds.target) >= 4:
            wf_y = _volatility_deviation_series(ds.target)
            wf_ctrl = ds.control[1:]
        elif detector == "deseasonalized_shift" and len(ds.target) >= 24:
            des = _deseasonalize_fft_dominant(ds.target)
            wf_y = des[0] if des is not None else ds.target
            wf_ctrl = ds.control
        else:
            wf_y = ds.target
            wf_ctrl = ds.control
        wf_n = len(wf_y)
        edges = np.linspace(0, wf_n, n_splits + 1).astype(int)

        for s in range(n_splits - 1):
            train_end = int(edges[s + 1])
            if train_end < 2 * cfg.min_group + 4:
                continue
            if detector == "detrended_jump":
                w_tr = max(6, min(w_primary, train_end // 3))
                idx_j, dj, _ = _local_linear_stats_vec(wf_y[:train_end], w_tr)
                if len(dj) == 0:
                    continue
                pos = int(np.argmax(np.abs(dj)))
                d_train = float(dj[pos])
                if abs(d_train) < 2.0:
                    abstentions += 1
                    details.append({"split": s, "abstained": True, "d_train": d_train})
                    continue
                cut_tr = int(idx_j[pos])
                thr = float(wf_ctrl[cut_tr])
                train_scan = {"status": "ok", "control_value": thr, "cohen_d": d_train}
            else:
                train_scan = scan_threshold(
                    wf_ctrl[:train_end],
                    wf_y[:train_end],
                    window=w_primary,
                    min_group=cfg.min_group,
                    n_boot=20,
                    n_perm=50,
                    seed=cfg.seed + s,
                    compute_scan_p=False,
                )
                if train_scan.get("status") != "ok":
                    continue
                if abs(train_scan.get("cohen_d", 0.0)) < 0.5:
                    abstentions += 1
                    details.append(
                        {
                            "split": s,
                            "abstained": True,
                            "d_train": float(train_scan.get("cohen_d", 0.0)),
                        }
                    )
                    continue

            thr = float(train_scan["control_value"])
            d_train = float(train_scan["cohen_d"])
            test_a, test_b = int(edges[s + 1]), int(edges[s + 2])
            if test_b - test_a < 8:
                continue

            test_ctrl, test_y = wf_ctrl[test_a:test_b], wf_y[test_a:test_b]
            train_ctrl, train_y = wf_ctrl[:train_end], wf_y[:train_end]
            train_pre = train_y[train_ctrl <= thr]
            train_post = train_y[train_ctrl > thr]
            mask_post = test_ctrl > thr

            if int(np.sum(mask_post)) >= 4 and int(np.sum(~mask_post)) >= 4:
                d_test = cohen_d(test_y[~mask_post], test_y[mask_post])
                ok = math.isfinite(d_test) and (d_test * d_train > 0.0) and abs(d_test) > 0.5
                successes += int(ok)
                trials += 1
                details.append(
                    {
                        "split": s,
                        "d_train": d_train,
                        "d_test": float(d_test),
                        "success": bool(ok),
                        "case": "both_sides",
                    }
                )
            elif int(np.sum(mask_post)) >= 4:
                if len(train_pre) < 4 or len(train_post) < 4:
                    continue
                d_vs_pre = cohen_d(train_pre, test_y)
                d_vs_post = cohen_d(train_post, test_y)
                ok = (
                    math.isfinite(d_vs_pre)
                    and math.isfinite(d_vs_post)
                    and abs(d_vs_post) < abs(d_vs_pre)
                    and abs(d_vs_pre) > 0.5
                )
                successes += int(ok)
                trials += 1
                details.append(
                    {
                        "split": s,
                        "d_train": d_train,
                        "d_test_future_vs_pre": float(d_vs_pre),
                        "d_test_future_vs_post": float(d_vs_post),
                        "success": bool(ok),
                        "case": "future_all_post",
                    }
                )
            elif int(np.sum(~mask_post)) >= 4:
                if len(train_pre) < 4 or len(train_post) < 4:
                    continue
                d_vs_pre = cohen_d(train_pre, test_y)
                d_vs_post = cohen_d(train_post, test_y)
                ok = (
                    math.isfinite(d_vs_pre)
                    and math.isfinite(d_vs_post)
                    and abs(d_vs_pre) < abs(d_vs_post)
                    and abs(d_vs_post) > 0.5
                )
                successes += int(ok)
                trials += 1
                details.append(
                    {
                        "split": s,
                        "d_train": d_train,
                        "d_test_future_vs_pre": float(d_vs_pre),
                        "d_test_future_vs_post": float(d_vs_post),
                        "success": bool(ok),
                        "case": "future_all_pre",
                    }
                )

    total = trials + abstentions
    if trials == 0:
        return {
            "status": "insufficient_splits",
            "mean_precision": 0.0,
            "mean_recall": 0.0,
            "mean_f1": 0.0,
            "mean_f1_conservative": 0.0,
            "n_splits_used": 0,
            "n_abstentions": int(abstentions),
            "n_splits_total": int(total),
            "successes": 0,
            "details": details,
        }

    f1 = successes / trials
    f1_cons = successes / total if total > 0 else 0.0
    return {
        "status": "ok",
        "mean_precision": float(f1),
        "mean_recall": float(f1),
        "mean_f1": float(f1),
        "mean_f1_conservative": float(f1_cons),
        "n_splits_used": int(trials),
        "n_abstentions": int(abstentions),
        "n_splits_total": int(total),
        "successes": int(successes),
        "details": details,
    }


def grade_evidence(
    best_scan: Dict[str, Any],
    falsif: Dict[str, Any],
    wf: Dict[str, Any],
    cfg: UniversalConfig,
) -> Dict[str, Any]:
    """Assign conservative evidence grade A/B/C/D for threshold existence (never prediction proof)."""
    rules = cfg.grade_rules
    if best_scan.get("status") != "ok":
        return {
            "grade": "D",
            "verdict": "no evidence — no finite threshold found",
            "reasons": ["scan produced no finite threshold"],
            "checks": {},
            "n_checks_passed": 0,
        }

    reasons: List[str] = []
    detector = best_scan.get("detector", "level_shift")

    ok_d = abs(best_scan["cohen_d"]) >= rules["min_cohen_d"]
    p_perm = best_scan.get("p_channel_adjusted", best_scan.get("p_scan_null"))
    null_model = str(best_scan.get("null_model", "target_permutation_scan"))
    n_fam = best_scan.get("n_channels_family")
    p_desc = (
        f"p[{null_model}; holm_m={n_fam}]"
        if (isinstance(n_fam, int) and n_fam > 1 and best_scan.get("p_channel_adjusted") is not None)
        else f"p[{null_model}]"
    )
    ok_p = p_perm is not None and math.isfinite(p_perm) and p_perm <= rules["max_p_perm"]

    stab = falsif.get("multi_window", {}).get("stability", 0.0)
    ok_stab = stab >= rules["min_stability"]

    wf_f1 = wf.get("mean_f1", 0.0) if isinstance(wf, dict) else 0.0
    wf_votes = wf.get("n_splits_used", 0) if isinstance(wf, dict) else 0
    ok_wf = (wf_f1 >= rules["min_walkforward_f1"]) and (wf_votes >= 2)

    if not ok_d:
        reasons.append(f"|effect_d|={abs(best_scan['cohen_d']):.2f} < {rules['min_cohen_d']}")
    if not ok_p:
        reasons.append(f"{p_desc}={p_perm} > {rules['max_p_perm']}")
    if not ok_stab:
        reasons.append(f"multi-window stability={stab:.2f} < {rules['min_stability']}")
    if not ok_wf:
        if wf_votes < 2:
            abst = wf.get("n_abstentions", 0) if isinstance(wf, dict) else 0
            reasons.append(f"walk-forward: only {wf_votes} voted split(s), need >= 2 (abstentions={abst})")
        else:
            reasons.append(f"walk-forward F1={wf_f1:.2f} < {rules['min_walkforward_f1']}")

    # White noise null check
    wn = falsif.get("white_noise_null", {})
    ok_white = True
    if math.isfinite(wn.get("cohen_d", float("nan"))) and math.isfinite(best_scan.get("cohen_d", float("nan"))):
        ok_white = abs(wn["cohen_d"]) < 0.5 * abs(best_scan["cohen_d"])
        if not ok_white:
            reasons.append("white-noise null achieved comparable |d| — non-specific detector")

    # Permuted control at fixed cut check
    pc = falsif.get("permuted_control_same_cut", {})
    ok_perm_ctrl = True
    if math.isfinite(pc.get("d_permuted_median", float("nan"))):
        ok_perm_ctrl = abs(pc["d_permuted_median"]) < 0.5 * abs(best_scan["cohen_d"])
        if not ok_perm_ctrl:
            reasons.append("permuted control did not collapse effect size")

    # Shuffled target null check
    sh = falsif.get("shuffled_target_null", {})
    ok_shuffled = True
    if math.isfinite(sh.get("cohen_d", float("nan"))):
        ok_shuffled = abs(sh["cohen_d"]) < 0.5 * abs(best_scan["cohen_d"])
        if not ok_shuffled:
            reasons.append("shuffled-target null produced comparable |d|")

    # Autocorrelated surrogate null check across ALL detectors and trends (AR(1) or ARIMA(1,1,0))
    rho = falsif.get("trend_spearman", {}).get("rho_control_target", float("nan"))
    is_strong_monotone = math.isfinite(rho) and abs(rho) >= 0.85
    ar1 = falsif.get("ar1_surrogate_null", {})
    ok_ar1 = True
    p_ar1 = ar1.get("p_ar1_scan_corrected", float("nan"))
    if math.isfinite(p_ar1):
        phi_sim = float(ar1.get("phi_simulated", 0.0))
        phi_within_ar = float(ar1.get("phi_within", phi_sim))
        surr_model = str(ar1.get("surrogate_model", "ar1"))
        triggered = (phi_within_ar >= 0.65) or (phi_sim >= 0.65) or (surr_model == "arima_1_1_0")
        if triggered and p_ar1 > rules["max_p_perm"]:
            ok_ar1 = False
            reasons.append(
                f"{surr_model.upper()} surrogate null failed ({detector}, phi={phi_sim:.2f}, p_ar1={p_ar1:.3f} > {rules['max_p_perm']}) — break may be autocorrelation/random-walk artifact"
            )

    # Split-half opposite significant signs check
    shalf = falsif.get("split_half", {})
    opp_sig = bool(shalf.get("opposite_significant_halves", False))
    if opp_sig:
        reasons.append("split-half produced opposite significant effects — non-stationary drift suspected")

    checks_list = [ok_d, ok_p, ok_stab, ok_wf, ok_white]
    n_ok = sum(checks_list)
    grade = "A" if n_ok == 5 and not reasons else ("B" if n_ok >= 4 else ("C" if n_ok == 3 else "D"))

    # Hard statistical caps:
    # 1) If scan-corrected null test fails, cap at D.
    if not ok_p:
        grade = "D"

    # 2) Strong monotonic trend cap (Spearman |rho| >= 0.85):
    #    level_shift on a smooth monotone axis -> cap at C (a mean split on a trend is not evidence)
    #    other detectors (slope_break, detrended_jump, etc.) -> cap at B (they measure shape changes)
    if is_strong_monotone:
        reasons.append(
            f"strong monotonic trend Spearman rho={rho:.2f} (detector={detector}) — break threshold may reflect smooth trend rather than discrete shift"
        )
        if detector == "level_shift":
            if grade in ("A", "B"):
                grade = "C"
        else:
            if grade == "A":
                grade = "B"

    # 3) Hard cap at C if multi-window stability or walk-forward fails (never call unstable/unvalidated break Grade B)
    if (not ok_stab or not ok_wf) and grade in ("A", "B"):
        grade = "C"

    # 4) Non-specific null or AR/random-walk failure or collapsing permutation control: cap at C
    if (not ok_white or not ok_perm_ctrl or not ok_shuffled or not ok_ar1 or opp_sig) and grade in ("A", "B"):
        grade = "C"

    # 5) Compound falsification failure -> Grade D (no evidence):
    #    - Failing the autocorrelated surrogate alongside instability, WF failure, white-noise failure, or split-half reversal
    #    - Multi-window instability combined with opposite significant split halves (non-stationary wandering drift)
    compound_ar = (not ok_ar1) and (not ok_stab or not ok_wf or not ok_white or opp_sig)
    compound_drift = (not ok_stab) and opp_sig
    if (compound_ar or compound_drift) and grade != "D":
        reasons.append(
            "compound falsification failure (autocorrelated/wandering non-stationary drift) — downgraded to Grade D"
        )
        grade = "D"

    if detector == "deseasonalized_shift" and is_strong_monotone and grade == "B":
        reasons.append(
            "boundary case: FFT deseasonalization followed by a mean shift on a strongly monotonic series "
            "is the border between a regime change and a trend with its seasonal cycle removed; "
            "Grade B names that border and is not a stronger claim"
        )

    verdict = {
        "A": "strong evidence (under this protocol — independent replication required)",
        "B": "moderate evidence",
        "C": "weak evidence — do not base decisions on this threshold",
        "D": "no evidence — do not announce a threshold",
    }[grade]

    return {
        "grade": grade,
        "verdict": verdict,
        "reasons": reasons,
        "checks": {
            "cohen_d_ok": ok_d,
            "p_perm_ok": ok_p,
            "stability_ok": ok_stab,
            "walkforward_ok": ok_wf,
            "white_null_ok": ok_white,
            "perm_control_ok": ok_perm_ctrl,
            "shuffled_null_ok": ok_shuffled,
            "ar1_surrogate_ok": ok_ar1,
            "split_half_consistent": not opp_sig,
        },
        "n_checks_passed": int(n_ok),
    }

## Module `levels.py`

In [ ]:
"""Universal Levels L1..L7 and segmented estimator interaction maps."""


import math
from typing import Any, Dict, List, Optional

import numpy as np



def compute_interactions(
    ds: UniversalDataset,
    cascade: Dict[str, Any],
    cfg: UniversalConfig,
) -> Dict[str, Any]:
    """Compute L7 convergence and divergence across a fixed grid centered on T0."""
    n = ds.n
    t0_idx: Optional[int] = None
    if isinstance(cascade, dict):
        v0 = cascade.get("T0", {})
        if isinstance(v0, dict) and v0.get("idx") is not None and math.isfinite(v0.get("idx", float("nan"))):
            t0_idx = int(v0["idx"])

    if t0_idx is None:
        return {
            "status": "insufficient_data",
            "segments": [],
            "per_flux": {},
            "summary": {
                "n_segments": 0,
                "n_fluxes": 0,
                "total_complementary_pairs": 0,
                "definition": "no T0 available; L7 grid unavailable",
            },
        }

    w = max(20, n // 10)
    bidx = sorted({b for b in [t0_idx - 2 * w, t0_idx - w, t0_idx + w, t0_idx + 2 * w] if 0 < b < n})
    min_sep = max(10, n // 20)
    kept: List[int] = []
    for b in bidx:
        if not kept or (b - kept[-1] >= min_sep and n - b >= min_sep):
            kept.append(b)
    bounds = [0] + kept + [n]

    segments: List[Dict[str, Any]] = []
    for s in range(len(bounds) - 1):
        a, b = bounds[s], bounds[s + 1]
        if b <= a:
            continue
        if b <= t0_idx:
            pos = "pre_T0"
        elif a >= t0_idx:
            pos = "post_T0"
        else:
            pos = "at_T0"
        segments.append(
            {
                "segment": s,
                "start_idx": int(a),
                "end_idx": int(b),
                "length": int(b - a),
                "control_start": float(ds.control[a]),
                "control_end": float(ds.control[b - 1]),
                "position": pos,
            }
        )

    per_flux: Dict[str, Any] = {}
    total_pairs = 0
    for fname in ds.flux_names:
        win_vals: List[List[Dict[str, float]]] = []
        series = ds.fluxes[fname]
        win_len = min(128, max(40, len(series) // 12))
        est_per_seg: List[Dict[str, float]] = []

        for seg in segments:
            seg_series = series[seg["start_idx"] : seg["end_idx"]]
            if len(seg_series) < win_len:
                est_per_seg.append({name: float("nan") for name, _, _ in UNIQUE_ESTIMATORS})
                win_vals.append([])
                continue
            windows = [seg_series[i : i + win_len] for i in range(0, len(seg_series) - win_len + 1, 5)]
            per_win = [apply_estimators(win) for win in windows]
            med: Dict[str, float] = {}
            for name, _, _ in UNIQUE_ESTIMATORS:
                vals = [d.get(name, float("nan")) for d in per_win]
                vals = [v for v in vals if math.isfinite(v)]
                med[name] = float(np.median(vals)) if vals else float("nan")
            est_per_seg.append(med)
            win_vals.append(per_win)

        profiles: Dict[str, Any] = {}
        for name, _, _ in UNIQUE_ESTIMATORS:
            vals = [est_per_seg[s].get(name, float("nan")) for s in range(len(segments))]
            finite = [v for v in vals if math.isfinite(v)]
            if len(finite) < 2:
                profiles[name] = {
                    "values_per_segment": vals,
                    "median_across_segments": float("nan"),
                    "informativeness_per_segment": [float("nan")] * len(segments),
                    "peak_segment": None,
                    "peak_informativeness": float("nan"),
                    "delta_post_minus_pre": float("nan"),
                    "sign_post_minus_pre": "unknown",
                }
                continue

            med_val = float(np.median(finite))
            infos: List[float] = []
            for s_idx in range(len(segments)):
                seg_win = [
                    d.get(name, float("nan"))
                    for d in (win_vals[s_idx] if s_idx < len(win_vals) else [])
                ]
                seg_win = [v for v in seg_win if math.isfinite(v)]
                oth_win: List[float] = []
                for o_idx in range(len(segments)):
                    if o_idx == s_idx:
                        continue
                    oth_win.extend(
                        d.get(name, float("nan"))
                        for d in (win_vals[o_idx] if o_idx < len(win_vals) else [])
                    )
                oth_win = [v for v in oth_win if math.isfinite(v)]
                if len(seg_win) >= 3 and len(oth_win) >= 5:
                    d_seg, lo_seg, hi_seg = bootstrap_ci_cohen_d(
                        np.asarray(oth_win),
                        np.asarray(seg_win),
                        n_boot=100,
                        seed=cfg.seed,
                    )
                    if (
                        math.isfinite(d_seg)
                        and math.isfinite(lo_seg)
                        and math.isfinite(hi_seg)
                        and (lo_seg > 0.0 or hi_seg < 0.0)
                    ):
                        infos.append(abs(d_seg))
                    else:
                        infos.append(0.0)
                else:
                    infos.append(float("nan"))

            fin_infos = [x for x in infos if math.isfinite(x)]
            peak = (
                int(np.nanargmax(np.asarray([x if math.isfinite(x) else -1.0 for x in infos])))
                if fin_infos
                else None
            )
            pre_vals = [
                vals[s]
                for s, seg in enumerate(segments)
                if seg["position"] == "pre_T0" and math.isfinite(vals[s])
            ]
            post_vals = [
                vals[s]
                for s, seg in enumerate(segments)
                if seg["position"] == "post_T0" and math.isfinite(vals[s])
            ]
            delta = float(np.mean(post_vals) - np.mean(pre_vals)) if pre_vals and post_vals else float("nan")
            sign = (
                ("up" if delta > 0.0 else ("down" if delta < 0.0 else "flat"))
                if math.isfinite(delta)
                else "unknown"
            )
            profiles[name] = {
                "values_per_segment": vals,
                "median_across_segments": med_val,
                "informativeness_per_segment": infos,
                "peak_segment": peak,
                "peak_informativeness": float(max(fin_infos)) if fin_infos else float("nan"),
                "delta_post_minus_pre": delta,
                "sign_post_minus_pre": sign,
            }

        signs = [
            pr["sign_post_minus_pre"]
            for pr in profiles.values()
            if pr["sign_post_minus_pre"] in ("up", "down")
        ]
        n_up = sum(1 for s in signs if s == "up")
        n_down = sum(1 for s in signs if s == "down")
        majority = "up" if (signs and n_up >= n_down) else ("down" if signs else "unknown")
        frac_maj = (max(n_up, n_down) / len(signs)) if signs else 0.0

        peaks = [pr["peak_segment"] for pr in profiles.values() if pr["peak_segment"] is not None]
        n_distinct_peaks = len(set(peaks)) if peaks else 0
        top_div = sorted(
            [
                (name, pr["peak_segment"], pr["peak_informativeness"])
                for name, pr in profiles.items()
                if pr["peak_segment"] is not None and math.isfinite(pr["peak_informativeness"])
            ],
            key=lambda x: x[2],
            reverse=True,
        )[:5]

        pairs: List[Dict[str, Any]] = []
        pos_map = {s: seg["position"] for s, seg in enumerate(segments)}
        for a_name, a_pr in profiles.items():
            if a_pr["peak_segment"] is None:
                continue
            if not math.isfinite(a_pr["peak_informativeness"]) or a_pr["peak_informativeness"] < 0.8:
                continue
            for b_name, b_pr in profiles.items():
                if b_name == a_name or b_pr["peak_segment"] is None:
                    continue
                if b_pr["peak_segment"] <= a_pr["peak_segment"]:
                    continue
                if not math.isfinite(b_pr["peak_informativeness"]) or b_pr["peak_informativeness"] < 0.8:
                    continue
                is_pre_post = (
                    pos_map.get(a_pr["peak_segment"]) == "pre_T0"
                    and pos_map.get(b_pr["peak_segment"]) == "post_T0"
                )
                pairs.append(
                    {
                        "estimator_a": a_name,
                        "estimator_b": b_name,
                        "peak_segment_a": int(a_pr["peak_segment"]),
                        "peak_segment_b": int(b_pr["peak_segment"]),
                        "position_a": pos_map.get(a_pr["peak_segment"]),
                        "position_b": pos_map.get(b_pr["peak_segment"]),
                        "informativeness_a": float(a_pr["peak_informativeness"]),
                        "informativeness_b": float(b_pr["peak_informativeness"]),
                        "min_informativeness": float(
                            min(a_pr["peak_informativeness"], b_pr["peak_informativeness"])
                        ),
                        "estimator_pre": a_name if is_pre_post else None,
                        "estimator_post": b_name if is_pre_post else None,
                    }
                )

        pairs = sorted(pairs, key=lambda x: x["min_informativeness"], reverse=True)[:10]
        total_pairs += len(pairs)

        per_seg_top: Dict[str, List[Dict[str, Any]]] = {}
        for s in range(len(segments)):
            cand = sorted(
                [
                    (name, pr["informativeness_per_segment"][s])
                    for name, pr in profiles.items()
                    if len(pr["informativeness_per_segment"]) > s
                    and math.isfinite(pr["informativeness_per_segment"][s])
                ],
                key=lambda x: x[1],
                reverse=True,
            )[:5]
            per_seg_top[f"segment_{s}"] = [
                {"estimator": nm, "informativeness": float(v)} for nm, v in cand
            ]

        per_flux[fname] = {
            "profiles": profiles,
            "convergence": {
                "n_up": int(n_up),
                "n_down": int(n_down),
                "majority_sign": majority,
                "fraction_majority": float(frac_maj),
                "note": "convergence is a tally, not forced consensus; divergence is also informative",
            },
            "divergence": {
                "n_distinct_peak_segments": int(n_distinct_peaks),
                "top_divergent": [
                    {"estimator": nm, "peak_segment": int(ps), "peak_informativeness": float(v)}
                    for nm, ps, v in top_div
                ],
            },
            "complementary_pairs": pairs,
            "per_segment_top": per_seg_top,
        }

    return {
        "status": "ok",
        "segments": segments,
        "per_flux": per_flux,
        "summary": {
            "n_segments": len(segments),
            "n_fluxes": len(ds.flux_names),
            "n_estimators_distinct": len(UNIQUE_ESTIMATORS),
            "n_estimator_names": len(ESTIMATORS),
            "total_complementary_pairs": int(total_pairs),
            "definition": "convergence and divergence are complementary signals; never average distinct lenses",
        },
    }


def level_universal(
    ds: UniversalDataset,
    best_scan: Dict[str, Any],
    cascade: Dict[str, Any],
    est_by_flux: Dict[str, Dict[str, float]],
    cfg: UniversalConfig,
    falsif: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """Compute domain-agnostic structural levels L1..L7."""
    target, control = ds.target, ds.control
    levels: Dict[str, Any] = {}

    if best_scan.get("status") == "ok":
        d = best_scan["cohen_d"]
        direction = "up" if d > 0.05 else ("down" if d < -0.05 else "flat")
        horizon_dirs: List[float] = []
        for h in cfg.horizons:
            idx_h = int(np.clip(best_scan["idx"] + h, cfg.min_group, len(target) - cfg.min_group - 1))
            dh = cohen_d(target[:idx_h], target[idx_h:])
            if math.isfinite(dh):
                horizon_dirs.append(dh)
        stab = stability_weight(horizon_dirs)
        levels["L1_direction"] = {
            "status": "ok",
            "direction": direction,
            "cohen_d": float(d),
            "ci_low": best_scan.get("cohen_d_ci_low"),
            "ci_high": best_scan.get("cohen_d_ci_high"),
            "p_permutation": best_scan.get("p_permutation"),
            "confidence": float(stab),
            "multi_horizon": {
                "horizons": cfg.horizons,
                "cohen_d_by_horizon": horizon_dirs,
                "stability": float(stab),
            },
            "definition": "direction = sign(empirical post-pre effect). Invariant to column names.",
        }

        # L2 Magnitude: stability reflects multi-horizon consistency rather than trivial singleton 1.0
        mean_pre = best_scan.get("mean_pre", float("nan"))
        mean_post = best_scan.get("mean_post", float("nan"))
        rel_change = (
            float((mean_post - mean_pre) / (abs(mean_pre) if mean_pre != 0.0 else 1.0))
            if math.isfinite(mean_pre) and math.isfinite(mean_post)
            else float("nan")
        )
        levels["L2_magnitude"] = {
            "status": "ok",
            "stat_kind": best_scan.get("stat_kind", "mean"),
            "pre_stat": best_scan.get("pre_stat", mean_pre),
            "post_stat": best_scan.get("post_stat", mean_post),
            "abs_cohen_d": float(abs(d)),
            "mean_pre": mean_pre,
            "mean_post": mean_post,
            "relative_change": rel_change,
            "confidence": float(stab),
        }

        # L3 Regime position relative to threshold
        frac_post = float(np.mean(control > best_scan["control_value"]))
        state = "pre" if frac_post < 0.15 else ("post" if frac_post > 0.85 else "transition")
        mw_stab = (
            falsif.get("multi_window", {}).get("stability", 0.0)
            if isinstance(falsif, dict)
            else 0.0
        )
        levels["L3_regime"] = {
            "status": "ok",
            "state": state,
            "fraction_post_threshold": frac_post,
            "confidence": float(mw_stab),
        }

        # L4 Signal strength
        p_corr = best_scan.get("p_channel_adjusted", best_scan.get("p_scan_null", best_scan.get("p_permutation", float("nan"))))
        sig_level = (
            "strong"
            if abs(d) > 1.0 and p_corr is not None and math.isfinite(p_corr) and p_corr <= 0.05
            else ("moderate" if abs(d) > 0.5 else "weak")
        )
        levels["L4_signal"] = {
            "status": "ok",
            "level": sig_level,
            "abs_cohen_d": float(abs(d)),
            "null_model": best_scan.get("null_model"),
            "p_channel_raw": best_scan.get("p_channel_raw", best_scan.get("p_scan_null")),
            "p_channel_adjusted": best_scan.get("p_channel_adjusted"),
            "n_channels_family": best_scan.get("n_channels_family"),
            "p_scan_null": best_scan.get("p_scan_null"),
            "p_permutation_scan_corrected": best_scan.get("p_permutation_scan_corrected"),
            "p_permutation_local": best_scan.get("p_permutation_local"),
            "p_permutation_global": best_scan.get("p_permutation_global"),
            "definition": "signal strength = |effect| + multiplicity-adjusted null p-value; not a risk score or probability",
        }
    else:
        levels["L1_direction"] = {"status": "insufficient_data"}
        levels["L2_magnitude"] = {"status": "insufficient_data"}
        levels["L3_regime"] = {"status": "insufficient_data"}
        levels["L4_signal"] = {"status": "insufficient_data", "level": "unknown"}

    # L5 Complexity: single-pass before/after T0 dimension roughness contrast (always on default path)
    # Evaluates the 13 unique dimension estimators (same subset displayed in reports/L5 contrast).
    # L7 sliding-window interactions & predict_universal() remain behind --complexity-diagnostics.
    t0_idx: Optional[int] = None
    if best_scan.get("status") == "ok" and best_scan.get("idx") is not None:
        t0_idx = int(best_scan["idx"])

    if t0_idx is not None and t0_idx > 15 and (ds.n - t0_idx) > 15:
        left_seg = ds.target[:t0_idx]
        right_seg = ds.target[t0_idx:]
        est_left = apply_dimension_estimators(left_seg)
        est_right = apply_dimension_estimators(right_seg)
        contrast: Dict[str, Dict[str, Any]] = {}
        for name, _, is_dim in UNIQUE_ESTIMATORS:
            if not is_dim:
                continue
            vl = est_left.get(name, float("nan"))
            vr = est_right.get(name, float("nan"))
            delta = float(vr - vl) if math.isfinite(vl) and math.isfinite(vr) else float("nan")
            contrast[name] = {
                "value_left": float(vl),
                "value_right": float(vr),
                "delta": delta,
                "is_dimension_estimator": True,
            }
        dim_deltas = [v["delta"] for v in contrast.values() if math.isfinite(v["delta"])]
        levels["L5_complexity"] = {
            "status": "ok",
            "exploratory": False,
            "mode": "single_pass_contrast",
            "t0_idx": t0_idx,
            "n_left": len(left_seg),
            "n_right": len(right_seg),
            "n_estimators_evaluated": len(contrast),
            "n_dimension_estimators_finite": len(dim_deltas),
            "contrast": contrast,
            "note": "Single-pass dimension roughness description around T0; does not vote in grade_evidence()",
        }
    else:
        levels["L5_complexity"] = {
            "status": "insufficient_data",
            "exploratory": False,
            "mode": "single_pass_contrast",
            "n_estimators_evaluated": 0,
            "note": "T0 not available or segments too short for roughness contrast",
        }

    if not getattr(cfg, "run_complexity_diagnostics", False):
        levels["L6_prediction"] = {
            "status": "disabled_in_core",
            "note": "prediction lives outside analyze() inside predict_universal() and stays locked unless criteria (including L7 exploratory run) pass",
        }
        levels["L7_interactions"] = {
            "status": "disabled_by_default",
            "exploratory": True,
            "segments": [],
            "per_flux": {},
            "summary": {
                "n_segments": 0,
                "n_fluxes": 0,
                "total_complementary_pairs": 0,
                "note": "Exploratory L7 sliding-window interactions disabled on default path; pass run_complexity_diagnostics=True",
            },
        }
        return levels

    # Preserve single-pass contrast computed above for merging into exploratory L5
    l5_contrast_data = levels.get("L5_complexity", {}).get("contrast")
    l5_n_left = levels.get("L5_complexity", {}).get("n_left")
    l5_n_right = levels.get("L5_complexity", {}).get("n_right")
    l5_n_dim_finite = levels.get("L5_complexity", {}).get("n_dimension_estimators_finite", 0)

    all_est: List[float] = []
    for est in est_by_flux.values():
        if isinstance(est, dict):
            all_est.extend(v for v in est.values() if isinstance(v, (int, float)) and math.isfinite(v))

    tlog = compute_tlog(
        est_by_flux.get(ds.target_name, {}) if isinstance(est_by_flux.get(ds.target_name), dict) else {},
        ds.n,
        series=ds.target,
        n_perm=25,
        seed=cfg.seed + 505,
    )
    alerts: List[str] = []
    if (
        tlog.get("calibrated")
        and math.isfinite(tlog.get("z_score", float("nan")))
        and abs(tlog["z_score"]) > 2.0
        and tlog.get("p_permutation", 1.0) <= 0.10
    ):
        alerts.append("t_log_deviates_from_permutation_null")

    levels["L5_complexity"] = {
        "status": "ok",
        "exploratory": True,
        "mode": "full_exploratory",
        "t_log": tlog,
        "n_estimators_evaluated": len(ESTIMATORS),
        "n_unique_estimators": len(UNIQUE_ESTIMATORS),
        "alerts": alerts,
        "confidence": float(stability_weight(all_est) if all_est else 0.0),
        "note": "Exploratory complexity diagnostic only (opt-in via run_complexity_diagnostics=True); never proof of a threshold",
    }
    if l5_contrast_data is not None:
        levels["L5_complexity"]["contrast"] = l5_contrast_data
        levels["L5_complexity"]["n_left"] = l5_n_left
        levels["L5_complexity"]["n_right"] = l5_n_right
        levels["L5_complexity"]["n_dimension_estimators_finite"] = l5_n_dim_finite

    # L6 Prediction disabled in core analysis
    levels["L6_prediction"] = {
        "status": "disabled_in_core",
        "note": "prediction lives outside analyze() inside predict_universal() and stays locked unless criteria pass",
    }

    # L7 Interactions (exploratory opt-in)
    try:
        l7_res = compute_interactions(ds, cascade, cfg)
        if isinstance(l7_res, dict):
            l7_res["exploratory"] = True
        levels["L7_interactions"] = l7_res
    except Exception as exc:
        levels["L7_interactions"] = {"status": "error", "exploratory": True, "error": str(exc)}

    return levels

## Module `report_html.py`

In [ ]:
"""Self-contained interactive HTML report generator for UNIVERSAL_LOG."""


import html
import json
import math
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np


def _esc(val: Any) -> str:
    return html.escape("" if val is None else str(val), quote=True)


def _is_finite(val: Any) -> bool:
    try:
        return val is not None and math.isfinite(float(val))
    except (TypeError, ValueError):
        return False


def _fmt(val: Any, fmt_spec: str = ".4g", fallback: str = "N/A") -> str:
    if not _is_finite(val):
        return fallback
    return format(float(val), fmt_spec)


def _downsample_xy(
    x: np.ndarray,
    y: np.ndarray,
    max_points: int = 800,
) -> Tuple[np.ndarray, np.ndarray]:
    """Deterministic min-max bucket downsampling preserving spikes and endpoints."""
    n = len(x)
    if n <= max_points or max_points < 4:
        return x, y
    n_buckets = max(2, max_points // 2)
    edges = np.linspace(0, n, n_buckets + 1, dtype=int)
    keep: List[int] = [0]
    for i in range(n_buckets):
        a, b = int(edges[i]), int(edges[i + 1])
        if b <= a:
            continue
        seg = y[a:b]
        fin = np.where(np.isfinite(seg))[0]
        if len(fin) == 0:
            continue
        i_min = a + int(fin[int(np.argmin(seg[fin]))])
        i_max = a + int(fin[int(np.argmax(seg[fin]))])
        for idx in sorted({i_min, i_max}):
            if idx != keep[-1]:
                keep.append(idx)
    if keep[-1] != n - 1:
        keep.append(n - 1)
    idx_arr = np.asarray(keep, dtype=int)
    return x[idx_arr], y[idx_arr]


def _series_svg(
    ctrl: np.ndarray,
    vals: np.ndarray,
    *,
    svg_id: str,
    title: str,
    control_name: str,
    y_label: str,
    t0_val: Optional[float],
    cascade_marks: List[Tuple[str, float]],
    mean_pre: Optional[float] = None,
    mean_post: Optional[float] = None,
    stroke_color: str = "#1d4ed8",
) -> str:
    mask = np.isfinite(ctrl) & np.isfinite(vals)
    if int(np.sum(mask)) < 2:
        return f"<div class='empty-box'>Insufficient finite points for {_esc(title)}</div>"

    cx = np.asarray(ctrl[mask], dtype=float)
    cy = np.asarray(vals[mask], dtype=float)
    order = np.argsort(cx, kind="mergesort")
    cx, cy = cx[order], cy[order]
    dx, dy = _downsample_xy(cx, cy, max_points=800)

    width, height = 920.0, 310.0
    pad_l, pad_r, pad_t, pad_b = 64.0, 24.0, 28.0, 40.0
    pw = width - pad_l - pad_r
    ph = height - pad_t - pad_b

    xmin, xmax = float(np.min(cx)), float(np.max(cx))
    if xmax <= xmin:
        xmax = xmin + 1.0
    ymin, ymax = float(np.min(cy)), float(np.max(cy))
    if ymax <= ymin:
        ymin -= 1.0
        ymax += 1.0
    ymargin = 0.08 * (ymax - ymin)
    ymin -= ymargin
    ymax += ymargin

    def sx(v: float) -> float:
        return pad_l + (v - xmin) / (xmax - xmin) * pw

    def sy(v: float) -> float:
        return pad_t + (ymax - v) / (ymax - ymin) * ph

    pts = " ".join(f"{sx(float(x)):.2f},{sy(float(y)):.2f}" for x, y in zip(dx, dy))
    area_pts = (
        f"{sx(float(dx[0])):.2f},{(pad_t + ph):.2f} "
        + pts
        + f" {sx(float(dx[-1])):.2f},{(pad_t + ph):.2f}"
    )

    grid_lines: List[str] = []
    for k in range(5):
        frac = k / 4.0
        yg = pad_t + frac * ph
        yval = ymax - frac * (ymax - ymin)
        grid_lines.append(
            f"<line x1='{pad_l:.1f}' y1='{yg:.1f}' x2='{(width - pad_r):.1f}' y2='{yg:.1f}' "
            f"stroke='#e2e8f0' stroke-width='1'/>"
            f"<text x='{(pad_l - 8):.1f}' y='{(yg + 4):.1f}' text-anchor='end' "
            f"class='svg-tick'>{_fmt(yval, '.3g')}</text>"
        )
    for k in range(6):
        frac = k / 5.0
        xg = pad_l + frac * pw
        xval = xmin + frac * (xmax - xmin)
        grid_lines.append(
            f"<line x1='{xg:.1f}' y1='{pad_t:.1f}' x2='{xg:.1f}' y2='{(height - pad_b):.1f}' "
            f"stroke='#f1f5f9' stroke-width='1'/>"
            f"<text x='{xg:.1f}' y='{(height - pad_b + 16):.1f}' text-anchor='middle' "
            f"class='svg-tick'>{_fmt(xval, '.4g')}</text>"
        )

    means_svg = ""
    if _is_finite(t0_val) and xmin <= float(t0_val) <= xmax:
        xt0 = sx(float(t0_val))
        if _is_finite(mean_pre):
            ypre = max(pad_t, min(pad_t + ph, sy(float(mean_pre))))
            means_svg += (
                f"<line x1='{pad_l:.1f}' y1='{ypre:.1f}' x2='{xt0:.1f}' y2='{ypre:.1f}' "
                f"stroke='#0284c7' stroke-width='1.8' stroke-dasharray='6,4'/>"
            )
        if _is_finite(mean_post):
            ypost = max(pad_t, min(pad_t + ph, sy(float(mean_post))))
            means_svg += (
                f"<line x1='{xt0:.1f}' y1='{ypost:.1f}' x2='{(width - pad_r):.1f}' y2='{ypost:.1f}' "
                f"stroke='#ea580c' stroke-width='1.8' stroke-dasharray='6,4'/>"
            )

    cascade_svg = ""
    label_offsets = {"T-2": 12, "T-1": 24, "T+1": 24, "T+2": 12}
    for label, cval in cascade_marks:
        if _is_finite(cval) and xmin <= float(cval) <= xmax:
            xc = sx(float(cval))
            dy_off = label_offsets.get(label, 14)
            cascade_svg += (
                f"<line x1='{xc:.1f}' y1='{pad_t:.1f}' x2='{xc:.1f}' y2='{(height - pad_b):.1f}' "
                f"stroke='#94a3b8' stroke-width='1.2' stroke-dasharray='3,3'/>"
                f"<text x='{(xc + 3):.1f}' y='{(pad_t + dy_off):.1f}' class='svg-cas'>{_esc(label)}</text>"
            )

    t0_svg = ""
    if _is_finite(t0_val) and xmin <= float(t0_val) <= xmax:
        xt0 = sx(float(t0_val))
        t0_svg = (
            f"<line x1='{xt0:.1f}' y1='{(pad_t - 4):.1f}' x2='{xt0:.1f}' y2='{(height - pad_b):.1f}' "
            f"stroke='#dc2626' stroke-width='2.3'/>"
            f"<rect x='{min(width - pad_r - 92, max(pad_l, xt0 - 46)):.1f}' y='4' width='92' height='19' rx='4' fill='#dc2626'/>"
            f"<text x='{min(width - pad_r - 46, max(pad_l + 46, xt0)):.1f}' y='17' text-anchor='middle' "
            f"fill='#ffffff' font-size='11' font-weight='700'>T0={_fmt(t0_val, '.4g')}</text>"
        )

    payload_pts = json.dumps(
        [[round(float(x), 6), round(float(y), 6)] for x, y in zip(dx, dy)],
        separators=(",", ":"),
    )

    return f"""
<div class="chart-wrap">
  <div class="chart-head">
    <span class="chart-title">{_esc(title)}</span>
    <span class="chart-tooltip" id="{_esc(svg_id)}_tip">Hover over chart to inspect values</span>
  </div>
  <svg id="{_esc(svg_id)}" viewBox="0 0 {int(width)} {int(height)}" class="series-svg"
       data-xmin="{xmin}" data-xmax="{xmax}" data-padl="{pad_l}" data-pw="{pw}"
       data-pts='{payload_pts}'>
    <defs>
      <linearGradient id="{_esc(svg_id)}_grad" x1="0" y1="0" x2="0" y2="1">
        <stop offset="0%" stop-color="{stroke_color}" stop-opacity="0.18"/>
        <stop offset="100%" stop-color="{stroke_color}" stop-opacity="0.01"/>
      </linearGradient>
    </defs>
    <rect x="{pad_l:.1f}" y="{pad_t:.1f}" width="{pw:.1f}" height="{ph:.1f}" fill="#ffffff" stroke="#cbd5e1"/>
    {''.join(grid_lines)}
    <polygon points="{area_pts}" fill="url(#{_esc(svg_id)}_grad)"/>
    <polyline fill="none" stroke="{stroke_color}" stroke-width="1.8" points="{pts}"/>
    {means_svg}
    {cascade_svg}
    {t0_svg}
    <line id="{_esc(svg_id)}_cross" x1="0" y1="{pad_t:.1f}" x2="0" y2="{(height - pad_b):.1f}"
          stroke="#0f172a" stroke-width="1" stroke-dasharray="2,2" visibility="hidden"/>
    <text x="{(pad_l + pw / 2):.1f}" y="{(height - 5):.1f}" text-anchor="middle" class="svg-axis">{_esc(control_name)}</text>
    <text x="16" y="{(pad_t + ph / 2):.1f}" text-anchor="middle" transform="rotate(-90 16 {(pad_t + ph / 2):.1f})" class="svg-axis">{_esc(y_label)}</text>
  </svg>
</div>
"""


def _null_bars_svg(observed_d: float, falsif: Dict[str, Any]) -> str:
    wn = falsif.get("white_noise_null", {}) if isinstance(falsif, dict) else {}
    sh = falsif.get("shuffled_target_null", {}) if isinstance(falsif, dict) else {}
    pc = falsif.get("permuted_control_same_cut", {}) if isinstance(falsif, dict) else {}
    ar1 = falsif.get("ar1_surrogate_null", {}) if isinstance(falsif, dict) else {}

    items = [
        ("Observed |d|", abs(observed_d) if _is_finite(observed_d) else 0.0, "#1d4ed8"),
        ("AR(1) Null p95 |d|", float(ar1.get("ar1_null_p95_abs_d")) if _is_finite(ar1.get("ar1_null_p95_abs_d")) else 0.0, "#64748b"),
        ("White-Noise Null |d|", abs(float(wn.get("cohen_d"))) if _is_finite(wn.get("cohen_d")) else 0.0, "#64748b"),
        ("Shuffled Null |d|", abs(float(sh.get("cohen_d"))) if _is_finite(sh.get("cohen_d")) else 0.0, "#64748b"),
        ("Permuted Cut Med |d|", abs(float(pc.get("d_permuted_median"))) if _is_finite(pc.get("d_permuted_median")) else 0.0, "#94a3b8"),
    ]
    vmax = max([v for _, v, _ in items] + [0.5]) * 1.12
    w, h = 440.0, 195.0
    pad_l, pad_r, pad_t = 152.0, 56.0, 14.0
    bw = w - pad_l - pad_r
    rows_svg: List[str] = []
    for idx, (label, val, color) in enumerate(items):
        y = pad_t + idx * 34.0
        bar_w = max(2.0, (val / vmax) * bw) if vmax > 0 else 2.0
        rows_svg.append(
            f"<text x='{(pad_l - 8):.1f}' y='{(y + 16):.1f}' text-anchor='end' class='svg-tick'>{_esc(label)}</text>"
            f"<rect x='{pad_l:.1f}' y='{y + 3:.1f}' width='{bw:.1f}' height='18' rx='4' fill='#f1f5f9'/>"
            f"<rect x='{pad_l:.1f}' y='{y + 3:.1f}' width='{bar_w:.1f}' height='18' rx='4' fill='{color}'/>"
            f"<text x='{(pad_l + bar_w + 6):.1f}' y='{(y + 16):.1f}' class='svg-tick-bold'>{val:.3f}</text>"
        )
    if _is_finite(observed_d) and abs(observed_d) > 0:
        half_x = pad_l + (0.5 * abs(observed_d) / vmax) * bw
        rows_svg.append(
            f"<line x1='{half_x:.1f}' y1='{pad_t:.1f}' x2='{half_x:.1f}' y2='{(h - 14):.1f}' "
            f"stroke='#dc2626' stroke-width='1.3' stroke-dasharray='4,3'/>"
            f"<text x='{half_x:.1f}' y='{(h - 2):.1f}' text-anchor='middle' fill='#dc2626' font-size='10'>0.5 * |d_obs|</text>"
        )
    return f"<svg viewBox='0 0 {int(w)} {int(h)}' class='bar-svg'>{''.join(rows_svg)}</svg>"


def _multi_window_svg(falsif: Dict[str, Any], ctrl: np.ndarray) -> str:
    mw = falsif.get("multi_window", {}) if isinstance(falsif, dict) else {}
    vals = [float(v) for v in mw.get("threshold_values", []) if _is_finite(v)]
    if len(vals) == 0 or len(ctrl) == 0:
        return "<div class='empty-box'>No multi-window thresholds recorded.</div>"
    xmin, xmax = float(np.nanmin(ctrl)), float(np.nanmax(ctrl))
    if xmax <= xmin:
        xmax = xmin + 1.0
    w, h = 440.0, 120.0
    pad_l, pad_r = 28.0, 28.0
    pw = w - pad_l - pad_r
    dots: List[str] = []
    for i, v in enumerate(vals):
        x = pad_l + max(0.0, min(1.0, (v - xmin) / (xmax - xmin))) * pw
        y = 34.0 + i * 18.0
        dots.append(
            f"<circle cx='{x:.1f}' cy='{y:.1f}' r='5.5' fill='#0284c7'/>"
            f"<text x='{(x + 9):.1f}' y='{(y + 4):.1f}' class='svg-tick'>w#{i+1}: {_fmt(v, '.4g')}</text>"
        )
    return (
        f"<svg viewBox='0 0 {int(w)} {int(h)}' class='bar-svg'>"
        f"<line x1='{pad_l:.1f}' y1='96' x2='{(w - pad_r):.1f}' y2='96' stroke='#94a3b8' stroke-width='2'/>"
        f"<text x='{pad_l:.1f}' y='112' class='svg-tick'>{_fmt(xmin, '.4g')}</text>"
        f"<text x='{(w - pad_r):.1f}' y='112' text-anchor='end' class='svg-tick'>{_fmt(xmax, '.4g')}</text>"
        f"{''.join(dots)}</svg>"
    )


def render_html_report(
    result: Dict[str, Any],
    control: Sequence[float],
    target: Sequence[float],
    active_control: Optional[Sequence[float]] = None,
    active_target: Optional[Sequence[float]] = None,
) -> str:
    """Render a completely standalone interactive HTML5/SVG report from analyze() output."""
    uc = result.get("universal_contract", {})
    scan = result.get("scan", {})
    level_scan = result.get("level_shift_scan", {})
    structural = result.get("structural_scans", {})
    mv = result.get("multivariate_scan", {})
    mt = result.get("multi_thresholds", {})
    cascade = result.get("cascade", {})
    levels = result.get("levels", {})
    falsif = result.get("falsification", {})
    wf = result.get("walk_forward", {})
    grade_info = result.get("grade", {})
    tlog = result.get("t_log_target", {})
    manifest = result.get("manifest", {})
    invar = result.get("invariance_rename_test", {})

    ctrl_arr = np.asarray(control, dtype=float)
    tgt_arr = np.asarray(target, dtype=float)
    control_name = str(uc.get("control", "control"))
    target_name = str(uc.get("target", "target"))
    grade = str(grade_info.get("grade", "D"))
    verdict = str(grade_info.get("verdict", ""))
    detector = str(scan.get("detector", "level_shift"))
    t0_val = scan.get("control_value") if scan.get("status") == "ok" else None

    cascade_marks: List[Tuple[str, float]] = []
    if isinstance(cascade, dict):
        for key, label in [
            ("T_minus2", "T-2"),
            ("T_minus1", "T-1"),
            ("T_plus1", "T+1"),
            ("T_plus2", "T+2"),
        ]:
            c_item = cascade.get(key, {})
            if isinstance(c_item, dict) and _is_finite(c_item.get("control_value")):
                cascade_marks.append((label, float(c_item["control_value"])))

    raw_svg = _series_svg(
        ctrl_arr,
        tgt_arr,
        svg_id="raw_chart",
        title=f"Raw Series: {target_name} vs {control_name}",
        control_name=control_name,
        y_label=target_name,
        t0_val=float(t0_val) if _is_finite(t0_val) else None,
        cascade_marks=cascade_marks,
        mean_pre=scan.get("mean_pre") if detector == "level_shift" else None,
        mean_post=scan.get("mean_post") if detector == "level_shift" else None,
        stroke_color="#1d4ed8",
    )

    active_svg_block = ""
    if active_target is not None and detector in ("deseasonalized_shift", "volatility_shift"):
        act_c = np.asarray(active_control if active_control is not None else ctrl_arr, dtype=float)
        act_y = np.asarray(active_target, dtype=float)
        if len(act_c) == len(act_y) and len(act_y) >= 2:
            active_svg_block = _series_svg(
                act_c,
                act_y,
                svg_id="active_chart",
                title=f"Active Channel ({detector}): Transformed Signal + Threshold T0",
                control_name=control_name,
                y_label=f"{target_name} ({detector})",
                t0_val=float(t0_val) if _is_finite(t0_val) else None,
                cascade_marks=cascade_marks,
                mean_pre=scan.get("mean_pre"),
                mean_post=scan.get("mean_post"),
                stroke_color="#0f766e",
            )

    grade_palette = {
        "A": ("#15803d", "#dcfce7", "#86efac"),
        "B": ("#0369a1", "#e0f2fe", "#7dd3fc"),
        "C": ("#b45309", "#fef3c7", "#fcd34d"),
        "D": ("#b91c1c", "#fee2e2", "#fca5a5"),
    }.get(grade, ("#475569", "#f1f5f9", "#cbd5e1"))

    reasons = grade_info.get("reasons", []) or []
    reasons_html = (
        "<ul>" + "".join(f"<li>{_esc(r)}</li>" for r in reasons) + "</ul>"
        if reasons
        else "<p class='ok-msg'>All quantitative and falsification criteria satisfied under this protocol.</p>"
    )

    checks = grade_info.get("checks", {}) or {}
    checks_html = "".join(
        f"<span class='chk {'chk-ok' if ok else 'chk-fail'}'>"
        f"{'&#10003;' if ok else '&#10007;'} {_esc(k)}</span>"
        for k, ok in checks.items()
    )

    null_svg = _null_bars_svg(float(scan.get("cohen_d", 0.0)) if _is_finite(scan.get("cohen_d")) else 0.0, falsif)
    mw_svg = _multi_window_svg(falsif, ctrl_arr)

    # Structural channels table rows
    chan_rows: List[str] = []
    channels = [
        ("level_shift", level_scan, "cohen_d"),
        ("detrended_jump", structural.get("detrended_jump", {}), "effect_d"),
        ("slope_break", structural.get("slope_break", {}), "t_stat"),
        ("volatility_shift", structural.get("volatility_shift", {}), "cohen_d"),
        ("deseasonalized_shift", structural.get("deseasonalized_shift", {}), "cohen_d"),
    ]
    for cname, cdata, stat_key in channels:
        if not isinstance(cdata, dict):
            continue
        is_selected = cname == detector and scan.get("status") == "ok"
        badge = "<span class='pill-primary'>PRIMARY</span>" if is_selected else ""
        extra_parts: List[str] = []
        if cdata.get("null_model"):
            extra_parts.append(str(cdata["null_model"]))
        if cname == "deseasonalized_shift" and _is_finite(cdata.get("dominant_power_fraction")):
            extra_parts.append(f"dom_fft={_fmt(cdata.get('dominant_power_fraction'), '.3f')}")
        elif "note" in cdata and cdata.get("status") != "ok":
            extra_parts.append(str(cdata.get("note", "")))
        p_raw_c = cdata.get("p_channel_raw", cdata.get("p_scan_null"))
        p_scr_c = cdata.get("p_screen")
        p_ref_c = cdata.get("p_refined")
        p_adj_c = cdata.get("p_channel_adjusted")
        chan_rows.append(
            f"<tr class='{'row-active' if is_selected else ''}'>"
            f"<td><code>{_esc(cname)}</code> {badge}</td>"
            f"<td>{_esc(cdata.get('status', 'N/A'))}</td>"
            f"<td>{_fmt(cdata.get('control_value'), '.4g')}</td>"
            f"<td><code>{_esc(stat_key)}</code>={_fmt(cdata.get(stat_key), '.3f')}</td>"
            f"<td>{_fmt(p_scr_c, '.4f')}</td>"
            f"<td>{_fmt(p_ref_c, '.4f')}</td>"
            f"<td>{_fmt(p_adj_c if p_adj_c is not None else p_raw_c, '.4f')}</td>"
            f"<td class='muted'>{_esc(' · '.join(extra_parts))}</td>"
            f"</tr>"
        )

    # Cascade table rows
    cas_rows: List[str] = []
    for key in ["T_minus2", "T_minus1", "T_plus1", "T_plus2"]:
        item = cascade.get(key, {}) if isinstance(cascade, dict) else {}
        if isinstance(item, dict):
            cas_rows.append(
                f"<tr><td><code>{_esc(key)}</code></td>"
                f"<td>{_fmt(item.get('control_value'), '.4g')}</td>"
                f"<td>{_esc(item.get('idx', 'N/A'))}</td></tr>"
            )

    # Levels L1..L7 summary cards
    l1 = levels.get("L1_direction", {})
    l2 = levels.get("L2_magnitude", {})
    l3 = levels.get("L3_regime", {})
    l4 = levels.get("L4_signal", {})
    l5 = levels.get("L5_complexity", {})
    l6 = levels.get("L6_prediction", {})
    l7 = levels.get("L7_interactions", {})
    l7_sum = l7.get("summary", {}) if isinstance(l7, dict) else {}

    # Top estimators from target (exploratory opt-in) or L5 single-pass contrast (default)
    raw_est = result.get("estimators_by_flux", {}) or {}
    est_target = raw_est.get(target_name, {}) if isinstance(raw_est.get(target_name), dict) else {}
    est_items = [(k, v) for k, v in est_target.items() if _is_finite(v)]

    # Build L5 roughness contrast table (shown on default path)
    l5_contrast = l5.get("contrast", {}) if l5.get("status") == "ok" else {}
    l5_dim_items = [(k, v) for k, v in l5_contrast.items() if v.get("is_dimension_estimator") and _is_finite(v.get("delta"))]
    l5_dim_items.sort(key=lambda x: abs(x[1]["delta"]), reverse=True)
    if l5_dim_items:
        l5_rows_html = (
            "<table class='tbl'><tr><th>Estimator</th><th>Left of T₀</th><th>Right of T₀</th><th>Δ</th></tr>"
            + "".join(
                f"<tr><td>{_esc(k)}</td><td>{_fmt(v['value_left'], '.4f')}</td>"
                f"<td>{_fmt(v['value_right'], '.4f')}</td>"
                f"<td style='color:{('var(--up)' if v['delta']>0 else 'var(--dn)')}'>{_fmt(v['delta'], '+.4f')}</td></tr>"
                for k, v in l5_dim_items
            )
            + "</table>"
            + f"<p class='muted'>Single-pass roughness description ({l5.get('n_left')}/{l5.get('n_right')} pts). Does not vote in grade.</p>"
        )
    else:
        l5_rows_html = "<div class='empty-box'>L5 roughness contrast not available (segments too short).</div>"

    if est_items:
        est_rows = "<div class='est-grid'>" + "".join(
            f"<div class='est-item'><span class='est-k'>{_esc(k)}</span>"
            f"<span class='est-v'>{_fmt(v, '.4f')}</span></div>"
            for k, v in est_items
        ) + "</div>"
    else:
        est_rows = (
            "<div class='empty-box'>Full 29 complexity estimators disabled on default path "
            "(enable with <code>run_complexity_diagnostics=True</code> or <code>--complexity-diagnostics</code>).</div>"
        )

    mw_info = falsif.get("multi_window", {}) if isinstance(falsif, dict) else {}
    ar1_info = falsif.get("ar1_surrogate_null", {}) if isinstance(falsif, dict) else {}
    trend_info = falsif.get("trend_spearman", {}) if isinstance(falsif, dict) else {}

    multi_thr_list = mt.get("thresholds", []) if isinstance(mt, dict) else []
    mt_html = (
        ", ".join(f"{_fmt(t.get('control_value'), '.4g')} (d={_fmt(t.get('cohen_d'), '.2f')})" for t in multi_thr_list)
        if multi_thr_list
        else "Single / none detected"
    )

    return f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8"/>
<meta name="viewport" content="width=device-width, initial-scale=1"/>
<title>UNIVERSAL_LOG Report — {_esc(target_name)} vs {_esc(control_name)} (Grade {_esc(grade)})</title>
<style>
  :root {{
    --bg: #f8fafc; --card: #ffffff; --border: #e2e8f0;
    --text: #0f172a; --muted: #64748b; --accent: #1d4ed8;
    --grade-fg: {grade_palette[0]}; --grade-bg: {grade_palette[1]}; --grade-bd: {grade_palette[2]};
  }}
  * {{ box-sizing: border-box; }}
  body {{
    margin: 0; padding: 24px; background: var(--bg); color: var(--text);
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Inter, sans-serif;
    line-height: 1.5;
  }}
  .container {{ max-width: 1180px; margin: 0 auto; display: flex; flex-direction: column; gap: 18px; }}
  .hero {{
    background: var(--card); border: 1px solid var(--border); border-radius: 14px;
    padding: 20px 24px; display: flex; flex-wrap: wrap; justify-content: space-between;
    align-items: center; gap: 16px; box-shadow: 0 1px 3px rgba(15,23,42,0.05);
  }}
  .hero h1 {{ margin: 0 0 4px 0; font-size: 1.35rem; letter-spacing: -0.01em; }}
  .subtitle {{ color: var(--muted); font-size: 0.9rem; margin: 0; }}
  .grade-box {{
    display: flex; align-items: center; gap: 14px; padding: 10px 16px; border-radius: 12px;
    background: var(--grade-bg); border: 1.5px solid var(--grade-bd); color: var(--grade-fg);
  }}
  .grade-letter {{ font-size: 2.1rem; font-weight: 800; line-height: 1; }}
  .grade-meta {{ font-size: 0.85rem; max-width: 340px; }}
  .kpi-grid {{
    display: grid; grid-template-columns: repeat(auto-fit, minmax(165px, 1fr)); gap: 12px;
  }}
  .kpi {{
    background: var(--card); border: 1px solid var(--border); border-radius: 10px; padding: 12px 14px;
  }}
  .kpi-label {{ font-size: 0.74rem; text-transform: uppercase; letter-spacing: 0.04em; color: var(--muted); }}
  .kpi-val {{ font-size: 1.18rem; font-weight: 700; margin-top: 2px; font-variant-numeric: tabular-nums; }}
  .kpi-sub {{ font-size: 0.78rem; color: var(--muted); }}
  .card {{
    background: var(--card); border: 1px solid var(--border); border-radius: 12px; padding: 18px 20px;
    box-shadow: 0 1px 2px rgba(15,23,42,0.03);
  }}
  .card h2 {{ margin: 0 0 12px 0; font-size: 1.05rem; border-bottom: 1px solid var(--border); padding-bottom: 8px; }}
  .two-col {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(430px, 1fr)); gap: 16px; }}
  .chart-wrap {{ margin-bottom: 14px; }}
  .chart-head {{ display: flex; justify-content: space-between; align-items: center; margin-bottom: 6px; font-size: 0.88rem; }}
  .chart-title {{ font-weight: 600; }}
  .chart-tooltip {{ font-family: ui-monospace, SFMono-Regular, monospace; font-size: 0.8rem; color: var(--muted); }}
  .series-svg, .bar-svg {{ width: 100%; height: auto; display: block; }}
  .svg-tick {{ font-size: 10.5px; fill: #475569; font-family: ui-monospace, monospace; }}
  .svg-tick-bold {{ font-size: 10.5px; fill: #0f172a; font-weight: 700; font-family: ui-monospace, monospace; }}
  .svg-axis {{ font-size: 11px; fill: #334155; font-weight: 600; }}
  .svg-cas {{ font-size: 10px; fill: #475569; font-weight: 700; }}
  .checks {{ display: flex; flex-wrap: wrap; gap: 6px; margin: 10px 0; }}
  .chk {{ font-size: 0.78rem; padding: 3px 9px; border-radius: 999px; font-weight: 600; }}
  .chk-ok {{ background: #dcfce7; color: #166534; }}
  .chk-fail {{ background: #fee2e2; color: #991b1b; }}
  .ok-msg {{ color: #166534; font-weight: 600; margin: 4px 0; }}
  table {{ width: 100%; border-collapse: collapse; font-size: 0.86rem; }}
  th, td {{ text-align: left; padding: 7px 10px; border-bottom: 1px solid var(--border); }}
  th {{ color: var(--muted); font-weight: 600; font-size: 0.78rem; text-transform: uppercase; }}
  .row-active {{ background: #eff6ff; }}
  .pill-primary {{ background: #1d4ed8; color: #fff; font-size: 0.68rem; padding: 2px 6px; border-radius: 4px; font-weight: 700; }}
  .muted {{ color: var(--muted); font-size: 0.82rem; }}
  .levels-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 10px; }}
  .lvl {{ border: 1px solid var(--border); border-radius: 8px; padding: 10px 12px; background: #fcfdff; }}
  .lvl strong {{ display: block; font-size: 0.84rem; margin-bottom: 4px; color: #1e293b; }}
  .lvl span {{ font-size: 0.82rem; color: #475569; }}
  .est-grid {{ display: grid; grid-template-columns: repeat(auto-fill, minmax(190px, 1fr)); gap: 6px; max-height: 260px; overflow-y: auto; }}
  .est-item {{ display: flex; justify-content: space-between; font-size: 0.8rem; padding: 4px 8px; background: #f8fafc; border-radius: 6px; }}
  .est-k {{ color: #334155; }}
  .est-v {{ font-family: ui-monospace, monospace; font-weight: 600; }}
  .empty-box {{ padding: 16px; background: #f1f5f9; border-radius: 8px; color: var(--muted); font-size: 0.86rem; }}
  code {{ font-family: ui-monospace, SFMono-Regular, monospace; font-size: 0.84em; background: #f1f5f9; padding: 1px 4px; border-radius: 4px; }}
  footer {{ text-align: center; color: var(--muted); font-size: 0.78rem; padding: 8px 0 18px; }}
</style>
</head>
<body>
<div class="container">
  <section class="hero">
    <div>
      <h1>UNIVERSAL_LOG <code>v{_esc(result.get('version', '7.4.0'))}</code></h1>
      <p class="subtitle">
        Control: <strong>{_esc(control_name)}</strong> &middot;
        Target: <strong>{_esc(target_name)}</strong> &middot;
        n = <strong>{_esc(uc.get('n'))}</strong> &middot;
        Fluxes: <strong>{_esc(len(uc.get('fluxes', [])))}</strong> &middot;
        Mode: <strong>{_esc(manifest.get('mode'))}</strong>
      </p>
    </div>
    <div class="grade-box">
      <div class="grade-letter">{_esc(grade)}</div>
      <div class="grade-meta">
        <strong>Evidence Grade {_esc(grade)}</strong><br/>
        {_esc(verdict)}<br/>
        Checks passed: {_esc(grade_info.get('n_checks_passed'))}/5
      </div>
    </div>
  </section>

  <section class="kpi-grid">
    <div class="kpi"><div class="kpi-label">Primary Detector</div><div class="kpi-val">{_esc(detector)}</div><div class="kpi-sub">status: {_esc(scan.get('status'))} &middot; {_esc(scan.get('stat_kind', 'mean'))}</div></div>
    <div class="kpi"><div class="kpi-label">Threshold T0</div><div class="kpi-val">{_fmt(t0_val, '.4g')}</div><div class="kpi-sub">idx = {_esc(scan.get('idx', 'N/A'))}</div></div>
    <div class="kpi"><div class="kpi-label">Effect Size</div><div class="kpi-val">{_fmt(scan.get('cohen_d'), '.3f')}</div><div class="kpi-sub">CI95% [{_fmt(scan.get('cohen_d_ci_low'), '.3f')}, {_fmt(scan.get('cohen_d_ci_high'), '.3f')}]</div></div>
    <div class="kpi"><div class="kpi-label">p Holm (m={_esc(scan.get('n_channels_family', 1))})</div><div class="kpi-val">{_fmt(scan.get('p_channel_adjusted', scan.get('p_scan_null')), '.4f')}</div><div class="kpi-sub">p_screen={_fmt(scan.get('p_screen'), '.4f')} &middot; p_refined={_fmt(scan.get('p_refined'), '.4f')} ({_esc(scan.get('null_model', 'N/A'))})</div></div>
    <div class="kpi"><div class="kpi-label">Walk-Forward F1</div><div class="kpi-val">{_fmt(wf.get('mean_f1'), '.3f')}</div><div class="kpi-sub">conservative = {_fmt(wf.get('mean_f1_conservative'), '.3f')} (splits={_esc(wf.get('n_splits_used', 0))})</div></div>
    <div class="kpi"><div class="kpi-label">Window Stability</div><div class="kpi-val">{_fmt(mw_info.get('stability'), '.3f')}</div><div class="kpi-sub">Spearman rho = {_fmt(trend_info.get('rho_control_target'), '.3f')}</div></div>
  </section>

  <section class="card">
    <h2>1. Series, Central Threshold T0 &amp; 5-Point Cascade</h2>
    {raw_svg}
    {active_svg_block}
  </section>

  <section class="card">
    <h2>2. Falsification &amp; Grading Diagnostics</h2>
    {reasons_html}
    <div class="checks">{checks_html}</div>
    <div class="two-col" style="margin-top:12px;">
      <div>
        <h3 style="font-size:0.9rem;margin:0 0 6px;">Null Models vs Observed |d|</h3>
        {null_svg}
        <p class="muted">AR(1) within-regime phi={_fmt(ar1_info.get('phi_within'), '.3f')}, simulated phi={_fmt(ar1_info.get('phi_simulated'), '.3f')}, p_ar1={_fmt(ar1_info.get('p_ar1_scan_corrected'), '.4f')}</p>
      </div>
      <div>
        <h3 style="font-size:0.9rem;margin:0 0 6px;">Multi-Window Threshold Consistency</h3>
        {mw_svg}
        <p class="muted">Windows evaluated: {_esc(mw_info.get('n_windows', 0))} &middot; std={_fmt(mw_info.get('std_thresholds'), '.4g')}</p>
      </div>
    </div>
  </section>

  <section class="two-col">
    <div class="card">
      <h2>3. Structural Detection Channels (m={_esc(scan.get('n_channels_family', '?'))})</h2>
      <table>
        <thead><tr><th>Channel</th><th>Status</th><th>T0 Value</th><th>Statistic</th><th>p screen</th><th>p refined</th><th>p Holm</th><th>Null Model / Notes</th></tr></thead>
        <tbody>{''.join(chan_rows)}</tbody>
      </table>
      <p class="muted" style="margin-top:10px;">
        <strong>Multivariate scan:</strong> status={_esc(mv.get('status', 'N/A'))}, T0={_fmt(mv.get('control_value'), '.4g')},
        composite_d={_fmt(mv.get('composite_effect'), '.3f')}, p={_fmt(mv.get('p_permutation_scan_corrected'), '.4f')}<br/>
        <strong>Multi-threshold splits:</strong> {_esc(mt_html)}
      </p>
    </div>
    <div class="card">
      <h2>4. 5-Point Threshold Cascade</h2>
      <table>
        <thead><tr><th>Point</th><th>Control Value</th><th>Index</th></tr></thead>
        <tbody>
          <tr class="row-active"><td><code>T0_central</code></td><td>{_fmt(t0_val, '.4g')}</td><td>{_esc(scan.get('idx', 'N/A'))}</td></tr>
          {''.join(cas_rows)}
        </tbody>
      </table>
      <p class="muted" style="margin-top:10px;">
        <strong>{'Slope per step' if scan.get('stat_kind') == 'slope_per_step' else _esc(scan.get('stat_kind', 'mean'))}:</strong>
        pre = {_fmt(scan.get('pre_stat', scan.get('mean_pre')), '.4g')} &middot;
        post = {_fmt(scan.get('post_stat', scan.get('mean_post')), '.4g')}
      </p>
    </div>
  </section>

  <section class="card">
    <h2>5. Universal Levels L1–L4 &amp; L5 Roughness Contrast</h2>
    <div class="levels-grid">
      <div class="lvl"><strong>L1 Direction</strong><span>{_esc(l1.get('direction', l1.get('status')))} (d={_fmt(l1.get('cohen_d'), '.3f')})</span></div>
      <div class="lvl"><strong>L2 Magnitude</strong><span>|d|={_fmt(l2.get('abs_cohen_d'), '.3f')} &middot; rel={_fmt(l2.get('relative_change'), '.3f')}</span></div>
      <div class="lvl"><strong>L3 Regime</strong><span>{_esc(l3.get('state', l3.get('status')))} (post={_fmt(l3.get('fraction_post_threshold'), '.2f')})</span></div>
      <div class="lvl"><strong>L4 Signal</strong><span>{_esc(l4.get('level', l4.get('status')))} (p_adj={_fmt(l4.get('p_channel_adjusted', l4.get('p_scan_null')), '.4f')})</span></div>
      <div class="lvl"><strong>L5 Roughness</strong><span>{f"{l5.get('n_estimators_evaluated', 0)} estimators, {l5.get('n_dimension_estimators_finite', 0)} finite dimension contrasts" if l5.get('status') == 'ok' else _esc(l5.get('status', 'N/A'))}</span></div>
      <div class="lvl"><strong>L6 Prediction Gate</strong><span>Locked outside analyze() — use predict_universal() (&ge;5 Grade A/B replications + L7 exploratory enabled)</span></div>
      <div class="lvl"><strong>L7 Interactions (Exploratory)</strong><span>{_esc(l7.get('status', 'N/A')) if l7.get('status') != 'ok' else f"pairs={_esc(l7_sum.get('total_complementary_pairs', 'N/A'))}, segs={_esc(l7_sum.get('n_segments', 'N/A'))}"}</span></div>
    </div>
    {l5_rows_html}
  </section>

  <section class="card">
    <h2>6. Target Flux Complexity Estimators (Exploratory: {len(est_items)})</h2>
    {est_rows}
  </section>

  <section class="card">
    <h2>7. Reproducibility Manifest &amp; Invariance</h2>
    <p class="muted" style="margin:0;">
      <strong>Source:</strong> <code>{_esc(manifest.get('data_source'))}</code> &middot;
      <strong>SHA-256:</strong> <code>{_esc(manifest.get('data_hash_sha256'))}</code><br/>
      <strong>Seed:</strong> {_esc(manifest.get('seed'))} &middot;
      <strong>Elapsed:</strong> {_fmt(manifest.get('elapsed_seconds'), '.2f')}s &middot;
      <strong>Rename Invariance (P3):</strong> {_esc(invar.get('passed'))} &middot;
      <strong>Python:</strong> {_esc(str(manifest.get('python', '')).split()[0])} &middot;
      <strong>NumPy:</strong> {_esc(manifest.get('numpy'))} &middot;
      <strong>Pandas:</strong> {_esc(manifest.get('pandas'))}
    </p>
  </section>

  <footer>
    Generated offline by UNIVERSAL_LOG v{_esc(result.get('version', '7.4.0'))}. Detection and falsification only; independent replication required prior to deployment.
  </footer>
</div>
<script>
(function() {{
  document.querySelectorAll("svg.series-svg").forEach(function(svg) {{
    var pts = JSON.parse(svg.getAttribute("data-pts") || "[]");
    if (!pts.length) return;
    var xmin = parseFloat(svg.getAttribute("data-xmin"));
    var xmax = parseFloat(svg.getAttribute("data-xmax"));
    var padl = parseFloat(svg.getAttribute("data-padl"));
    var pw = parseFloat(svg.getAttribute("data-pw"));
    var tip = document.getElementById(svg.id + "_tip");
    var cross = document.getElementById(svg.id + "_cross");
    svg.addEventListener("mousemove", function(e) {{
      var rect = svg.getBoundingClientRect();
      var svgX = ((e.clientX - rect.left) / rect.width) * 920.0;
      var frac = Math.max(0, Math.min(1, (svgX - padl) / pw));
      var xTarget = xmin + frac * (xmax - xmin);
      var best = pts[0], bestDist = Math.abs(pts[0][0] - xTarget);
      for (var i = 1; i < pts.length; i++) {{
        var d = Math.abs(pts[i][0] - xTarget);
        if (d < bestDist) {{ bestDist = d; best = pts[i]; }}
      }}
      var cx = padl + ((best[0] - xmin) / (xmax - xmin || 1)) * pw;
      if (cross) {{
        cross.setAttribute("x1", cx.toFixed(1));
        cross.setAttribute("x2", cx.toFixed(1));
        cross.setAttribute("visibility", "visible");
      }}
      if (tip) {{
        tip.textContent = "x=" + best[0].toPrecision(5) + " | y=" + best[1].toPrecision(5);
      }}
    }});
    svg.addEventListener("mouseleave", function() {{
      if (cross) cross.setAttribute("visibility", "hidden");
      if (tip) tip.textContent = "Hover over chart to inspect values";
    }});
  }});
}})();
</script>
</body>
</html>
"""

## Module `prediction.py`

In [ ]:
"""Locked pattern-level prediction across validated analyses."""


import math
from collections import Counter
from typing import Any, Dict, List, Set, Tuple


def _motif_of_analysis(res: Dict[str, Any]) -> Set[Tuple[str, str]]:
    """Extract complementary estimator pairs from the target flux only."""
    target_name = None
    try:
        target_name = res.get("universal_contract", {}).get("target")
    except Exception:
        pass

    pairs: Set[Tuple[str, str]] = set()
    try:
        inter = res.get("levels", {}).get("L7_interactions", {})
        per = inter.get("per_flux", {})
        if target_name and target_name in per:
            per = {target_name: per[target_name]}
        for pf in per.values():
            for pr in pf.get("complementary_pairs", []):
                if pr.get("estimator_a") and pr.get("estimator_b"):
                    pairs.add((str(pr["estimator_a"]), str(pr["estimator_b"])))
                elif pr.get("estimator_pre") and pr.get("estimator_post"):
                    pairs.add((str(pr["estimator_pre"]), str(pr["estimator_post"])))
    except Exception:
        pass
    return pairs


def _holdout_motif_check(res: Dict[str, Any]) -> Tuple[bool, str]:
    """Temporal holdout check: build motif from earlier segments and verify on final segment."""
    try:
        inter = res.get("levels", {}).get("L7_interactions", {})
        segs = inter.get("segments", [])
        if len(segs) < 3:
            return False, "fewer than 3 segments"
        last = len(segs) - 1
        per_flux = inter.get("per_flux", {})
        tgt = res.get("universal_contract", {}).get("target")
        if tgt and tgt in per_flux:
            per_flux = {tgt: per_flux[tgt]}
        if not per_flux:
            return False, "missing per_flux interactions"

        oks, tot = 0, 0
        for pf in per_flux.values():
            profiles = pf.get("profiles", {})
            train_candidates: List[Tuple[str, float]] = []
            for name, profile in profiles.items():
                vals = profile.get("informativeness_per_segment", [])
                train_vals = [
                    v for v in vals[:last] if isinstance(v, (int, float)) and math.isfinite(v)
                ]
                if train_vals:
                    train_candidates.append((name, float(max(train_vals))))
            if not train_candidates:
                continue
            train_candidates.sort(key=lambda z: z[1], reverse=True)
            predicted = {name for name, _ in train_candidates[:3]}
            last_top = pf.get("per_segment_top", {}).get(f"segment_{last}", [])
            last_names = {x.get("estimator") for x in last_top}
            tot += 1
            if predicted & last_names:
                oks += 1
        if tot == 0:
            return False, "no computable motif on training segments"
        return (oks / tot >= 0.5), f"{oks}/{tot} flux(es): training top overlaps held-out segment"
    except Exception as exc:
        return False, f"holdout error: {exc}"


def predict_universal(analyses: List[Dict[str, Any]], question: str) -> Dict[str, Any]:
    """Locked cross-study motif prediction; refuses unless every quality gate passes."""
    reasons: List[str] = []
    if not isinstance(question, str) or not question.strip():
        reasons.append(
            "refined research question missing: state the hypothesis and falsification criterion"
        )
    if not isinstance(analyses, list) or len(analyses) < 5:
        n_got = len(analyses) if isinstance(analyses, list) else 0
        reasons.append(f"insufficient analyses: {n_got}/5 minimum required on the same protocol")
    if reasons:
        return {
            "status": "refused",
            "prediction": None,
            "reasons": reasons,
            "note": "prediction refused — gather >= 5 comparable validated analyses first",
        }

    metas = []
    for i, res in enumerate(analyses):
        md = res.get("prediction_metadata", {}) if isinstance(res, dict) else {}
        qid, pid = md.get("question_id"), md.get("protocol_id")
        if not qid or not pid:
            reasons.append(f"analysis {i}: missing question_id/protocol_id (protocol identity required)")
        else:
            metas.append((str(qid), str(pid)))
    if metas and (len(metas) != len(analyses) or len(set(metas)) != 1):
        reasons.append("incompatible analyses: question_id or protocol_id mismatch")

    for i, res in enumerate(analyses):
        g = res.get("grade", {}).get("grade") if isinstance(res, dict) else None
        if g not in ("A", "B"):
            reasons.append(f"analysis {i}: grade is {g} (requires A or B)")

        cfg_res = res.get("config", {}) if isinstance(res, dict) else {}
        l7_res = res.get("levels", {}).get("L7_interactions", {}) if isinstance(res, dict) else {}
        if cfg_res.get("run_complexity_diagnostics") is not True or l7_res.get("status") != "ok":
            reasons.append(
                f"analysis {i}: exploratory L7 complexity diagnostics not active/available "
                "(requires run_complexity_diagnostics=True)"
            )

        scan = res.get("scan", {}) if isinstance(res, dict) else {}
        p_corr = scan.get(
            "p_channel_adjusted",
            scan.get("p_scan_null", scan.get("p_permutation_scan_corrected", scan.get("p_permutation"))),
        )
        if not (isinstance(p_corr, (int, float)) and math.isfinite(p_corr) and p_corr <= 0.05):
            reasons.append(f"analysis {i}: adjusted scan p={p_corr} (requires <= 0.05)")

        wf_dict = res.get("walk_forward", {}) if isinstance(res, dict) else {}
        wf = wf_dict.get("mean_f1")
        wf_votes = wf_dict.get("n_splits_used", 0)
        if not (isinstance(wf, (int, float)) and wf >= 0.50 and wf_votes >= 2):
            reasons.append(f"analysis {i}: walk-forward F1={wf}, votes={wf_votes} (requires >= 0.50 and >= 2)")

        stab = (
            res.get("falsification", {}).get("multi_window", {}).get("stability")
            if isinstance(res, dict)
            else None
        )
        if not (isinstance(stab, (int, float)) and stab >= 0.60):
            reasons.append(f"analysis {i}: multi-window stability={stab} (requires >= 0.60)")

    motifs = [_motif_of_analysis(r) for r in analyses]
    cnt: Counter[Tuple[str, str]] = Counter()
    for m in motifs:
        for pair in m:
            cnt[pair] += 1
    common = [pair for pair, c in cnt.items() if c >= 3]
    if not common:
        reasons.append("no shared motif: no complementary pair appears in >= 3 analyses")

    hold_ok: List[bool] = []
    hold_details: List[str] = []
    for i, res in enumerate(analyses):
        ok, det = _holdout_motif_check(res)
        hold_ok.append(ok)
        hold_details.append(f"analysis {i}: {det}")
    if sum(hold_ok) / max(len(hold_ok), 1) < 0.5:
        reasons.append(
            f"held-out segment check failed: {sum(hold_ok)}/{len(hold_ok)} pass (requires >= 50%)"
        )

    if reasons:
        return {
            "status": "refused",
            "prediction": None,
            "reasons": reasons,
            "holdout_details": hold_details,
            "note": "prediction refused — motif is not yet stable across replications",
        }

    common_sorted = sorted(common, key=lambda pr: cnt[pr], reverse=True)[:3]
    motif = [
        {"estimator_a": a, "estimator_b": b, "n_analyses_sharing": int(cnt[(a, b)])}
        for a, b in common_sorted
    ]
    return {
        "status": "allowed",
        "question": question,
        "motif": motif,
        "prediction": {
            "type": "next_segment_profile",
            "statement": (
                "IF the regime holds, THEN the next segment should show late estimator (b) "
                "informative (|Cohen-d| >= 0.8) and early estimator (a) non-informative (< 0.5)."
            ),
            "pairs": motif,
            "interval": "late informativeness |Cohen-d| >= 0.8, early < 0.5 (never point prediction)",
            "condition": "same protocol, same question, grade A/B on incoming segment",
        },
        "falsifiable_test": (
            "Analyze the next segment with analyze(): if late estimator (b) is NOT among top "
            "informative estimators, the motif prediction is falsified and rejected."
        ),
        "holdout_details": hold_details,
        "note": "Interval motif prediction only; must be verified on unseen future segment.",
    }

## Module `pipeline.py`

In [ ]:
"""Main analysis pipeline: analyze(), auto-channel selection, and report generation."""


import copy
import hashlib
import json
import math
import sys
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd



DEFAULT_ROUTING_GATES: Dict[str, float] = {
    "slope_min_abs_rho": 0.85,
    "slope_max_phi_within": 0.45,
    "slope_min_abs_t": 4.0,
    "jump_min_phi_raw": 0.55,
    "jump_max_phi_dy": -0.15,
    "jump_min_abs_d": 2.0,
    "jump_min_t": 4.0,
    "vol_min_phi_raw": 0.70,
    "vol_max_abs_phi_dy": 0.25,
    "vol_min_abs_d": 0.50,
    "deseas_min_dom_frac": 0.50,
    "deseas_min_abs_d": 0.80,
    "deseas_min_ratio": 1.50,
}


def _auto_select_primary_scan(
    ds: UniversalDataset,
    level_scan: Dict[str, Any],
    structural: Dict[str, Any],
    cfg: UniversalConfig,
    gates: Optional[Dict[str, float]] = None,
) -> Dict[str, Any]:
    """Select the most structurally appropriate break detector without domain priors."""
    if cfg.detection_mode != "auto" or level_scan.get("status") != "ok":
        if level_scan.get("status") == "ok":
            out_m = dict(level_scan)
            out_m.setdefault("stat_kind", "mean")
            out_m.setdefault("pre_stat", level_scan.get("mean_pre"))
            out_m.setdefault("post_stat", level_scan.get("mean_post"))
            return out_m
        return level_scan

    g = dict(DEFAULT_ROUTING_GATES)
    if gates:
        g.update(gates)

    y = ds.target
    x = ds.control
    n = ds.n
    idx_raw = int(level_scan["idx"])
    phi_raw = within_regime_ar1(y, idx_raw)

    dy = np.diff(y)
    phi_dy = 0.0
    if len(dy) >= 10 and np.std(dy) > 0.0:
        phi_dy = float(np.corrcoef(dy[:-1], dy[1:])[0, 1])
        if not math.isfinite(phi_dy):
            phi_dy = 0.0

    try:
        rho = float(pd.Series(x).corr(pd.Series(y), method="spearman"))
    except Exception:
        rho = 0.0

    # 1. Monotonic piecewise slope break (e.g. slope change without level jump)
    sb = structural.get("slope_break", {})
    if (
        math.isfinite(rho)
        and abs(rho) >= g["slope_min_abs_rho"]
        and sb.get("status") == "ok"
        and sb.get("phi_within", 1.0) < g["slope_max_phi_within"]
        and sb.get("p_residual_block_perm_scan", 1.0) <= 0.05
        and abs(sb.get("t_stat", 0.0)) >= g["slope_min_abs_t"]
    ):
        idx_s = int(sb["idx"])
        w_s = int(sb["window_used"])
        t_arr = np.arange(w_s, dtype=float)
        pre_y = y[idx_s - w_s : idx_s]
        post_y = y[idx_s : idx_s + w_s]
        pre_b, pre_a = np.polyfit(t_arr, pre_y, 1)
        post_b, post_a = np.polyfit(t_arr, post_y, 1)
        pre_s = float(pre_b)
        post_s = float(post_b)
        d_eq = float(sb["cohen_d_equiv"])
        # Residual bootstrap CI on the exact same slope-break standardized effect metric
        rng_sb = np.random.default_rng(cfg.seed)
        res_l = pre_y - (pre_a + pre_b * t_arr)
        res_r = post_y - (post_a + post_b * t_arr)
        tc = t_arr - float(np.mean(t_arr))
        denom_t = float(np.sum(tc * tc))
        boot_vals = []
        for _ in range(int(cfg.n_bootstrap)):
            yb_l = (pre_a + pre_b * t_arr) + rng_sb.choice(res_l, size=w_s, replace=True)
            yb_r = (post_a + post_b * t_arr) + rng_sb.choice(res_r, size=w_s, replace=True)
            bl_b = float(np.sum(tc * (yb_l - np.mean(yb_l))) / denom_t)
            br_b = float(np.sum(tc * (yb_r - np.mean(yb_r))) / denom_t)
            al_b = float(np.mean(yb_l)) - bl_b * float(np.mean(t_arr))
            ar_b = float(np.mean(yb_r)) - br_b * float(np.mean(t_arr))
            sse_b = float(
                np.sum((yb_l - (al_b + bl_b * t_arr)) ** 2)
                + np.sum((yb_r - (ar_b + br_b * t_arr)) ** 2)
            )
            sd_b = math.sqrt(max(sse_b / max(2 * w_s - 4, 1), 1e-18))
            se_b = sd_b * math.sqrt(2.0 / denom_t)
            t_b = (br_b - bl_b) / max(se_b, 1e-18)
            boot_vals.append(t_b / math.sqrt(w_s))
        lo_d, hi_d = np.percentile(boot_vals, [2.5, 97.5]) if len(boot_vals) >= 10 else (d_eq, d_eq)
        p_sb = float(sb["p_residual_block_perm_scan"])
        return {
            "status": "ok",
            "detector": "slope_break",
            "stat_kind": "slope_per_step",
            "pre_stat": pre_s,
            "post_stat": post_s,
            "slope_pre": pre_s,
            "slope_post": post_s,
            "idx": idx_s,
            "control_value": float(x[idx_s]),
            "cohen_d": d_eq,
            "t_stat": float(sb["t_stat"]),
            "n_pre": w_s,
            "n_post": w_s,
            "mean_pre": float(np.mean(pre_y)),
            "mean_post": float(np.mean(post_y)),
            "window_used": w_s,
            "window": w_s,
            "cohen_d_ci_low": float(lo_d) if math.isfinite(lo_d) else d_eq,
            "cohen_d_ci_high": float(hi_d) if math.isfinite(hi_d) else d_eq,
            "null_model": sb.get("null_model", "residual_block_permutation_scan"),
            "null_block_len": sb.get("null_block_len"),
            "p_residual_block_perm_scan": p_sb,
            "p_screen": sb.get("p_screen"),
            "p_refined": sb.get("p_refined"),
            "n_scan_perm_used": sb.get("n_scan_perm_used"),
            "p_scan_null": p_sb,
            "p_channel_raw": p_sb,
            "p_permutation": p_sb,
            "p_permutation_local": p_sb,
            "p_permutation_global": p_sb,
            "p_permutation_scan_corrected": None,
            "cohen_d_global": d_eq,
            "selection_reason": "monotonic trend with clean piecewise-linear slope break",
        }

    # 2. Abrupt discontinuity masked by smooth non-random-walk cycle/trend
    dj = structural.get("detrended_jump", {})
    w_sm = int(level_scan.get("window_used", max(6, n // 10)))
    if (
        phi_raw >= g["jump_min_phi_raw"]
        and phi_dy < g["jump_max_phi_dy"]
        and dj.get("status") == "ok"
        and abs(dj.get("effect_d", 0.0)) >= g["jump_min_abs_d"]
        and dj.get("t_stat", 0.0) >= g["jump_min_t"]
        and dj.get("p_residual_block_perm_scan", 1.0) <= 0.05
        and abs(int(dj["idx"]) - idx_raw) > max(4, w_sm // 2)
    ):
        idx_j = int(dj["idx"])
        w_j = int(dj["window_used"])
        t_arr = np.arange(w_j, dtype=float)
        tc_j = t_arr - float(np.mean(t_arr))
        denom_j = float(np.sum(tc_j * tc_j))
        yl = y[idx_j - w_j : idx_j]
        yr = y[idx_j : idx_j + w_j]
        bl, al = np.polyfit(t_arr, yl, 1)
        br, ar = np.polyfit(t_arr, yr, 1)
        rl = yl - (al + bl * t_arr)
        rr = yr - (ar + br * t_arr)
        # Bootstrap exact same OLS boundary-jump/sd estimator (no mixed ddof variance mismatch)
        rng_dj = np.random.default_rng(cfg.seed)
        boot_dj: List[float] = []
        for _ in range(int(cfg.n_bootstrap)):
            yl_b = (al + bl * t_arr) + rng_dj.choice(rl, size=w_j, replace=True)
            yr_b = (ar + br * t_arr) + rng_dj.choice(rr, size=w_j, replace=True)
            bl_b = float(np.sum(tc_j * (yl_b - np.mean(yl_b))) / denom_j)
            br_b = float(np.sum(tc_j * (yr_b - np.mean(yr_b))) / denom_j)
            al_b = float(np.mean(yl_b)) - bl_b * float(np.mean(t_arr))
            ar_b = float(np.mean(yr_b)) - br_b * float(np.mean(t_arr))
            sse_b = float(
                np.sum((yl_b - (al_b + bl_b * t_arr)) ** 2)
                + np.sum((yr_b - (ar_b + br_b * t_arr)) ** 2)
            )
            sd_b = math.sqrt(max(sse_b / max(2 * w_j - 4, 1), 1e-18))
            boot_dj.append(float((ar_b - (al_b + bl_b * w_j)) / sd_b))
        d_eff_j = float(dj["effect_d"])
        lo_j, hi_j = np.percentile(boot_dj, [2.5, 97.5]) if len(boot_dj) >= 10 else (d_eff_j, d_eff_j)
        p_j_loc = permutation_p_value(rl, rr + float(ar - (al + bl * w_j)), n_perm=cfg.n_permutations, seed=cfg.seed)
        p_dj = float(dj["p_residual_block_perm_scan"])
        pre_ext = float(al + bl * w_j)
        post_ext = float(ar)
        return {
            "status": "ok",
            "detector": "detrended_jump",
            "stat_kind": "extrapolated_boundary_level",
            "pre_stat": pre_ext,
            "post_stat": post_ext,
            "idx": idx_j,
            "control_value": float(x[idx_j]),
            "cohen_d": d_eff_j,
            "t_stat": float(dj["t_stat"]),
            "n_pre": w_j,
            "n_post": w_j,
            "mean_pre": float(np.mean(yl)),
            "mean_post": float(np.mean(yr)),
            "window_used": w_j,
            "window": w_j,
            "cohen_d_ci_low": float(lo_j) if math.isfinite(lo_j) else d_eff_j,
            "cohen_d_ci_high": float(hi_j) if math.isfinite(hi_j) else d_eff_j,
            "null_model": dj.get("null_model", "residual_block_permutation_scan"),
            "null_block_len": dj.get("null_block_len"),
            "p_residual_block_perm_scan": p_dj,
            "p_screen": dj.get("p_screen"),
            "p_refined": dj.get("p_refined"),
            "n_scan_perm_used": dj.get("n_scan_perm_used"),
            "p_scan_null": p_dj,
            "p_channel_raw": p_dj,
            "p_permutation": p_dj,
            "p_permutation_local": float(p_j_loc),
            "p_permutation_global": p_dj,
            "p_permutation_scan_corrected": None,
            "cohen_d_global": d_eff_j,
            "selection_reason": "level shift confounded by smooth background oscillation; abrupt detrended jump detected",
        }

    # 3. Volatility / dispersion regime change when level series is a non-stationary random walk
    vs = structural.get("volatility_shift", {})
    if (
        phi_raw >= g["vol_min_phi_raw"]
        and abs(phi_dy) < g["vol_max_abs_phi_dy"]
        and vs.get("status") == "ok"
        and abs(vs.get("cohen_d", 0.0)) >= g["vol_min_abs_d"]
        and vs.get("p_permutation_scan_corrected", 1.0) <= 0.05
    ):
        idx_v = int(vs["idx"])
        w_v = int(vs["window_used"])
        vy = _volatility_deviation_series(y)
        v_cut = int(vs.get("idx_vy", max(1, min(len(vy) - 1, idx_v - 1))))
        pre_v = vy[max(0, v_cut - w_v) : v_cut]
        post_v = vy[v_cut : min(len(vy), v_cut + w_v)]
        d_v, lo_v, hi_v = bootstrap_ci_cohen_d(pre_v, post_v, n_boot=cfg.n_bootstrap, seed=cfg.seed)
        p_v_loc = permutation_p_value(pre_v, post_v, n_perm=cfg.n_permutations, seed=cfg.seed)
        d_v_glob = cohen_d(vy[:v_cut], vy[v_cut:])
        d_v_win = float(vs["cohen_d"])
        mean_pre_v = float(np.mean(pre_v)) if len(pre_v) else float("nan")
        mean_post_v = float(np.mean(post_v)) if len(post_v) else float("nan")
        p_vol = float(vs["p_permutation_scan_corrected"])
        return {
            "status": "ok",
            "detector": "volatility_shift",
            "stat_kind": "volatility_deviation_mean",
            "pre_stat": mean_pre_v,
            "post_stat": mean_post_v,
            "idx": idx_v,
            "control_value": float(x[idx_v]),
            "cohen_d": d_v_win if not math.isfinite(d_v) else float(d_v),
            "n_pre": int(len(pre_v)),
            "n_post": int(len(post_v)),
            "mean_pre": mean_pre_v,
            "mean_post": mean_post_v,
            "window_used": w_v,
            "window": w_v,
            "cohen_d_ci_low": float(lo_v) if math.isfinite(lo_v) else d_v_win,
            "cohen_d_ci_high": float(hi_v) if math.isfinite(hi_v) else d_v_win,
            "null_model": vs.get("null_model", "target_permutation_scan"),
            "p_screen": vs.get("p_screen"),
            "p_refined": vs.get("p_refined"),
            "n_scan_perm_used": vs.get("n_scan_perm_used"),
            "p_scan_null": p_vol,
            "p_channel_raw": p_vol,
            "p_permutation": p_vol,
            "p_permutation_local": float(p_v_loc),
            "p_permutation_global": p_vol,
            "p_permutation_scan_corrected": p_vol,
            "cohen_d_global": float(d_v_glob) if math.isfinite(d_v_glob) else d_v_win,
            "selection_reason": "level series exhibits unit-root random walk; significant dispersion/volatility shift detected",
        }

    # 4. Dominant periodic/seasonal cycle masking an underlying shift
    ds_chan = structural.get("deseasonalized_shift", {})
    if (
        ds_chan.get("status") == "ok"
        and ds_chan.get("dominant_power_fraction", 0.0) >= g["deseas_min_dom_frac"]
        and ds_chan.get("p_permutation_scan_corrected", 1.0) <= 0.05
        and abs(ds_chan.get("cohen_d", 0.0)) >= max(g["deseas_min_abs_d"], g["deseas_min_ratio"] * abs(level_scan.get("cohen_d", 0.0)))
    ):
        des_out = _deseasonalize_fft_dominant(y)
        if des_out is not None:
            y_des = des_out[0]
            idx_d = int(ds_chan["idx"])
            w_d = int(ds_chan["window_used"])
            pre_d = y_des[max(0, idx_d - w_d) : idx_d]
            post_d = y_des[idx_d : min(n, idx_d + w_d)]
            d_val, lo_d, hi_d = bootstrap_ci_cohen_d(pre_d, post_d, n_boot=cfg.n_bootstrap, seed=cfg.seed)
            p_loc_d = permutation_p_value(pre_d, post_d, n_perm=cfg.n_permutations, seed=cfg.seed)
            d_glob = cohen_d(y_des[:idx_d], y_des[idx_d:])
            p_glob_d = permutation_p_value(y_des[:idx_d], y_des[idx_d:], n_perm=cfg.n_permutations, seed=cfg.seed + 7)
            mean_pre_d = float(np.mean(pre_d))
            mean_post_d = float(np.mean(post_d))
            p_des = float(ds_chan["p_permutation_scan_corrected"])
            return {
                "status": "ok",
                "detector": "deseasonalized_shift",
                "stat_kind": "deseasonalized_mean",
                "pre_stat": mean_pre_d,
                "post_stat": mean_post_d,
                "idx": idx_d,
                "control_value": float(x[idx_d]),
                "cohen_d": float(d_val) if math.isfinite(d_val) else float(ds_chan["cohen_d"]),
                "n_pre": int(len(pre_d)),
                "n_post": int(len(post_d)),
                "mean_pre": mean_pre_d,
                "mean_post": mean_post_d,
                "window_used": w_d,
                "window": w_d,
                "cohen_d_ci_low": float(lo_d) if math.isfinite(lo_d) else float(ds_chan["cohen_d"]),
                "cohen_d_ci_high": float(hi_d) if math.isfinite(hi_d) else float(ds_chan["cohen_d"]),
                "null_model": ds_chan.get("null_model", "target_permutation_scan"),
                "p_screen": ds_chan.get("p_screen"),
                "p_refined": ds_chan.get("p_refined"),
                "n_scan_perm_used": ds_chan.get("n_scan_perm_used"),
                "p_scan_null": p_des,
                "p_channel_raw": p_des,
                "p_permutation": p_des,
                "p_permutation_local": float(p_loc_d),
                "p_permutation_global": float(p_glob_d),
                "p_permutation_scan_corrected": p_des,
                "cohen_d_global": float(d_glob),
                "dominant_power_fraction": float(ds_chan["dominant_power_fraction"]),
                "removed_fft_bins": ds_chan.get("removed_fft_bins", []),
                "selection_reason": "strong periodic cycle masked underlying regime shift; FFT deseasonalization applied",
            }

    out_lvl = dict(level_scan)
    out_lvl["stat_kind"] = "mean"
    out_lvl["pre_stat"] = level_scan.get("mean_pre")
    out_lvl["post_stat"] = level_scan.get("mean_post")
    return out_lvl


def _apply_channel_multiplicity(
    best: Dict[str, Any],
    level_scan: Dict[str, Any],
    structural_scans: Dict[str, Any],
    detection_mode: str,
) -> None:
    """Apply Holm-Bonferroni correction across the family of finite evaluated channels."""
    candidates: List[Tuple[str, Dict[str, Any], float]] = []
    pool: List[Tuple[str, Dict[str, Any]]] = (
        [("level_shift", level_scan)]
        if detection_mode != "auto"
        else [
            ("level_shift", level_scan),
            ("detrended_jump", structural_scans.get("detrended_jump", {})),
            ("slope_break", structural_scans.get("slope_break", {})),
            ("volatility_shift", structural_scans.get("volatility_shift", {})),
            ("deseasonalized_shift", structural_scans.get("deseasonalized_shift", {})),
        ]
    )
    for cname, cdict in pool:
        if isinstance(cdict, dict) and cdict.get("status") == "ok":
            p_raw = cdict.get("p_channel_raw", cdict.get("p_scan_null"))
            if isinstance(p_raw, (int, float)) and math.isfinite(float(p_raw)):
                candidates.append((cname, cdict, float(p_raw)))

    m = len(candidates)
    if m == 0:
        return

    candidates.sort(key=lambda item: item[2])
    method = "single_channel" if m == 1 else "holm_bonferroni"
    running_max = 0.0
    adjusted_by_name: Dict[str, float] = {}
    for rank_idx, (cname, cdict, p_raw) in enumerate(candidates):
        adj = min(1.0, (m - rank_idx) * p_raw)
        running_max = max(running_max, adj)
        cdict["p_channel_raw"] = float(p_raw)
        cdict["p_channel_adjusted"] = float(running_max)
        cdict["n_channels_family"] = int(m)
        cdict["multiplicity_method"] = method
        adjusted_by_name[cname] = float(running_max)

    if best.get("status") == "ok":
        det = str(best.get("detector", "level_shift"))
        p_raw_best = best.get("p_channel_raw", best.get("p_scan_null"))
        best["n_channels_family"] = int(m)
        best["multiplicity_method"] = method
        if det in adjusted_by_name:
            best["p_channel_adjusted"] = adjusted_by_name[det]
        elif isinstance(p_raw_best, (int, float)) and math.isfinite(float(p_raw_best)):
            best["p_channel_adjusted"] = float(min(1.0, m * float(p_raw_best)))
        best["p_channel_raw"] = float(p_raw_best) if p_raw_best is not None else None


def _strict_json(obj: Any) -> Any:
    """Replace non-finite floats with None so results.json is strict JSON."""
    if isinstance(obj, dict):
        return {str(k): _strict_json(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_strict_json(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return _strict_json(obj.tolist())
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        value = float(obj)
        return value if math.isfinite(value) else None
    if isinstance(obj, float):
        return obj if math.isfinite(obj) else None
    if isinstance(obj, np.bool_):
        return bool(obj)
    return obj


def analyze(
    df_or_path: Union[pd.DataFrame, str, Path],
    control: Optional[Any] = None,
    target: Optional[Any] = None,
    fluxes: Optional[Sequence[Any]] = None,
    config: Optional[UniversalConfig] = None,
    output_dir: Optional[Union[str, Path]] = None,
) -> Dict[str, Any]:
    """Run full domain-agnostic threshold detection, falsification, and L1..L7 diagnostics."""
    cfg = copy.deepcopy(config) if config is not None else UniversalConfig()
    defaults = UniversalConfig()
    if cfg.mode in MODE_PRESETS:
        for k, v in MODE_PRESETS[cfg.mode].items():
            if k == "window_sizes":
                continue
            if hasattr(cfg, k) and getattr(cfg, k) == getattr(defaults, k):
                setattr(cfg, k, v)
    cfg.output_dir = str(output_dir or cfg.output_dir)
    t_start = time.time()

    # Load input
    if isinstance(df_or_path, (str, Path)):
        path = Path(df_or_path)
        src_hash = hashlib.sha256(path.read_bytes()).hexdigest() if path.exists() else ""
        src_name = str(path)
        suffix = path.suffix.lower()
        if suffix in (".parquet", ".pq"):
            df = pd.read_parquet(path)
        elif suffix == ".json":
            df = pd.read_json(path)
        else:
            df = pd.read_csv(path)
    else:
        df = df_or_path
        src_hash = hashlib.sha256(
            pd.util.hash_pandas_object(df, index=True).values.tobytes()
        ).hexdigest()
        src_name = "dataframe"

    ds = load_universal_dataset(
        df,
        control=control,
        target=target,
        fluxes=fluxes,
        source_name=src_name,
        source_hash=src_hash,
    )

    if cfg.use_relative_windows:
        ws = sorted(
            {
                max(cfg.min_group * 2, ds.n // 10),
                max(cfg.min_group * 2, ds.n // 5),
                max(cfg.min_group * 2, ds.n // 3),
            }
        )
        ws = [w for w in ws if w < ds.n // 2]
        if ws:
            cfg.window_sizes = ws

    log_message(
        cfg,
        f"[UNIVERSAL_LOG] n={ds.n} control={ds.control_name} target={ds.target_name} "
        f"fluxes={len(ds.flux_names)} windows={cfg.window_sizes} mode={cfg.mode}",
    )
    if ds.n < cfg.min_length:
        log_message(
            cfg,
            f"[WARNING] n={ds.n} < min_length={cfg.min_length}: exploratory results only.",
        )

    # Exploratory 29 complexity estimators (opt-in via run_complexity_diagnostics=True)
    if cfg.run_complexity_diagnostics:
        est_by_flux: Dict[str, Any] = {
            name: apply_estimators(ds.fluxes[name]) for name in ds.flux_names
        }
    else:
        est_by_flux = {
            "status": "disabled_by_default",
            "exploratory": True,
            "note": "enable with run_complexity_diagnostics=True or --complexity-diagnostics",
        }

    # Primary level-shift scan + multi-channel structural scans
    w_sm = cfg.window_sizes[0] if cfg.window_sizes else 80
    w_lg = cfg.window_sizes[-1] if cfg.window_sizes else max(w_sm, ds.n // 3)
    level_scan = scan_threshold(
        ds.control,
        ds.target,
        window=w_sm,
        min_group=cfg.min_group,
        seed=cfg.seed,
        n_boot=cfg.n_bootstrap,
        n_perm=cfg.n_permutations,
        n_scan_perm=cfg.n_scan_permutations,
        compute_scan_p=True,
        n_refined_perm=int(cfg.n_scan_refined_permutations),
    )
    structural_scans = scan_structural_channels(
        ds.control,
        ds.target,
        w_small=w_sm,
        w_large=w_lg,
        min_group=cfg.min_group,
        n_perm=int(cfg.n_scan_permutations),
        seed=cfg.seed,
        n_refined_perm=int(cfg.n_scan_refined_permutations),
    )
    best = _auto_select_primary_scan(
        ds,
        level_scan,
        structural_scans,
        cfg,
        gates=getattr(cfg, "routing_gates", None),
    )
    _apply_channel_multiplicity(best, level_scan, structural_scans, cfg.detection_mode)
    multivariate_scan = scan_multivariate_channels(
        ds.control,
        ds.fluxes,
        window=w_sm,
        min_group=cfg.min_group,
        n_perm=min(200, cfg.n_scan_permutations),
        seed=cfg.seed,
    )
    if best.get("detector") == "volatility_shift" and ds.n >= 4:
        mt_ctrl = ds.control[1:]
        mt_tgt = _volatility_deviation_series(ds.target)
    elif best.get("detector") == "deseasonalized_shift" and ds.n >= 24:
        des_mt = _deseasonalize_fft_dominant(ds.target)
        mt_ctrl = ds.control
        mt_tgt = des_mt[0] if des_mt is not None else ds.target
    else:
        mt_ctrl, mt_tgt = ds.control, ds.target
    multi_thresholds = detect_multiple_thresholds(
        mt_ctrl,
        mt_tgt,
        min_group=cfg.min_group,
        n_perm=min(150, cfg.n_scan_permutations),
        seed=cfg.seed,
    )

    cascade = (
        cascade_5_thresholds(ds.control, ds.target, best["idx"], seed=cfg.seed)
        if best.get("status") == "ok"
        else {"status": "no_threshold"}
    )

    # Optional segmented MC + trajectory validation suite
    if cfg.run_estimator_validation:
        cut_idx = best.get("idx") if best.get("status") == "ok" else None
        validation_by_flux = {
            name: estimator_validation_suite(
                ds.fluxes[name],
                seed=cfg.seed,
                n_mc_paths=cfg.n_mc_paths,
                control=ds.control,
                threshold_idx=cut_idx,
            )
            for name in ds.flux_names
        }
    else:
        validation_by_flux = {
            "status": "skipped_by_default",
            "note": "enable with run_estimator_validation=True or --full-validation",
        }

    # Falsification + walk-forward + L1..L7 + conservative grading
    falsif = falsification_suite(ds, best, cfg)
    wf = walk_forward(ds, best, cfg)
    levels = level_universal(ds, best, cascade, est_by_flux, cfg, falsif=falsif)
    grade = grade_evidence(best, falsif, wf, cfg)

    # Column-rename invariance check (P3)
    rename_ok: Optional[bool] = None
    try:
        df_ren = df.rename(columns={c: f"X{i}" for i, c in enumerate(df.columns)})
        cols_list = list(df.columns)
        ctrl_r = f"X{cols_list.index(ds.control_name)}" if ds.control_name in cols_list else None
        tgt_r = f"X{cols_list.index(ds.target_name)}" if ds.target_name in cols_list else None
        if ctrl_r and tgt_r:
            ds_r = load_universal_dataset(df_ren, control=ctrl_r, target=tgt_r)
            best_r = scan_threshold(
                ds_r.control,
                ds_r.target,
                window=w_sm,
                min_group=cfg.min_group,
                n_boot=20,
                n_perm=50,
                seed=cfg.seed,
                compute_scan_p=False,
            )
            rename_ok = bool(
                level_scan.get("status") == "ok"
                and best_r.get("status") == "ok"
                and abs(level_scan["cohen_d"] - best_r["cohen_d"]) < 1e-9
            )
            det_name = best.get("detector", "level_shift")
            if rename_ok and best.get("status") == "ok" and det_name != "level_shift":
                w_act = int(best.get("window_used", w_sm))
                if det_name == "slope_break":
                    _, _, ts_r = _local_linear_stats_vec(ds_r.target, w_act)
                    val_r = float(ts_r[int(np.argmax(np.abs(ts_r)))] / math.sqrt(w_act)) if len(ts_r) else float("nan")
                    rename_ok = math.isfinite(val_r) and abs(float(best["cohen_d"]) - val_r) < 1e-9
                elif det_name == "detrended_jump":
                    _, dj_r, _ = _local_linear_stats_vec(ds_r.target, w_act)
                    val_r = float(dj_r[int(np.argmax(np.abs(dj_r)))]) if len(dj_r) else float("nan")
                    rename_ok = math.isfinite(val_r) and abs(float(best["cohen_d"]) - val_r) < 1e-9
                elif det_name == "volatility_shift":
                    vy_r = _volatility_deviation_series(ds_r.target)
                    dv_r, iv_r = _scan_max_d(ds_r.control[1:], vy_r, window=w_act, min_group=cfg.min_group)
                    rename_ok = iv_r is not None and math.isfinite(dv_r) and abs(float(ds_r.control[min(ds_r.n - 1, iv_r + 1)]) - float(best["control_value"])) < 1e-9
                elif det_name == "deseasonalized_shift":
                    des_r = _deseasonalize_fft_dominant(ds_r.target)
                    if des_r is not None:
                        dd_r, id_r = _scan_max_d(ds_r.control, des_r[0], window=w_act, min_group=cfg.min_group)
                        rename_ok = id_r is not None and math.isfinite(dd_r) and abs(float(ds_r.control[id_r]) - float(best["control_value"])) < 1e-9
                    else:
                        rename_ok = False
    except Exception:
        rename_ok = False

    elapsed = float(time.time() - t_start)
    l5_info = levels.get("L5_complexity", {})
    t_log_target = l5_info.get(
        "t_log",
        {"status": "not_computed", "calibrated": False, "t_log": float("nan")},
    )

    result: Dict[str, Any] = {
        "version": VERSION,
        "version_master": VERSION_MASTER,
        "universal_contract": {
            "control": ds.control_name,
            "target": ds.target_name,
            "fluxes": ds.flux_names,
            "n": ds.n,
            "dropped_rows": ds.dropped_rows,
            "control_monotone": ds.control_monotone,
            "note": "No domain lexicon, no hand weights. Output is invariant to column naming.",
        },
        "config": cfg.to_dict(),
        "scan": best,
        "level_shift_scan": level_scan,
        "structural_scans": structural_scans,
        "multivariate_scan": multivariate_scan,
        "multi_thresholds": multi_thresholds,
        "cascade": cascade,
        "levels": levels,
        "falsification": falsif,
        "walk_forward": wf,
        "grade": grade,
        "estimators_by_flux": est_by_flux,
        "estimator_validation_by_flux": validation_by_flux,
        "prediction_metadata": {
            "question_id": None,
            "protocol_id": None,
            "note": "Populate explicitly before cross-study predict_universal() aggregation.",
        },
        "t_log_target": t_log_target,
        "invariance_rename_test": {
            "passed": rename_ok,
            "definition": "renaming columns must leave numerical outputs invariant (P3)",
        },
        "manifest": {
            "data_hash_sha256": src_hash,
            "data_source": src_name,
            "code_version": VERSION,
            "seed": cfg.seed,
            "mode": cfg.mode,
            "python": sys.version,
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "elapsed_seconds": elapsed,
            "reproducibility": "identical input + identical seed + identical version = identical output",
        },
    }

    # Write outputs. NaN/Inf become null so the file is strict JSON (parseable by json.loads).
    outdir = Path(cfg.output_dir)
    outdir.mkdir(parents=True, exist_ok=True)
    with open(outdir / "results.json", "w", encoding="utf-8") as f:
        json.dump(_strict_json(result), f, indent=2, ensure_ascii=False, allow_nan=False)

    rows: List[Dict[str, Any]] = []
    if best.get("status") == "ok":
        rows.append(
            {
                "threshold": "T0_central",
                "detector": best.get("detector", "level_shift"),
                "stat_kind": best.get("stat_kind", "mean"),
                "pre_stat": best.get("pre_stat"),
                "post_stat": best.get("post_stat"),
                "control_value": best["control_value"],
                "cohen_d": best["cohen_d"],
                "ci_low": best.get("cohen_d_ci_low"),
                "ci_high": best.get("cohen_d_ci_high"),
                "null_model": best.get("null_model"),
                "p_screen": best.get("p_screen"),
                "p_refined": best.get("p_refined"),
                "n_scan_perm_used": best.get("n_scan_perm_used"),
                "p_channel_raw": best.get("p_channel_raw", best.get("p_scan_null")),
                "p_channel_adjusted": best.get("p_channel_adjusted"),
                "n_channels_family": best.get("n_channels_family"),
                "p_permutation": best.get("p_permutation"),
            }
        )
    if isinstance(cascade, dict):
        for k in ["T_minus2", "T_minus1", "T_plus1", "T_plus2"]:
            v = cascade.get(k, {})
            rows.append(
                {
                    "threshold": k,
                    "detector": best.get("detector", "level_shift"),
                    "stat_kind": None,
                    "pre_stat": None,
                    "post_stat": None,
                    "control_value": v.get("control_value"),
                    "cohen_d": None,
                    "ci_low": None,
                    "ci_high": None,
                    "null_model": None,
                    "p_screen": None,
                    "p_refined": None,
                    "n_scan_perm_used": None,
                    "p_channel_raw": None,
                    "p_channel_adjusted": None,
                    "n_channels_family": None,
                    "p_permutation": None,
                }
            )
    pd.DataFrame(rows).to_csv(outdir / "thresholds.csv", index=False)

    lines: List[str] = [
        f"UNIVERSAL_LOG {VERSION} — Universal Threshold & Falsification Report",
        f"Control: {ds.control_name} | Target: {ds.target_name} | n = {ds.n} | Fluxes: {len(ds.flux_names)}",
        f"Grade: {grade['grade']} — {grade['verdict']}",
    ]
    if grade.get("reasons"):
        lines.append("Reasons: " + "; ".join(grade["reasons"]))
    if best.get("status") == "ok":
        p_raw_val = best.get("p_channel_raw", best.get("p_scan_null"))
        p_adj_val = best.get("p_channel_adjusted", p_raw_val)
        n_fam_val = best.get("n_channels_family", 1)
        null_name = best.get("null_model", "target_permutation_scan")
        p_scr_val = best.get("p_screen")
        p_ref_val = best.get("p_refined")
        p_scr_str = f"{p_scr_val:.4f}" if isinstance(p_scr_val, (int, float)) and math.isfinite(p_scr_val) else "N/A"
        p_ref_str = f"{p_ref_val:.4f}" if isinstance(p_ref_val, (int, float)) and math.isfinite(p_ref_val) else "N/A"
        stat_kind = best.get("stat_kind", "mean")
        stat_label = "slope per step" if stat_kind == "slope_per_step" else stat_kind
        pre_s_val = best.get("pre_stat")
        post_s_val = best.get("post_stat")
        lines.append(
            f"Threshold T0 ({best.get('detector', 'level_shift')}): {ds.control_name} = {best['control_value']:.4g} | "
            f"effect_d = {best['cohen_d']:.3f} CI95% [{best.get('cohen_d_ci_low'):.3f}, {best.get('cohen_d_ci_high'):.3f}] | "
            f"p_screen = {p_scr_str} | p_refined = {p_ref_str} | p_raw[{null_name}] = {p_raw_val:.4f} | p_holm(m={n_fam_val}) = {p_adj_val:.4f}"
        )
        lines.append(
            f"Pre/Post ({stat_label}): pre_stat = {pre_s_val:.4g} | post_stat = {post_s_val:.4g}"
            if (pre_s_val is not None and post_s_val is not None and math.isfinite(float(pre_s_val)) and math.isfinite(float(post_s_val)))
            else f"Pre/Post ({stat_label}): N/A"
        )
        lines.append(
            f"Direction L1: {levels['L1_direction'].get('direction')} | "
            f"Regime L3: {levels['L3_regime'].get('state')} | "
            f"Signal L4: {levels['L4_signal'].get('level')} | "
            f"p_channel_adjusted = {p_adj_val}"
        )
        if l5_info.get("status") == "ok":
            contrast = l5_info.get("contrast", {})
            dim_items = [(k, v) for k, v in contrast.items() if v.get("is_dimension_estimator") and math.isfinite(v.get("delta", float("nan")))]
            if dim_items:
                lines.append(f"L5 roughness contrast around T0 ({l5_info.get('n_left')}/{l5_info.get('n_right')} pts, {len(dim_items)} dimension estimators):")
                for name, v in sorted(dim_items, key=lambda x: abs(x[1]["delta"]), reverse=True)[:8]:
                    lines.append(f"  {name:24s}: left={v['value_left']:.4f}  right={v['value_right']:.4f}  delta={v['delta']:+.4f}")
            # Also show T_log if full exploratory diagnostics were run
            if l5_info.get("t_log") and l5_info["t_log"].get("calibrated"):
                lines.append(
                    f"T_log (exploratory diagnostic only): t={t_log_target.get('t_log')} | "
                    f"d_median={t_log_target.get('d_median')} | "
                    f"d_null={t_log_target.get('d_null_median')} | p={t_log_target.get('p_permutation')}"
                )
            inter = levels.get("L7_interactions", {})
            if isinstance(inter, dict) and inter.get("status") == "ok":
                summ = inter.get("summary", {})
                lines.append(
                    f"L7 interactions (exploratory): {summ.get('total_complementary_pairs')} complementary pairs, "
                    f"{summ.get('n_segments')} segments"
                )
        elif l5_info.get("status") == "insufficient_data":
            lines.append("L5 roughness contrast: insufficient data (segments too short)")
    else:
        lines.append("No finite threshold found — do not announce a threshold.")

    # Structural scans summary (all channels explicitly shown with null model & raw/Holm p)
    sb = structural_scans.get("slope_break", {})
    dj = structural_scans.get("detrended_jump", {})
    vs = structural_scans.get("volatility_shift", {})
    ds_s = structural_scans.get("deseasonalized_shift", {})
    lines.append(
        "Evaluated channels (m="
        f"{best.get('n_channels_family', '?')}): "
        f"level={level_scan.get('control_value', 'N/A')} (d={level_scan.get('cohen_d', float('nan')):.2f}, p_adj={level_scan.get('p_channel_adjusted', float('nan')):.4f}) | "
        f"jump={dj.get('control_value', 'N/A')} (d={dj.get('effect_d', float('nan')):.2f}, p_adj={dj.get('p_channel_adjusted', float('nan')):.4f}) | "
        f"slope={sb.get('control_value', 'N/A')} (t={sb.get('t_stat', float('nan')):.2f}, p_adj={sb.get('p_channel_adjusted', float('nan')):.4f}) | "
        f"vol={vs.get('control_value', 'N/A')} (d={vs.get('cohen_d', float('nan')):.2f}, p_adj={vs.get('p_channel_adjusted', float('nan')):.4f}) | "
        f"deseas={ds_s.get('control_value', 'N/A')} (d={ds_s.get('cohen_d', float('nan')):.2f}, p_adj={ds_s.get('p_channel_adjusted', float('nan')):.4f})"
    )
    if multivariate_scan.get("status") == "ok":
        lines.append(
            f"Multivariate composite: {ds.control_name} = {multivariate_scan['control_value']:.4g} | "
            f"composite_d = {multivariate_scan['composite_effect']:.3f} | "
            f"p = {multivariate_scan['p_permutation_scan_corrected']:.4f}"
        )
    if multi_thresholds.get("n_thresholds", 0) > 1:
        mt_vals = ", ".join(f"{t['control_value']:.4g}" for t in multi_thresholds["thresholds"])
        lines.append(f"Multi-threshold segmentation ({multi_thresholds['n_thresholds']} breaks): [{mt_vals}]")
    lines.append(
        f"Walk-forward F1: {wf.get('mean_f1')} (conservative: {wf.get('mean_f1_conservative')}) | "
        f"Multi-window stability: {falsif.get('multi_window', {}).get('stability')}"
    )
    lines.append(f"Rename invariance: {rename_ok} (P3)")
    lines.append(
        f"Manifest: sha256={src_hash[:16]}... seed={cfg.seed} mode={cfg.mode} elapsed={elapsed:.2f}s"
    )
    lines.append("Limits: detection + falsification only. Independent replication required before action.")

    report_text = "\n".join(lines)
    (outdir / "report.txt").write_text(report_text, encoding="utf-8")
    is_transformed = best.get("detector") in ("volatility_shift", "deseasonalized_shift")
    html_report = render_html_report(
        result,
        ds.control,
        ds.target,
        active_control=mt_ctrl if is_transformed else None,
        active_target=mt_tgt if is_transformed else None,
    )
    (outdir / "report.html").write_text(html_report, encoding="utf-8")
    log_message(cfg, report_text)
    return result

## Module `regression.py`

In [ ]:
"""Fast built-in regression test suite (--self-test)."""


import contextlib
import io
import math
import tempfile
from typing import Any, Dict, List

import numpy as np



def run_regression_tests() -> Dict[str, Any]:
    """Run comprehensive non-regression checks across core pipeline invariants."""
    with tempfile.TemporaryDirectory() as tmpdir:
        checks: List[Dict[str, Any]] = []

        def _check(name: str, ok: bool, detail: str = "") -> None:
            checks.append({"name": name, "passed": bool(ok), "detail": str(detail)[:200]})

        cfg = UniversalConfig(mode="fast", verbose=False)
        with contextlib.redirect_stdout(io.StringIO()):
            res = analyze(demo_dataset("step", n=200), config=cfg, output_dir=f"{tmpdir}/step")

        _check("step finds threshold", res.get("scan", {}).get("status") == "ok")
        _check(
            "adjusted scan null p present",
            math.isfinite(res.get("scan", {}).get("p_channel_adjusted", float("nan"))),
        )
        _check(
            "L6 disabled in core",
            res.get("levels", {}).get("L6_prediction", {}).get("status") == "disabled_in_core",
        )
        _check(
            "L5 single-pass contrast on default path",
            res.get("levels", {}).get("L5_complexity", {}).get("status") == "ok"
            and res.get("levels", {}).get("L5_complexity", {}).get("mode") == "single_pass_contrast"
            and res.get("levels", {}).get("L7_interactions", {}).get("status") == "disabled_by_default",
        )
        _check(
            "L4 signal present",
            res.get("levels", {}).get("L4_signal", {}).get("status") == "ok",
        )
        _check(
            "relative windows applied",
            res.get("config", {}).get("window_sizes")
            == sorted({max(8, 200 // 10), max(8, 200 // 5), max(8, 200 // 3)}),
        )

        # Explicit opt-in enables exploratory T_log and L7 interactions
        cfg_exp = UniversalConfig(mode="fast", verbose=False, run_complexity_diagnostics=True)
        with contextlib.redirect_stdout(io.StringIO()):
            res_exp = analyze(demo_dataset("step", n=200), config=cfg_exp, output_dir=f"{tmpdir}/step_exp")
        _check(
            "T_log calibrated on step (exploratory opt-in)",
            bool(res_exp.get("t_log_target", {}).get("calibrated")),
        )
        _check(
            "L7 interactions ok (exploratory opt-in)",
            res_exp.get("levels", {}).get("L7_interactions", {}).get("status") == "ok",
        )

        # Null series MUST produce Grade D
        cfg2 = UniversalConfig(mode="fast", verbose=False)
        with contextlib.redirect_stdout(io.StringIO()):
            res_null = analyze(
                demo_dataset("null", n=200), config=cfg2, output_dir=f"{tmpdir}/null"
            )
        _check(
            "null yields grade D",
            res_null.get("grade", {}).get("grade") == "D",
            f"grade={res_null.get('grade', {}).get('grade')}",
        )

        # Predict refused by default
        _check("predict refused on empty", predict_universal([], "").get("status") == "refused")
        _check(
            "predict refused on mixed/weak",
            predict_universal([res, res_null], "test question").get("status") == "refused",
        )

        # Permuted control collapses
        pc = res.get("falsification", {}).get("permuted_control_same_cut", {})
        _check(
            "permuted control collapses",
            math.isfinite(pc.get("d_permuted_median", float("nan")))
            and abs(pc.get("d_permuted_median", 999)) < 0.5 * abs(pc.get("d_observed", 0)),
        )

        # Walk-forward tracks abstentions & conservative F1
        wf = res.get("walk_forward", {})
        _check(
            "walk-forward tracks abstentions",
            "n_abstentions" in wf and "mean_f1_conservative" in wf,
        )

        # Candidates respect [w, n-w) for n > 400
        cfg_big = UniversalConfig(mode="fast", verbose=False)
        with contextlib.redirect_stdout(io.StringIO()):
            res_big = analyze(
                demo_dataset("step", n=1000, seed=42),
                config=cfg_big,
                output_dir=f"{tmpdir}/big",
            )
        w_big = res_big.get("config", {}).get("window_sizes", [80])[0]
        idx_big = res_big.get("scan", {}).get("idx")
        _check(
            "candidate index inside [w, n-w)",
            idx_big is not None and w_big <= idx_big < 1000 - w_big,
            f"idx={idx_big} w={w_big}",
        )

        # 29 estimators registered and LZ / BoxCount distinguish noise from line
        _check("29 estimators registered", len(ESTIMATORS) == 29, f"unique={len(UNIQUE_ESTIMATORS)}")
        rng = np.random.default_rng(42)
        white = rng.normal(0.0, 1.0, 200)
        line = np.linspace(0.0, 10.0, 200)
        _check(
            "LZ distinguishes noise vs line",
            est_lz_complexity(white) > 3.0 * est_lz_complexity(line),
            f"white={est_lz_complexity(white):.3f} line={est_lz_complexity(line):.3f}",
        )
        _check(
            "BoxCount distinguishes noise vs line",
            est_boxcount(white) > est_boxcount(line) + 0.2,
            f"white={est_boxcount(white):.3f} line={est_boxcount(line):.3f}",
        )

        # Config immutability across consecutive analyze() calls
        cfg_mut = UniversalConfig(mode="fast", verbose=False)
        ws_before = list(cfg_mut.window_sizes)
        with contextlib.redirect_stdout(io.StringIO()):
            analyze(demo_dataset("step", n=120), config=cfg_mut, output_dir=f"{tmpdir}/mut1")
            analyze(demo_dataset("step", n=300), config=cfg_mut, output_dir=f"{tmpdir}/mut2")
        _check("config not mutated by analyze()", list(cfg_mut.window_sizes) == ws_before)

        # Single window stability == 0
        cfg_1w = UniversalConfig(
            mode="fast", verbose=False, use_relative_windows=False, window_sizes=[50]
        )
        with contextlib.redirect_stdout(io.StringIO()):
            res_1w = analyze(
                demo_dataset("step", n=200), config=cfg_1w, output_dir=f"{tmpdir}/1w"
            )
        _check(
            "single window stability is 0.0",
            res_1w.get("falsification", {}).get("multi_window", {}).get("stability") == 0.0,
        )

        # Optional segmented validation smoke check
        suite = estimator_validation_suite(
            np.sin(np.linspace(0, 20, 100)) + np.random.default_rng(7).normal(0, 0.1, 100),
            seed=123,
            n_mc_paths=50,
            threshold_idx=50,
        )
        _check(
            "29 estimators across 3 segmented cohorts",
            all(
                len(suite.get("stages", {}).get(st, {}).get("per_estimator", {})) == 29
                for st in ("initial", "pre_threshold", "post_threshold")
            ),
        )

        ok_all = all(c["passed"] for c in checks)
        print(
            f"[REGRESSION] {'ALL PASSED' if ok_all else 'FAILED'}: "
            f"{sum(c['passed'] for c in checks)}/{len(checks)}"
        )
        for c in checks:
            print(f"  {'PASS' if c['passed'] else 'FAIL'} {c['name']} {c['detail']}")
        return {"all_passed": ok_all, "checks": checks}

## Module `cli.py`

In [ ]:
"""Command-line interface for UNIVERSAL_LOG."""


import argparse
from typing import Any, List, Optional



def describe_limits() -> str:
    """Return a human-readable summary of frozen gates, stubs, and non-claims."""
    lines = [
        f"UNIVERSAL_LOG {VERSION} — frozen limits and non-claims",
        "",
        "Routing gates (DEFAULT_ROUTING_GATES):",
    ]
    for k, v in DEFAULT_ROUTING_GATES.items():
        lines.append(f"  {k} = {v}")
    n_stubs = 1  # Danco
    lines += [
        "",
        f"Estimators registered: {len(ESTIMATORS)} total "
        f"(13 unique dimension estimators evaluated in default L5 contrast; "
        f"Danco is a permanent NaN stub; PersistentHomology needs optional ripser; "
        f"{n_stubs} stub total).",
        "Surrogate null: AR(1), or integrated ARIMA(1,1,0) when phi_within >= 0.75 and phi_diff1 >= 0.25.",
        "Grading: compound falsification failure (AR/ARIMA + unstable/WF/white-noise/split-half, or unstable + opposite split-half) drops to Grade D.",
        "T_log: permutation-calibrated diagnostic, n_perm=25 -> p-value floor 1/26 ~= 0.0385 "
        "(resolution limit, not proof).",
        "T_log never changes the A/B/C/D grade.",
        "",
        "Non-claims: detection + falsification only. NOT causal proof, NOT medical diagnosis, "
        "NOT financial advice.",
        "Any threshold (even Grade A) requires independent replication before action.",
    ]
    return "\n".join(lines)


def main(argv: Optional[List[str]] = None) -> Any:
    """Parse CLI arguments and dispatch analysis, demo, or regression self-test."""
    ap = argparse.ArgumentParser(
        description=f"UNIVERSAL_LOG ({VERSION}) — Domain-agnostic threshold detection & falsification."
    )
    ap.add_argument("--input", type=str, default=None, help="Input CSV, Parquet, or JSON file.")
    ap.add_argument(
        "--describe-limits",
        action="store_true",
        help="Print frozen gates, placeholders, and non-claims, then exit.",
    )
    ap.add_argument(
        "--control",
        type=str,
        default=None,
        help="Ordered control column name (auto-inferred if omitted).",
    )
    ap.add_argument(
        "--target",
        type=str,
        default=None,
        help="Target variable column name (auto-inferred if omitted).",
    )
    ap.add_argument(
        "--mode",
        type=str,
        default="fast",
        choices=["fast", "robust", "full"],
        help="Sampling intensity preset.",
    )
    ap.add_argument(
        "--detection-mode",
        type=str,
        default="auto",
        choices=["auto", "mean"],
        help="'auto' evaluates structural channels (jump/slope/vol) alongside level shifts; 'mean' restricts primary scan to level shift.",
    )
    ap.add_argument(
        "--output",
        type=str,
        default="./results_v7",
        help="Output directory for results.json, thresholds.csv, report.txt, and report.html.",
    )
    ap.add_argument(
        "--demo",
        type=str,
        default=None,
        choices=["step", "logistic", "null"],
        help="Generate and analyze a synthetic domain-free demo dataset.",
    )
    ap.add_argument(
        "--harmonic-ablation",
        action="store_true",
        help="Run comparative harmonic-jitter ablation test (P4).",
    )
    ap.add_argument(
        "--complexity-diagnostics",
        action="store_true",
        help="Enable exploratory 29-estimator complexity (L5 T_log) and L7 sliding interactions.",
    )
    ap.add_argument(
        "--full-validation",
        action="store_true",
        help="Run the heavy 3-cohort segmented MC + trajectory validation suite.",
    )
    ap.add_argument(
        "--self-test",
        action="store_true",
        help="Run the built-in non-regression test suite.",
    )
    ap.add_argument("--seed", type=int, default=42, help="Random seed (default: 42).")
    ap.add_argument("--quiet", action="store_true", help="Suppress verbose console output.")

    args = ap.parse_args(argv)
    cfg = UniversalConfig(
        mode=args.mode,
        seed=args.seed,
        output_dir=args.output,
        harmonic_ablation=args.harmonic_ablation,
        run_complexity_diagnostics=args.complexity_diagnostics,
        run_estimator_validation=args.full_validation,
        detection_mode=args.detection_mode,
        verbose=not args.quiet,
    )

    if args.describe_limits:
        print(describe_limits())
        return None

    if args.self_test:
        res = run_regression_tests()
        if not res.get("all_passed"):
            raise SystemExit(1)
        return res

    if args.demo:
        path = generate_example_csv(args.demo, f"./demo_{args.demo}.csv", seed=args.seed)
        if cfg.verbose:
            print(f"[DEMO {args.demo}] generated: {path}", flush=True)
        return analyze(path, config=cfg, output_dir=args.output)

    if not args.input:
        ap.error("--input is required (or specify --demo step|logistic|null or --self-test)")

    return analyze(
        args.input,
        control=args.control,
        target=args.target,
        config=cfg,
        output_dir=args.output,
    )

## Démo — détection de seuil (step)

In [ ]:
cfg = UniversalConfig(mode='fast', verbose=False)
res = analyze(demo_dataset('step', n=300), config=cfg, output_dir='./results_v7/notebook_demo')
print('grade:', res['grade']['grade'], '| threshold T0:', res['scan']['control_value'])
res['grade']['verdict']

## Démo — série nulle (doit donner D)

In [ ]:
res_null = analyze(demo_dataset('null', n=300), config=cfg, output_dir='./results_v7/notebook_null')
print('grade:', res_null['grade']['grade'])
res_null['grade']['grade']

## Non-régression (self-test intégré)

In [ ]:
result = run_regression_tests()
result['all_passed']